# 08 - Recomendador híbrido final

Este notebook implementa un primer prototipo seguro del recomendador híbrido final. Sustituye LightFM como ranking principal y combina varias señales trazables:

- perfil de contenido explicable;
- filtrado colaborativo item-item basado en MovieLens;
- ratings y vistas reales de Trakt;
- filtros de calidad;
- diversidad;
- explicaciones breves de cada recomendación.

LightFM queda como experimento independiente en el notebook 07 y no se usa en este ranking final.

## 1. Imports y configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("default")

RANDOM_STATE = 42

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS_RESULTADOS = ROOT / "reports" / "resultados"
TAGS_SEMANTIC_CLEAN_PATH = DATA_PROCESSED / "tags_semantic_clean.csv"
TAG_SEMANTIC_STATS_PATH = DATA_PROCESSED / "tag_semantic_stats.csv"

REPORTS_RESULTADOS.mkdir(parents=True, exist_ok=True)

## 2. Funciones auxiliares de carga

In [2]:
def require_file(path, message):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo esperado: {path}. {message}")
    return path


def load_csv_checked(path, message):
    return pd.read_csv(require_file(path, message))

## 3. Carga de datos

In [3]:
ratings = load_csv_checked(
    DATA_RAW / "ratings.csv",
    "Ejecuta 01_carga_datos.ipynb o verifica la carpeta data/raw.",
)
movies = load_csv_checked(
    DATA_PROCESSED / "movies_clean.csv",
    "Ejecuta 02_limpieza_transformacion.ipynb.",
)
trakt_ratings = load_csv_checked(
    DATA_PROCESSED / "trakt_ratings_mapped.csv",
    "Ejecuta 06_trakt_api_integracion.ipynb.",
)
trakt_watched = load_csv_checked(
    DATA_PROCESSED / "trakt_watched_mapped.csv",
    "Ejecuta 06_trakt_api_integracion.ipynb.",
)

datasets = {
    "ratings": ratings,
    "movies": movies,
    "trakt_ratings": trakt_ratings,
    "trakt_watched": trakt_watched,
}

for name, df in datasets.items():
    print(f"{name}: shape={df.shape}")
    print(f"Columnas: {list(df.columns)}\n")

print(f"Usuarios MovieLens: {ratings['userId'].nunique() if 'userId' in ratings.columns else 'columna userId no encontrada'}")
print(f"Películas MovieLens en ratings: {ratings['movieId'].nunique() if 'movieId' in ratings.columns else 'columna movieId no encontrada'}")
print(f"Ratings MovieLens: {len(ratings):,}")
print(f"Ratings Trakt mapeados: {len(trakt_ratings):,}")
print(f"Vistas Trakt mapeadas: {len(trakt_watched):,}")

ratings: shape=(32000204, 4)
Columnas: ['userId', 'movieId', 'rating', 'timestamp']

movies: shape=(87585, 26)
Columnas: ['movieId', 'title', 'genres', 'year', 'title_clean', 'rating_mean', 'rating_count', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

trakt_ratings: shape=(147, 23)
Columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 'movieId_tmdb', 'movieId_imdb', 'movieId', 'match_source', 'title_ml', 'title_clean_ml', 'year_ml', 'genres_ml', 'rating_mean_ml', 'rating_count_ml']

trakt_watched: shape=(256, 23)
Columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 

## 4. Normalización defensiva de columnas

In [4]:
def _normalized_name(name):
    return str(name).strip().lower().replace("_", "").replace("-", "").replace(" ", "")


def find_column(df, candidates):
    exact = {str(col): col for col in df.columns}
    for candidate in candidates:
        if candidate in exact:
            return exact[candidate]
    normalized = {_normalized_name(col): col for col in df.columns}
    for candidate in candidates:
        col = normalized.get(_normalized_name(candidate))
        if col is not None:
            return col
    return None


def rename_first_match(df, target, candidates, required=True):
    if target in df.columns:
        return df
    source = find_column(df, candidates)
    if source is None:
        if required:
            raise KeyError(f"No se pudo encontrar una columna equivalente a '{target}'. Candidatas: {candidates}")
        return df
    return df.rename(columns={source: target})


ratings = rename_first_match(ratings, "userId", ["userId", "user_id", "userid"])
ratings = rename_first_match(ratings, "movieId", ["movieId", "movie_id", "movieid"])
ratings = rename_first_match(ratings, "rating", ["rating", "score"])

movies = rename_first_match(movies, "movieId", ["movieId", "movie_id", "movieid"])
movies = rename_first_match(movies, "title", ["title", "movie_title", "name"])
movies = rename_first_match(movies, "genres", ["genres", "genre", "movie_genres"])
movies = rename_first_match(movies, "year", ["year", "release_year"], required=False)
movies = rename_first_match(movies, "rating_mean", ["rating_mean", "mean_rating", "avg_rating", "average_rating"], required=False)
movies = rename_first_match(movies, "rating_count", ["rating_count", "num_ratings", "n_ratings", "rating_n"], required=False)

trakt_ratings = rename_first_match(trakt_ratings, "movieId", ["movieId", "movie_id", "movieid"])
trakt_watched = rename_first_match(trakt_watched, "movieId", ["movieId", "movie_id", "movieid"])

for df in [ratings, movies, trakt_ratings, trakt_watched]:
    df["movieId"] = pd.to_numeric(df["movieId"], errors="coerce")

ratings["userId"] = pd.to_numeric(ratings["userId"], errors="coerce")
ratings["rating"] = pd.to_numeric(ratings["rating"], errors="coerce")
ratings = ratings.dropna(subset=["userId", "movieId", "rating"]).copy()
ratings["userId"] = ratings["userId"].astype(int)
ratings["movieId"] = ratings["movieId"].astype(int)

movies = movies.dropna(subset=["movieId", "title", "genres"]).copy()
movies["movieId"] = movies["movieId"].astype(int)

if "year" not in movies.columns:
    movies["year"] = movies["title"].astype(str).str.extract(r"\((\d{4})\)", expand=False)
movies["year"] = pd.to_numeric(movies["year"], errors="coerce")

rating_stats = ratings.groupby("movieId")["rating"].agg(rating_mean_calc="mean", rating_count_calc="count").reset_index()
movies = movies.merge(rating_stats, on="movieId", how="left")
if "rating_mean" not in movies.columns:
    movies["rating_mean"] = movies["rating_mean_calc"]
else:
    movies["rating_mean"] = pd.to_numeric(movies["rating_mean"], errors="coerce").fillna(movies["rating_mean_calc"])
if "rating_count" not in movies.columns:
    movies["rating_count"] = movies["rating_count_calc"]
else:
    movies["rating_count"] = pd.to_numeric(movies["rating_count"], errors="coerce").fillna(movies["rating_count_calc"])
movies = movies.drop(columns=["rating_mean_calc", "rating_count_calc"])
movies["rating_count"] = movies["rating_count"].fillna(0).astype(int)

personal_rating_col = find_column(
    trakt_ratings,
    ["user_rating_normalized", "rating_normalized", "rating", "user_rating"],
)
if personal_rating_col is None:
    raise KeyError("No se encontró una columna de rating personal en Trakt.")

trakt_ratings["user_rating_5"] = pd.to_numeric(trakt_ratings[personal_rating_col], errors="coerce")
if trakt_ratings["user_rating_5"].dropna().max() > 5:
    trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"] / 2.0
trakt_ratings["user_rating_5"] = trakt_ratings["user_rating_5"].clip(0, 5)

trakt_ratings = trakt_ratings.dropna(subset=["movieId", "user_rating_5"]).copy()
trakt_watched = trakt_watched.dropna(subset=["movieId"]).copy()
trakt_ratings["movieId"] = trakt_ratings["movieId"].astype(int)
trakt_watched["movieId"] = trakt_watched["movieId"].astype(int)

print(f"Columna de rating personal detectada: {personal_rating_col}")
print(f"Películas limpias disponibles: {len(movies):,}")

Columna de rating personal detectada: user_rating_normalized
Películas limpias disponibles: 80,505


## 5. Diagnóstico de perfil Trakt

In [5]:
# Diagnóstico de perfil Trakt

movie_profile_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
movie_profile_cols = [col for col in movie_profile_cols if col in movies.columns]

trakt_ratings_profile = trakt_ratings.merge(
    movies[movie_profile_cols],
    on="movieId",
    how="left",
    suffixes=("_trakt", "_movie")
)

# Resolver columnas duplicadas tras el merge
if "title" not in trakt_ratings_profile.columns:
    if "title_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_movie"]
    elif "title_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["title"] = trakt_ratings_profile["title_trakt"]
    else:
        trakt_ratings_profile["title"] = "movieId=" + trakt_ratings_profile["movieId"].astype(str)

if "genres" not in trakt_ratings_profile.columns:
    if "genres_movie" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_movie"]
    elif "genres_trakt" in trakt_ratings_profile.columns:
        trakt_ratings_profile["genres"] = trakt_ratings_profile["genres_trakt"]
    else:
        trakt_ratings_profile["genres"] = ""

# Asegurar user_rating_5
if "user_rating_5" not in trakt_ratings_profile.columns:
    raise KeyError(
        "No existe la columna user_rating_5. Revisa la celda anterior de normalización de ratings de Trakt."
    )

liked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] >= 4.0].copy()
neutral_movies = trakt_ratings_profile[
    (trakt_ratings_profile["user_rating_5"] > 2.5)
    & (trakt_ratings_profile["user_rating_5"] < 4.0)
].copy()
disliked_movies = trakt_ratings_profile[trakt_ratings_profile["user_rating_5"] <= 2.5].copy()

print(f"Películas gustadas: {len(liked_movies)}")
print(f"Películas neutras: {len(neutral_movies)}")
print(f"Películas no gustadas: {len(disliked_movies)}")

if len(liked_movies) < 3:
    warnings.warn("Perfil positivo muy pequeño; las recomendaciones pueden ser poco estables.")

display_cols = ["title", "user_rating_5"]
display(liked_movies.sort_values("user_rating_5", ascending=False)[display_cols].head(10))
display(disliked_movies.sort_values("user_rating_5")[display_cols].head(10))

Películas gustadas: 47
Películas neutras: 56
Películas no gustadas: 20


,title,user_rating_5
96,Interstellar (2014),5.0
26,Toy Story 3 (2010),4.5
38,Toy Story 2 (1999),4.5
33,Shrek 2 (2004),4.5
112,No Country for Old Men (2007),4.5
82,Donnie Darko (2001),4.5
23,Once Upon a Time in Hollywood (2019),4.5
104,Monty Python's Life of Brian (1979),4.5
88,Shutter Island (2010),4.5
110,Arrival (2016),4.5


,title,user_rating_5
74,Now You See Me (2013),1.0
91,Aftersun (2022),1.0
86,Get Out (2017),1.5
10,My Fault (2023),1.5
73,"Matrix Revolutions, The (2003)",2.0
75,The Revenant (2015),2.0
64,Bullet Train (2022),2.0
66,Gravity (2013),2.0
76,Iron Man (2008),2.0
63,"Maze Runner, The (2014)",2.0


## 6. Preparación de matriz colaborativa item-item

In [6]:
valid_movie_ids = set(movies["movieId"].dropna().astype(int))
popular_movie_ids = set(movies.loc[movies["rating_count"] >= 50, "movieId"].astype(int))

ratings_cf = ratings[ratings["movieId"].isin(valid_movie_ids & popular_movie_ids)].copy()
user_counts = ratings_cf.groupby("userId").size()
eligible_users = user_counts[user_counts >= 20].index
ratings_cf = ratings_cf[ratings_cf["userId"].isin(eligible_users)].copy()

if ratings_cf.empty:
    warnings.warn("La matriz colaborativa queda vacía tras los filtros de calidad. Revisa ratings.csv y movies_clean.csv.")
    user_item_matrix = sparse.csr_matrix((0, 0), dtype=np.float32)
    movie_id_to_col = {}
    col_to_movie_id = {}
    movie_id_to_title = {}
else:
    ratings_cf["mean_rating_user"] = ratings_cf.groupby("userId")["rating"].transform("mean")
    ratings_cf["rating_centered"] = ratings_cf["rating"] - ratings_cf["mean_rating_user"]

    user_codes, user_ids = pd.factorize(ratings_cf["userId"], sort=True)
    movie_codes, movie_ids = pd.factorize(ratings_cf["movieId"], sort=True)

    user_item_matrix = sparse.csr_matrix(
        (ratings_cf["rating_centered"].astype(np.float32), (user_codes, movie_codes)),
        shape=(len(user_ids), len(movie_ids)),
        dtype=np.float32,
    )

    movie_id_to_col = {int(movie_id): int(col) for col, movie_id in enumerate(movie_ids)}
    col_to_movie_id = {int(col): int(movie_id) for col, movie_id in enumerate(movie_ids)}
    movie_id_to_title = movies.set_index("movieId")["title"].to_dict()

matrix_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]
density = user_item_matrix.nnz / matrix_cells if matrix_cells else 0
print(f"Dimensiones matriz usuario-película: {user_item_matrix.shape}")
print(f"Interacciones no nulas: {user_item_matrix.nnz:,}")
print(f"Densidad aproximada: {density:.6f}")

Dimensiones matriz usuario-película: (200526, 15947)
Interacciones no nulas: 31,460,128
Densidad aproximada: 0.009838


## 7. Función de similitud item-item bajo demanda

In [7]:
SIMILAR_MOVIES_COLUMNS = ["movieId", "similarity", "title", "genres", "year", "rating_mean", "rating_count"]


def get_similar_movies_item_item(
    target_movie_id,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n=50,
    min_similarity=0.05,
):
    target_movie_id = int(target_movie_id)
    if target_movie_id not in movie_id_to_col or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=SIMILAR_MOVIES_COLUMNS)

    target_col = movie_id_to_col[target_movie_id]
    target_vector = user_item_matrix[:, target_col].T
    similarities = cosine_similarity(target_vector, user_item_matrix.T).ravel()

    result = pd.DataFrame(
        {
            "movieId": [col_to_movie_id[col] for col in range(len(similarities))],
            "similarity": similarities,
        }
    )
    result = result[(result["movieId"] != target_movie_id) & (result["similarity"] > min_similarity)]
    result = result.sort_values("similarity", ascending=False).head(top_n)

    metadata_cols = ["movieId", "title", "genres", "year", "rating_mean", "rating_count"]
    result = result.merge(movies_df[metadata_cols], on="movieId", how="left")
    return result[SIMILAR_MOVIES_COLUMNS]

## 8. Sanity check de similitudes

In [8]:
liked_available = liked_movies[liked_movies["movieId"].isin(movie_id_to_col.keys())].head(3)

if liked_available.empty:
    warnings.warn("Ninguna película gustada está disponible en la matriz colaborativa item-item.")
else:
    for _, liked_row in liked_available.iterrows():
        title = liked_row.get("title", f"movieId={liked_row['movieId']}")
        print(f"\nPelícula base: {title}")
        similar = get_similar_movies_item_item(
            liked_row["movieId"],
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies,
            top_n=10,
            min_similarity=0.05,
        )
        display(similar)


Película base: Cinema Paradiso (Nuovo cinema Paradiso) (1989)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,912,0.155976,Casablanca (1942),Drama|Romance,1942.0,4.194517,32190
1,1225,0.155691,Amadeus (1984),Drama,1984.0,4.072641,24986
2,1131,0.151308,Jean de Florette (1986),Drama|Mystery,1986.0,4.103292,3524
3,1193,0.148326,One Flew Over the Cuckoo's Nest (1975),Drama,1975.0,4.204229,44592
4,1207,0.147705,To Kill a Mockingbird (1962),Drama,1962.0,4.134469,18785
5,904,0.143327,Rear Window (1954),Mystery|Thriller,1954.0,4.226701,24883
6,2324,0.139511,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,1997.0,4.150374,29819
7,1132,0.139169,Manon of the Spring (Manon des sources) (1986),Drama,1986.0,4.063254,3233
8,58,0.136045,"Postman, The (Postino, Il) (1994)",Comedy|Drama|Romance,1994.0,3.967750,11659
9,1247,0.135198,"Graduate, The (1967)",Comedy|Drama|Romance,1967.0,4.021081,22319



Película base: City of God (Cidade de Deus) (2002)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,2959,0.224863,Fight Club (1999),Action|Crime|Drama|Thriller,1999.0,4.228780,77332
1,2329,0.222359,American History X (1998),Crime|Drama,1998.0,4.130444,38967
2,1213,0.219109,Goodfellas (1990),Crime|Drama,1990.0,4.188427,42003
3,4226,0.217764,Memento (2000),Mystery|Thriller,2000.0,4.141601,53658
4,44555,0.217555,"Lives of Others, The (Das leben der Anderen) (...",Drama|Romance|Thriller,2006.0,4.198770,12273
5,858,0.217434,"Godfather, The (1972)",Crime|Drama,1972.0,4.317030,66440
6,4235,0.215905,Amores Perros (Love's a Bitch) (2000),Drama|Thriller,2000.0,3.992665,7703
7,27773,0.213912,Old Boy (2003),Mystery|Thriller,2003.0,4.089464,17331
8,1221,0.209897,"Godfather: Part II, The (1974)",Crime|Drama,1974.0,4.264468,43111
9,7361,0.201881,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,2004.0,4.065294,44430



Película base: Edward Scissorhands (1990)


,movieId,similarity,title,genres,year,rating_mean,rating_count
0,551,0.204613,"Nightmare Before Christmas, The (1993)",Animation|Children|Fantasy|Musical,1993.0,3.742149,26081
1,2174,0.173241,Beetlejuice (1988),Comedy|Fantasy,1988.0,3.516450,24681
2,7361,0.132263,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,2004.0,4.065294,44430
3,7147,0.131229,Big Fish (2003),Drama|Fantasy|Romance,2003.0,3.798002,22072
4,4878,0.129285,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,2001.0,3.935533,35452
5,4973,0.127355,"Amelie (Fabuleux destin d'Amélie Poulain, Le) ...",Comedy|Romance,2001.0,4.082169,43459
6,2997,0.126717,Being John Malkovich (1999),Comedy|Drama|Fantasy,1999.0,3.933888,36151
7,1258,0.126577,"Shining, The (1980)",Horror,1980.0,4.036439,38640
8,1206,0.124933,"Clockwork Orange, A (1971)",Crime|Drama|Sci-Fi|Thriller,1971.0,3.985442,36818
9,1682,0.123761,"Truman Show, The (1998)",Comedy|Drama|Sci-Fi,1998.0,3.891833,44140


## 9. Score colaborativo personalizado

In [9]:
COLLAB_COLUMNS = [
    "movieId",
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
]


def _short_unique_titles(titles, max_titles=3):
    clean_titles = []
    for title in titles:
        if pd.notna(title) and str(title) not in clean_titles:
            clean_titles.append(str(title))
        if len(clean_titles) >= max_titles:
            break
    return ", ".join(clean_titles)


def _robust_minmax(series):
    series = pd.to_numeric(series, errors="coerce").fillna(0)
    if series.empty:
        return series
    low, high = np.nanpercentile(series, [5, 95])
    if np.isclose(low, high):
        return pd.Series(np.where(series > 0, 1.0, 0.0), index=series.index)
    return ((series - low) / (high - low)).clip(0, 1)


def compute_item_item_collab_scores(
    user_ratings_df,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies_df,
    top_n_per_seed=200,
    min_similarity=0.03,
):
    if user_ratings_df.empty or user_item_matrix.shape[1] == 0:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    seed_titles = movies_df.set_index("movieId")["title"].to_dict()
    aggregates = {}

    for _, user_row in user_ratings_df.dropna(subset=["movieId", "user_rating_5"]).iterrows():
        seed_movie_id = int(user_row["movieId"])
        if seed_movie_id not in movie_id_to_col:
            continue

        personal_weight = float(user_row["user_rating_5"]) - 3.0
        if np.isclose(personal_weight, 0):
            continue

        similar_movies = get_similar_movies_item_item(
            seed_movie_id,
            user_item_matrix,
            movie_id_to_col,
            col_to_movie_id,
            movies_df,
            top_n=top_n_per_seed,
            min_similarity=min_similarity,
        )
        seed_title = seed_titles.get(seed_movie_id, str(seed_movie_id))

        for _, similar_row in similar_movies.iterrows():
            candidate_id = int(similar_row["movieId"])
            contribution = float(similar_row["similarity"]) * personal_weight
            candidate = aggregates.setdefault(
                candidate_id,
                {
                    "collab_raw_score": 0.0,
                    "positive_collab_score": 0.0,
                    "negative_collab_score": 0.0,
                    "n_collab_evidence": 0,
                    "similar_liked_movies": [],
                    "similar_disliked_movies": [],
                },
            )
            candidate["collab_raw_score"] += contribution
            candidate["n_collab_evidence"] += 1
            if contribution > 0:
                candidate["positive_collab_score"] += contribution
                candidate["similar_liked_movies"].append(seed_title)
            elif contribution < 0:
                candidate["negative_collab_score"] += abs(contribution)
                candidate["similar_disliked_movies"].append(seed_title)

    if not aggregates:
        return pd.DataFrame(columns=COLLAB_COLUMNS)

    rows = []
    for movie_id, values in aggregates.items():
        rows.append(
            {
                "movieId": movie_id,
                "collab_raw_score": values["collab_raw_score"],
                "positive_collab_score": values["positive_collab_score"],
                "negative_collab_score": values["negative_collab_score"],
                "n_collab_evidence": values["n_collab_evidence"],
                "similar_liked_movies": _short_unique_titles(values["similar_liked_movies"]),
                "similar_disliked_movies": _short_unique_titles(values["similar_disliked_movies"]),
            }
        )

    scores = pd.DataFrame(rows)
    scores["collab_positive_raw"] = scores["collab_raw_score"].clip(lower=0)
    scores["item_item_collab_score"] = scores["collab_positive_raw"].rank(pct=True)
    scores.loc[scores["collab_positive_raw"] <= 0, "item_item_collab_score"] = 0
    scores["item_item_negative_collab_score"] = scores["negative_collab_score"].rank(pct=True)
    scores.loc[scores["negative_collab_score"] <= 0, "item_item_negative_collab_score"] = 0
    return scores[COLLAB_COLUMNS]


collab_scores = compute_item_item_collab_scores(
    trakt_ratings,
    user_item_matrix,
    movie_id_to_col,
    col_to_movie_id,
    movies,
)

print(f"Películas con señal colaborativa personalizada: {len(collab_scores):,}")
display(collab_scores.sort_values("item_item_collab_score", ascending=False).head(10))

Películas con señal colaborativa personalizada: 3,309


,movieId,collab_raw_score,collab_positive_raw,item_item_collab_score,item_item_negative_collab_score,positive_collab_score,negative_collab_score,n_collab_evidence,similar_liked_movies,similar_disliked_movies
200,2959,6.422709,6.422709,1.000000,0.904503,6.627034,0.204324,57,"City of God (Cidade de Deus) (2002), Dallas Bu...","Kick-Ass (2010), The Revenant (2015), Get Out ..."
93,4226,6.329096,6.329096,0.999698,0.921426,6.570744,0.241648,61,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"Gravity (2013), Kick-Ass (2010), The Revenant ..."
58,1213,5.755333,5.755333,0.999396,0.894228,5.945597,0.190264,55,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."
105,296,5.611456,5.611456,0.999093,0.871260,5.774753,0.163297,48,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Get Out (2017)"
10,858,5.593983,5.593983,0.998791,0.702025,5.655693,0.061710,53,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,The Revenant (2015)
125,1089,5.537346,5.537346,0.998489,0.896041,5.731123,0.193777,51,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."
203,7361,5.513310,5.513310,0.998187,0.906618,5.726481,0.213171,59,"City of God (Cidade de Deus) (2002), Dallas Bu...","Gravity (2013), The Revenant (2015), Get Out (..."
55,318,5.468758,5.468758,0.997885,0.907223,5.682389,0.213630,54,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,Harry Potter and the Deathly Hallows: Part 2 (...
24,1221,5.463455,5.463455,0.997582,0.689030,5.521255,0.057799,52,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,The Revenant (2015)
3,1193,5.354545,5.354545,0.997280,0.734361,5.432955,0.078410,49,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...,"The Revenant (2015), Dead Alive (Braindead) (1..."


## 10. Candidatos base y filtros

In [10]:
rated_movie_ids = set(trakt_ratings["movieId"].dropna().astype(int))
watched_movie_ids = set(trakt_watched["movieId"].dropna().astype(int))
excluded_movie_ids = rated_movie_ids | watched_movie_ids

base_candidates = movies.copy()
base_candidates["genres_text"] = base_candidates["genres"].fillna("").astype(str).str.strip()

candidate_mask = (
    ~base_candidates["movieId"].isin(excluded_movie_ids)
    & base_candidates["year"].notna()
    & base_candidates["genres_text"].ne("")
    & base_candidates["genres_text"].ne("(no genres listed)")
    & (base_candidates["rating_mean"] >= 3.2)
    & (base_candidates["rating_count"] >= 300)
)
candidates = base_candidates[candidate_mask].copy()

if len(candidates) < 100:
    warnings.warn("Quedan menos de 100 candidatos con rating_count >= 300; se relaja rating_count a 100.")
    candidate_mask = (
        ~base_candidates["movieId"].isin(excluded_movie_ids)
        & base_candidates["year"].notna()
        & base_candidates["genres_text"].ne("")
        & base_candidates["genres_text"].ne("(no genres listed)")
        & (base_candidates["rating_mean"] >= 3.2)
        & (base_candidates["rating_count"] >= 100)
    )
    candidates = base_candidates[candidate_mask].copy()

print(f"Candidatos tras filtros: {len(candidates):,}")

Candidatos tras filtros: 4,863


## 11. Score de contenido simple y explicable

In [11]:
def split_genres(genres):
    if pd.isna(genres):
        return []
    genres = str(genres).strip()
    if not genres or genres == "(no genres listed)":
        return []
    return [genre.strip() for genre in genres.split("|") if genre.strip()]


def build_genre_weights(profile_df):
    counts = {}
    for genres in profile_df["genres"].dropna():
        for genre in split_genres(genres):
            counts[genre] = counts.get(genre, 0) + 1
    if not counts:
        return {}
    max_count = max(counts.values())
    return {genre: count / max_count for genre, count in counts.items()}


def genre_affinity(genres, weights):
    genre_list = split_genres(genres)
    if not genre_list or not weights:
        return 0.0
    score = sum(weights.get(genre, 0.0) for genre in genre_list)
    normalizer = sum(weights.values()) if weights else 1.0
    return float(np.clip(score / normalizer, 0, 1))


positive_genre_weights = build_genre_weights(liked_movies)
negative_genre_weights = build_genre_weights(disliked_movies)

tag_columns = [col for col in ["tags", "tags_clean", "tags_features", "tags_features_en", "tag_features"] if col in movies.columns]
if tag_columns:
    print(f"Columnas de tags detectadas para fase futura, no usadas en este prototipo: {tag_columns}")

candidates["genre_profile_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, positive_genre_weights))
candidates["negative_genre_score"] = candidates["genres"].apply(lambda value: genre_affinity(value, negative_genre_weights))
candidates["content_profile_score"] = candidates["genre_profile_score"]
candidates["negative_similarity_score"] = candidates["negative_genre_score"]

print("Pesos positivos de géneros:")
print(positive_genre_weights)
print("Pesos negativos de géneros:")
print(negative_genre_weights)

Pesos positivos de géneros:
{'Drama': 1.0, 'Action': 0.25, 'Adventure': 0.2857142857142857, 'Crime': 0.35714285714285715, 'Thriller': 0.42857142857142855, 'Fantasy': 0.2857142857142857, 'Romance': 0.25, 'Comedy': 0.75, 'Animation': 0.21428571428571427, 'Children': 0.21428571428571427, 'IMAX': 0.07142857142857142, 'Horror': 0.07142857142857142, 'Musical': 0.03571428571428571, 'Sci-Fi': 0.39285714285714285, 'War': 0.07142857142857142, 'Mystery': 0.07142857142857142}
Pesos negativos de géneros:
{'Drama': 1.0, 'Romance': 0.375, 'Action': 0.875, 'Adventure': 0.625, 'Fantasy': 0.5, 'Mystery': 0.5, 'IMAX': 0.5, 'Comedy': 0.625, 'Thriller': 0.625, 'Sci-Fi': 0.5, 'Crime': 0.25, 'Horror': 0.25}


## Perfil semántico del usuario con tags/genome de MovieLens

In [12]:
import re


tags_path = DATA_RAW / "tags.csv"
genome_scores_path = DATA_RAW / "genome-scores.csv"
genome_tags_path = DATA_RAW / "genome-tags.csv"

semantic_files = {
    "tags_semantic_clean.csv": TAGS_SEMANTIC_CLEAN_PATH.exists(),
    "tag_semantic_stats.csv": TAG_SEMANTIC_STATS_PATH.exists(),
    "tags.csv": tags_path.exists(),
    "genome-scores.csv": genome_scores_path.exists(),
    "genome-tags.csv": genome_tags_path.exists(),
}
print("Archivos semánticos disponibles:")
for filename, exists in semantic_files.items():
    print(f"- {filename}: {'sí' if exists else 'no'}")

if TAGS_SEMANTIC_CLEAN_PATH.exists() and TAG_SEMANTIC_STATS_PATH.exists():
    semantic_source = "processed_tags"
elif tags_path.exists():
    warnings.warn("No existen tags semánticos procesados. Ejecuta primero notebooks/09_preprocesado_tags_semanticos.ipynb.")
    semantic_source = "raw_tags_fallback"
else:
    semantic_source = "none"
print(f"Fuente semántica usada: {semantic_source}")

SEMANTIC_COLUMNS = [
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
]

candidates["semantic_raw_score"] = 0.0
candidates["negative_semantic_raw_score"] = 0.0
candidates["semantic_profile_score"] = 0.0
candidates["negative_semantic_score"] = 0.0
candidates["semantic_explanation_terms"] = ""
candidates["negative_semantic_terms"] = ""
top_positive_semantic_terms = []
top_negative_semantic_terms = []
positive_tag_profile = pd.DataFrame(columns=["tag_clean", "positive_weight"])
negative_tag_profile = pd.DataFrame(columns=["tag_clean", "negative_weight"])


def normalize_positive_rank(raw_scores):
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores.fillna(0) > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized


def top_terms_from_contrib(contrib_df, term_col="tag", value_col="term_contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def normalize_tag_text(tag):
    if pd.isna(tag):
        return None
    tag_clean = str(tag).lower().strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    tag_clean = tag_clean.replace("_", " ")
    protected_hyphen_terms = {"thought-provoking", "post-apocalyptic"}
    if tag_clean not in protected_hyphen_terms:
        tag_clean = re.sub(r"\s*-\s*", " ", tag_clean)
    tag_clean = re.sub(r"\s+", " ", tag_clean).strip()
    tag_clean = tag_clean.strip('"\'`´“”‘’')
    if not tag_clean or tag_clean in {"nan", "none", "null"}:
        return None
    return tag_clean


def is_bad_tag(tag_clean):
    if tag_clean is None or not str(tag_clean).strip():
        return True
    tag_clean = str(tag_clean).strip()
    if len(tag_clean) < 3:
        return True
    if tag_clean.isnumeric():
        return True
    if re.fullmatch(r"[\W_]+", tag_clean):
        return True
    if re.search(r"\b(18|19|20)\d{2}\b", tag_clean):
        return True
    if re.fullmatch(r"\d{1,2}[/-]\d{1,2}([/-]\d{2,4})?", tag_clean):
        return True
    if "http" in tag_clean or "www." in tag_clean:
        return True
    if not re.search(r"[a-z]", tag_clean):
        return True

    exact_whitelist = {
        "great soundtrack", "visuals", "visually appealing", "dark comedy", "black comedy",
        "social commentary", "twist ending", "thought-provoking", "mindfuck", "time travel",
        "artificial intelligence", "post-apocalyptic", "coming of age", "atmospheric", "surreal",
        "psychological", "dystopia", "dreamlike", "stylized", "cinematography", "soundtrack",
        "bittersweet", "quirky", "nonlinear",
    }
    administrative_patterns = [
        "imdb", "oscar", "academy award", "criterion", "netflix", "dvd", "blu", "owned",
        "watchlist", "want to see", "seen", "watched", "top 250", "1001 movies", "afi",
        "award nominated", "award winner", "bd ", "bd-",
    ]
    if any(pattern in tag_clean for pattern in administrative_patterns):
        return True
    if tag_clean in exact_whitelist:
        return False

    exact_blacklist = {
        "owned", "own", "seen", "watched", "watchlist", "want to see", "to watch", "dvd",
        "bd-r", "bd r", "bdr", "blu-ray", "blu ray", "bluray", "blue-ray", "blue ray",
        "netflix", "amazon", "hulu", "criterion", "criterion collection", "library", "collection",
        "on dvr", "recorded", "vhs", "tv", "tivo", "imdb", "imdb top 250", "top 250",
        "top 100", "afi", "afi 100", "1001 movies", "1001 movies you must see before you die",
        "must see", "classic", "classics", "cult classic", "oscar", "oscars", "oscar winner",
        "oscar nominated", "academy award", "academy awards", "award winner", "award nominated",
        "best picture", "golden globe", "palme d'or", "good", "great", "best", "bad", "worst",
        "boring", "funny", "very funny", "hilarious", "overrated", "underrated", "favorite",
        "favourite", "favorites", "favourites", "excellent", "amazing", "awesome", "terrible",
        "awful", "mediocre", "masterpiece", "movie", "movies", "film", "films", "cinema",
        "cinematic", "based on", "based on book", "based on a book", "adapted from", "adaptation",
        "remake", "sequel", "franchise", "original", "drama", "comedy", "action", "thriller",
        "romance", "horror", "sci-fi", "science fiction", "crime", "adventure", "animation",
        "children", "fantasy", "mystery", "documentary", "war", "western", "musical", "film-noir",
        "film noir", "noir", "f word", "f-word",
    }
    if tag_clean in exact_blacklist:
        return True
    return False


def build_clean_tags_dataframe(tags_raw):
    tags_clean = tags_raw.copy()
    original_rows = len(tags_clean)
    original_unique_tags = tags_clean["tag"].nunique(dropna=True) if "tag" in tags_clean.columns else 0
    tags_clean["tag_clean"] = tags_clean["tag"].apply(normalize_tag_text)
    tags_clean["bad_tag"] = tags_clean["tag_clean"].apply(is_bad_tag)
    tags_clean = tags_clean[tags_clean["tag_clean"].notna() & ~tags_clean["bad_tag"]].copy()
    keep_cols = [col for col in ["userId", "movieId", "tag", "tag_clean", "timestamp"] if col in tags_clean.columns]
    tags_clean = tags_clean[keep_cols].copy()
    tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
    tags_clean = tags_clean.dropna(subset=["movieId"])
    tags_clean["movieId"] = tags_clean["movieId"].astype(int)
    if "userId" in tags_clean.columns:
        tags_clean["userId"] = pd.to_numeric(tags_clean["userId"], errors="coerce")
        tags_clean = tags_clean.drop_duplicates(["userId", "movieId", "tag_clean"])
    else:
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
    removed_rows = original_rows - len(tags_clean)
    removed_pct = removed_rows / original_rows * 100 if original_rows else 0
    print(f"Filas originales tags.csv: {original_rows:,}")
    print(f"Tags únicos originales: {original_unique_tags:,}")
    print(f"Filas tras limpieza básica: {len(tags_clean):,}")
    print(f"Tags únicos tras limpieza: {tags_clean['tag_clean'].nunique():,}")
    print(f"Filas eliminadas: {removed_rows:,} ({removed_pct:.2f}%)")
    return tags_clean


def build_tag_statistics(tags_clean):
    n_movies_total = tags_clean["movieId"].nunique()
    stats = tags_clean.groupby("tag_clean").agg(n_uses=("movieId", "size"), n_movies=("movieId", "nunique")).reset_index()
    if "userId" in tags_clean.columns:
        stats_users = tags_clean.groupby("tag_clean")["userId"].nunique().rename("n_users").reset_index()
        user_tag_counts = tags_clean.groupby(["tag_clean", "userId"]).size().rename("user_uses").reset_index()
        top_user = user_tag_counts.groupby("tag_clean")["user_uses"].max().rename("top_user_uses").reset_index()
        stats = stats.merge(stats_users, on="tag_clean", how="left").merge(top_user, on="tag_clean", how="left")
        stats["top_user_share"] = stats["top_user_uses"] / stats["n_uses"]
    else:
        stats["n_users"] = np.nan
        stats["top_user_uses"] = np.nan
        stats["top_user_share"] = 0.0
    stats["pct_movies"] = stats["n_movies"] / n_movies_total if n_movies_total else 0
    stats["idf"] = np.log((1 + n_movies_total) / (1 + stats["n_movies"])) + 1
    low_movies = stats["n_movies"] < 5
    low_users = stats["n_users"] < 5 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    high_pct_movies = stats["pct_movies"] > 0.05
    high_top_user_share = stats["top_user_share"] > 0.60 if "userId" in tags_clean.columns else pd.Series(False, index=stats.index)
    stats["is_reliable_tag"] = ~(low_movies | low_users | high_pct_movies | high_top_user_share)
    print(f"Tags antes del filtro de fiabilidad: {len(stats):,}")
    print(f"Tags fiables: {stats['is_reliable_tag'].sum():,}")
    print(f"Eliminados por n_movies bajo: {low_movies.sum():,}")
    print(f"Eliminados por n_users bajo: {low_users.sum():,}")
    print(f"Eliminados por pct_movies alto: {high_pct_movies.sum():,}")
    print(f"Eliminados por top_user_share alto: {high_top_user_share.sum():,}")
    reliable = stats[stats["is_reliable_tag"]].copy()
    display(reliable.sort_values("n_uses", ascending=False).head(30))
    display(reliable.sort_values("n_movies", ascending=False).head(30))
    display(reliable[reliable["n_movies"] >= 5].sort_values("idf", ascending=False).head(30))
    display(stats[high_top_user_share].sort_values("top_user_share", ascending=False).head(30))
    return stats


def build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats):
    idf_map = tag_stats.set_index("tag_clean")["idf"]
    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = tags_clean.merge(positive_seed, on="movieId", how="inner")
        positive_tags["idf"] = positive_tags["tag_clean"].map(idf_map).fillna(1.0)
        positive_tags["personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["tag_weight"] = positive_tags["personal_weight"] * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean").agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        positive_profile = positive_profile[positive_profile["positive_n_seed_movies"] >= 1]
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tags_clean.merge(negative_seed, on="movieId", how="inner")
        negative_tags["idf"] = negative_tags["tag_clean"].map(idf_map).fillna(1.0)
        negative_tags["personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["tag_weight"] = negative_tags["personal_weight"] * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean").agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        ).reset_index()
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= 0.5
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= 0.5
        positive_profile = positive_profile.sort_values("positive_weight", ascending=False)
        negative_profile = negative_profile.sort_values("negative_weight", ascending=False)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies):
    required_columns = {"movieId", "tag_clean", "idf"}
    if not required_columns.issubset(tags_clean.columns):
        missing = sorted(required_columns - set(tags_clean.columns))
        raise ValueError(f"tags_semantic_clean.csv no tiene las columnas esperadas: {missing}")

    profile_tags = tags_clean[["movieId", "tag_clean", "idf"]].copy()
    profile_tags["movieId"] = pd.to_numeric(profile_tags["movieId"], errors="coerce")
    profile_tags["idf"] = pd.to_numeric(profile_tags["idf"], errors="coerce").fillna(1.0)
    profile_tags = profile_tags.dropna(subset=["movieId", "tag_clean"])
    profile_tags["movieId"] = profile_tags["movieId"].astype(int)
    profile_tags = profile_tags.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_profile = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies"])
    if not positive_seed.empty:
        positive_tags = profile_tags.merge(positive_seed, on="movieId", how="inner")
        positive_tags["tag_weight"] = (positive_tags["user_rating_5"] - 3.0) * positive_tags["idf"]
        positive_profile = positive_tags.groupby("tag_clean", as_index=False).agg(
            positive_weight=("tag_weight", "sum"),
            positive_n_seed_movies=("movieId", "nunique"),
        )

    negative_profile = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = profile_tags.merge(negative_seed, on="movieId", how="inner")
        negative_tags["tag_weight"] = (3.0 - negative_tags["user_rating_5"]) * negative_tags["idf"]
        negative_profile = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("tag_weight", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    overlap = sorted(set(positive_profile["tag_clean"]) & set(negative_profile["tag_clean"]))
    if overlap:
        ambiguity_penalty = 0.25
        positive_profile.loc[positive_profile["tag_clean"].isin(overlap), "positive_weight"] *= ambiguity_penalty
        negative_profile.loc[negative_profile["tag_clean"].isin(overlap), "negative_weight"] *= ambiguity_penalty

    positive_profile = positive_profile.sort_values("positive_weight", ascending=False).head(80)
    negative_profile = negative_profile.sort_values("negative_weight", ascending=False).head(80)

    display(positive_profile.head(30))
    display(negative_profile.head(30))
    print(f"Tags en intersección positivo/negativo: {overlap[:30]}")
    print(f"Número de tags positivos: {len(positive_profile)}")
    print(f"Número de tags negativos: {len(negative_profile)}")
    return positive_profile, negative_profile


def compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_raw_score"] = 0.0
    result["negative_semantic_raw_score"] = 0.0
    result["semantic_profile_score"] = 0.0
    result["negative_semantic_score"] = 0.0
    result["semantic_explanation_terms"] = ""
    result["negative_semantic_terms"] = ""
    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    if not positive_tag_profile.empty:
        positive_contrib = candidate_tags.merge(positive_tag_profile[["tag_clean", "positive_weight"]], on="tag_clean", how="inner")
        positive_contrib["contribution"] = positive_contrib["positive_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_profile_score"] = normalize_positive_rank(result["semantic_raw_score"])
        result["semantic_explanation_terms"] = result["movieId"].map(
            top_terms_from_contrib(positive_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")

    if not negative_tag_profile.empty:
        negative_contrib = candidate_tags.merge(negative_tag_profile[["tag_clean", "negative_weight"]], on="tag_clean", how="inner")
        negative_contrib["contribution"] = negative_contrib["negative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["negative_semantic_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["negative_semantic_score"] = normalize_positive_rank(result["negative_semantic_raw_score"])
        result["negative_semantic_terms"] = result["movieId"].map(
            top_terms_from_contrib(negative_contrib, term_col="tag_clean", value_col="contribution", max_terms=5)
        ).fillna("")
    return result


def compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile):
    return compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)


positive_seed_ratings = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
negative_seed_ratings = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
positive_seed_ratings["movieId"] = positive_seed_ratings["movieId"].astype(int)
negative_seed_ratings["movieId"] = negative_seed_ratings["movieId"].astype(int)

if semantic_source == "processed_tags":
    tags_clean = pd.read_csv(TAGS_SEMANTIC_CLEAN_PATH)
    tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)
    required_clean_cols = {"movieId", "tag_clean", "idf"}
    required_stats_cols = {"tag_clean", "idf", "is_reliable_tag"}
    if not required_clean_cols.issubset(tags_clean.columns) or not required_stats_cols.issubset(tag_stats.columns):
        warnings.warn("Los tags semánticos procesados no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = tags_clean.copy()
        tags_clean["movieId"] = pd.to_numeric(tags_clean["movieId"], errors="coerce")
        tags_clean["idf"] = pd.to_numeric(tags_clean["idf"], errors="coerce").fillna(1.0)
        tags_clean = tags_clean.dropna(subset=["movieId", "tag_clean"])
        tags_clean["movieId"] = tags_clean["movieId"].astype(int)
        tags_clean = tags_clean.drop_duplicates(["movieId", "tag_clean"])
        print(f"Tags limpios cargados: {len(tags_clean):,}")
        print(f"Tags únicos limpios: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con tags limpios: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_processed_tags(tags_clean, liked_movies, disliked_movies)
        semantic_scores = compute_semantic_scores_from_processed_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

elif semantic_source == "genome":
    genome_scores = pd.read_csv(genome_scores_path)
    genome_tags = pd.read_csv(genome_tags_path)
    required_genome_cols = {"movieId", "tagId", "relevance"}
    required_tag_cols = {"tagId", "tag"}
    if not required_genome_cols.issubset(genome_scores.columns) or not required_tag_cols.issubset(genome_tags.columns):
        warnings.warn("Los archivos genome no tienen las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        genome_scores["movieId"] = pd.to_numeric(genome_scores["movieId"], errors="coerce")
        genome_scores["tagId"] = pd.to_numeric(genome_scores["tagId"], errors="coerce")
        genome_scores["relevance"] = pd.to_numeric(genome_scores["relevance"], errors="coerce")
        genome_scores = genome_scores.dropna(subset=["movieId", "tagId", "relevance"]).copy()
        genome_scores["movieId"] = genome_scores["movieId"].astype(int)
        genome_scores["tagId"] = genome_scores["tagId"].astype(int)
        genome_tags["tagId"] = pd.to_numeric(genome_tags["tagId"], errors="coerce")
        genome_tags = genome_tags.dropna(subset=["tagId", "tag"]).copy()
        genome_tags["tagId"] = genome_tags["tagId"].astype(int)

        relevant_movie_ids = set(candidates["movieId"].astype(int)) | set(positive_seed_ratings["movieId"]) | set(negative_seed_ratings["movieId"])
        genome_scores = genome_scores[genome_scores["movieId"].isin(relevant_movie_ids)].copy()
        candidate_genome = genome_scores[genome_scores["movieId"].isin(candidates["movieId"].astype(int))].copy()

        positive_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not positive_seed_ratings.empty:
            positive_profile = genome_scores.merge(positive_seed_ratings, on="movieId", how="inner")
            positive_profile["personal_weight"] = positive_profile["user_rating_5"] - 3.0
            positive_profile["weighted_relevance"] = positive_profile["relevance"] * positive_profile["personal_weight"]
            positive_profile = positive_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            positive_profile = positive_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_positive = positive_profile["weighted_relevance"].max()
            positive_profile["tag_weight"] = positive_profile["weighted_relevance"] / max_positive if max_positive > 0 else 0

        negative_profile = pd.DataFrame(columns=["tagId", "tag_weight"])
        if not negative_seed_ratings.empty:
            negative_profile = genome_scores.merge(negative_seed_ratings, on="movieId", how="inner")
            negative_profile["personal_weight"] = 3.0 - negative_profile["user_rating_5"]
            negative_profile["weighted_relevance"] = negative_profile["relevance"] * negative_profile["personal_weight"]
            negative_profile = negative_profile.groupby("tagId", as_index=False)["weighted_relevance"].sum()
            negative_profile = negative_profile.sort_values("weighted_relevance", ascending=False).head(100)
            max_negative = negative_profile["weighted_relevance"].max()
            negative_profile["tag_weight"] = negative_profile["weighted_relevance"] / max_negative if max_negative > 0 else 0

        if not positive_profile.empty and not candidate_genome.empty:
            positive_contrib = candidate_genome.merge(positive_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            positive_contrib["term_contribution"] = positive_contrib["relevance"] * positive_contrib["tag_weight"]
            raw_positive = positive_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["semantic_raw_score"] = candidates["movieId"].map(raw_positive).fillna(0.0)
            candidates["semantic_profile_score"] = normalize_positive_rank(candidates["semantic_raw_score"])
            positive_terms = positive_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["semantic_explanation_terms"] = candidates["movieId"].map(top_terms_from_contrib(positive_terms)).fillna("")
            top_positive_semantic_terms = positive_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

        if not negative_profile.empty and not candidate_genome.empty:
            negative_contrib = candidate_genome.merge(negative_profile[["tagId", "tag_weight"]], on="tagId", how="inner")
            negative_contrib["term_contribution"] = negative_contrib["relevance"] * negative_contrib["tag_weight"]
            raw_negative = negative_contrib.groupby("movieId")["term_contribution"].sum()
            candidates["negative_semantic_raw_score"] = candidates["movieId"].map(raw_negative).fillna(0.0)
            candidates["negative_semantic_score"] = normalize_positive_rank(candidates["negative_semantic_raw_score"])
            negative_terms = negative_contrib.merge(genome_tags, on="tagId", how="left")
            candidates["negative_semantic_terms"] = candidates["movieId"].map(top_terms_from_contrib(negative_terms)).fillna("")
            top_negative_semantic_terms = negative_profile.merge(genome_tags, on="tagId", how="left")["tag"].dropna().head(15).tolist()

elif semantic_source == "raw_tags_fallback":
    tags_raw = pd.read_csv(tags_path)
    if not {"movieId", "tag"}.issubset(tags_raw.columns):
        warnings.warn("tags.csv no tiene las columnas esperadas; se dejan scores semánticos en 0.")
        semantic_source = "none"
    else:
        tags_clean = build_clean_tags_dataframe(tags_raw)
        tag_stats = build_tag_statistics(tags_clean)
        reliable_tags = set(tag_stats.loc[tag_stats["is_reliable_tag"], "tag_clean"])
        tags_clean = tags_clean[tags_clean["tag_clean"].isin(reliable_tags)].copy()
        print(f"Filas finales de tags fiables: {len(tags_clean):,}")
        print(f"Tags únicos finales: {tags_clean['tag_clean'].nunique():,}")
        print(f"Películas con al menos un tag fiable: {tags_clean['movieId'].nunique():,}")
        positive_tag_profile, negative_tag_profile = build_semantic_profiles_from_tags(tags_clean, liked_movies, disliked_movies, tag_stats)
        semantic_scores = compute_semantic_scores_from_tags(candidates, tags_clean, positive_tag_profile, negative_tag_profile)
        candidates = candidates.drop(columns=[col for col in SEMANTIC_COLUMNS if col in candidates.columns]).merge(
            semantic_scores, on="movieId", how="left"
        )
        for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
            candidates[col] = candidates[col].fillna(0.0)
        for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
            candidates[col] = candidates[col].fillna("")
        top_positive_semantic_terms = positive_tag_profile["tag_clean"].dropna().head(20).tolist()
        top_negative_semantic_terms = negative_tag_profile["tag_clean"].dropna().head(20).tolist()

if semantic_source == "none":
    warnings.warn("No hay genome-scores/genome-tags ni tags.csv; los scores semánticos se dejan en 0.")

for col in ["semantic_raw_score", "negative_semantic_raw_score", "semantic_profile_score", "negative_semantic_score"]:
    candidates[col] = candidates[col].fillna(0.0)
for col in ["semantic_explanation_terms", "negative_semantic_terms"]:
    candidates[col] = candidates[col].fillna("")

FORBIDDEN_SEMANTIC_TERMS = [
    "hans zimmer", "tom hanks", "coen brothers", "christopher nolan", "pixar",
    "great acting", "good acting", "cult film", "inspirational", "touching", "slow paced",
]
found_forbidden_terms = set()
if "tag_clean" in positive_tag_profile.columns:
    found_forbidden_terms.update(set(positive_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
if "tag_clean" in negative_tag_profile.columns:
    found_forbidden_terms.update(set(negative_tag_profile["tag_clean"].dropna()) & set(FORBIDDEN_SEMANTIC_TERMS))
semantic_terms_text = " ".join(candidates["semantic_explanation_terms"].fillna("").astype(str).str.lower())
for term in FORBIDDEN_SEMANTIC_TERMS:
    if term in semantic_terms_text:
        found_forbidden_terms.add(term)
if found_forbidden_terms:
    print("Advertencia: siguen apareciendo tags que deberían haber sido filtrados por el 09.")
    print(sorted(found_forbidden_terms))

positive_semantic_count = (candidates["semantic_profile_score"] > 0).sum()
negative_semantic_count = (candidates["negative_semantic_score"] > 0).sum()
print(f"Fuente semántica usada final: {semantic_source}")
print(f"Candidatos con semantic_profile_score > 0: {positive_semantic_count:,} ({positive_semantic_count / len(candidates) * 100:.2f}%)")
print(f"Candidatos con negative_semantic_score > 0: {negative_semantic_count:,} ({negative_semantic_count / len(candidates) * 100:.2f}%)")
print("Top 20 tags positivos finales:")
print(top_positive_semantic_terms[:20])
print("Top 20 tags negativos finales:")
print(top_negative_semantic_terms[:20])


Archivos semánticos disponibles:
- tags_semantic_clean.csv: sí
- tag_semantic_stats.csv: sí
- tags.csv: sí
- genome-scores.csv: no
- genome-tags.csv: no
Fuente semántica usada: processed_tags
Tags limpios cargados: 50,897
Tags únicos limpios: 534
Películas con tags limpios: 16,795


C:\Users\alexo\AppData\Local\Temp\ipykernel_19276\1710737712.py:347: DtypeWarning: Columns (0: semantic_category, 1: semantic_match_reason, 2: rejection_reason_statistical) have mixed types. Specify dtype option on import or set low_memory=False.
  tag_stats = pd.read_csv(TAG_SEMANTIC_STATS_PATH)


,tag_clean,positive_weight,positive_n_seed_movies
232,tense,64.307744,12
182,psychology,54.468484,10
157,nostalgic,53.348209,7
93,existentialism,52.521941,8
12,ambiguous ending,52.335802,6
77,dreamlike,50.012098,8
229,surrealism,49.732575,10
238,time travel,46.161323,8
152,neo-noir,46.075157,8
237,time loop,45.113847,5


,tag_clean,negative_weight,negative_n_seed_movies
51,long takes,22.442782,2
83,shaky camera,14.091218,1
91,stylized violence,11.987656,2
39,gory,9.819927,2
98,tragic hero,9.819927,2
55,memory loss,9.401857,2
53,massachusetts institute of technology,9.031525,1
56,military industrial complex,8.032996,1
38,giant robots,8.032996,1
86,space program,7.932912,1


Tags en intersección positivo/negativo: ['absurd', 'absurdism', 'afterlife', 'amazing photography', 'android(s)/cyborg(s)', 'apocalypse', 'artificial intelligence', 'atmospheric', 'bad ending', 'black comedy', 'brutal', 'brutal violence', 'cinematography', 'class differences', 'colorful', 'coming of age', 'dark', 'dark comedy', 'dark hero', 'dark humor', 'disappointing ending', 'disturbing', 'dry humor', 'dystopia', 'emotional', 'ending', 'existential', 'existential crisis', 'faith', 'family relationships']
Número de tags positivos: 80
Número de tags negativos: 80
Fuente semántica usada final: processed_tags
Candidatos con semantic_profile_score > 0: 2,893 (59.49%)
Candidatos con negative_semantic_score > 0: 3,137 (64.51%)
Top 20 tags positivos finales:
['tense', 'psychology', 'nostalgic', 'existentialism', 'ambiguous ending', 'dreamlike', 'surrealism', 'time travel', 'neo-noir', 'time loop', 'creepy', 'morality', 'depressing', 'psychological', 'heartwarming', 'hope', 'confusing', 'bit

## 12. Scores de calidad y popularidad

In [13]:
candidates["rating_score"] = ((candidates["rating_mean"] - 3.0) / 2.0).clip(0, 1)

log_counts = np.log1p(candidates["rating_count"].clip(lower=0))
if np.isclose(log_counts.max(), log_counts.min()):
    candidates["popularity_score"] = 0.0
else:
    candidates["popularity_score"] = ((log_counts - log_counts.min()) / (log_counts.max() - log_counts.min())).clip(0, 1)

display(candidates[["title", "rating_mean", "rating_count", "rating_score", "popularity_score"]].head())

,title,rating_mean,rating_count,rating_score,popularity_score
0,Jumanji (1995),3.275758,28904,0.137879,0.782331
1,Heat (1995),3.868277,29490,0.434139,0.785770
2,Sabrina (1995),3.363968,13585,0.181984,0.652937
3,GoldenEye (1995),3.427850,32474,0.213925,0.802290
4,"American President, The (1995)",3.657160,19127,0.328580,0.711571


## 13. Fusionar señales

In [14]:
candidates_scored = candidates.merge(collab_scores, on="movieId", how="left")

numeric_collab_cols = [
    "collab_raw_score",
    "collab_positive_raw",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "n_collab_evidence",
]
for col in numeric_collab_cols:
    candidates_scored[col] = candidates_scored[col].fillna(0)

for col in ["similar_liked_movies", "similar_disliked_movies"]:
    candidates_scored[col] = candidates_scored[col].fillna("")

print(f"Candidatos con scoring fusionado: {len(candidates_scored):,}")

Candidatos con scoring fusionado: 4,863


## Diagnóstico adaptativo del perfil de usuario

In [15]:
def _clip01(value):
    return float(np.clip(value, 0.0, 1.0))


def rank_normalize_positive(series):
    raw_scores = pd.to_numeric(pd.Series(series), errors="coerce").fillna(0.0)
    normalized = pd.Series(0.0, index=raw_scores.index, dtype=float)
    positive_mask = raw_scores > 0
    if positive_mask.any():
        normalized.loc[positive_mask] = raw_scores.loc[positive_mask].rank(pct=True)
    return normalized.clip(0.0, 1.0)


def _rank_positive(raw_scores):
    return rank_normalize_positive(raw_scores)


def safe_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip().lower()


def contains_any(text, terms):
    text = safe_text(text)
    return any(safe_text(term) in text for term in terms if safe_text(term))


def _top_terms_from_weighted_tags(contrib_df, term_col="tag_clean", value_col="contribution", max_terms=5):
    if contrib_df.empty:
        return pd.Series(dtype=str)
    ordered = contrib_df.sort_values(["movieId", value_col], ascending=[True, False])
    return ordered.groupby("movieId")[term_col].apply(
        lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_terms))
    )


def main_genre_from_genres(genres):
    genre_list = split_genres(genres)
    return genre_list[0] if genre_list else "Unknown"


SEMANTIC_BRANCHES = [
    "psychological_thriller", "psychological_drama", "crime_thriller",
    "sci_fi_reflective", "surreal_fantasy", "animation_dark_surreal",
    "animation_family", "documentary_reflective", "comedy_satire",
    "romantic_comedy_drama", "dark_drama", "classic_adventure", "general",
]


def _row_text_terms(row):
    pieces = [
        row.get("core_semantic_explanation_terms", ""),
        row.get("semantic_explanation_terms", ""),
        row.get("core_negative_semantic_terms", ""),
    ]
    return " ".join(safe_text(piece) for piece in pieces if safe_text(piece))


def _branch_warning_for_row(row):
    genres = set(split_genres(row.get("genres", "")))
    branch = row.get("semantic_branch", "")
    all_text = (" ".join(sorted(genres)) + " " + _row_text_terms(row)).lower()
    thriller_genres = {"Thriller", "Mystery", "Horror", "Crime", "Sci-Fi"}
    very_strong = ["mindfuck", "paranoia", "unreliable narrator", "hallucination", "psychological thriller"]
    if {"Comedy", "Romance"}.issubset(genres) and branch == "psychological_thriller":
        return "comedy_romance_as_psychological_thriller"
    if branch == "psychological_thriller" and not (genres & thriller_genres) and not contains_any(all_text, very_strong):
        return "no_thriller_genre_as_psychological_thriller"
    if branch == "animation_family" and float(row.get("semantic_relevance_adjusted_score", 0.0) or 0.0) > 0.85:
        return "animation_family_high_score"
    return ""


def assign_semantic_branch(row):
    genres = set(split_genres(row.get("genres", "")))
    main_genre = row.get("main_genre", "")
    if pd.notna(main_genre) and str(main_genre).strip():
        genres.add(str(main_genre).strip())
    genres_text = " ".join(sorted(genres)).lower()
    tags_text = _row_text_terms(row)
    all_text = f"{genres_text} {tags_text}".strip()
    year = pd.to_numeric(row.get("year"), errors="coerce")

    def has(terms):
        return contains_any(all_text, terms)

    thriller_genres = {"Thriller", "Mystery", "Horror", "Crime", "Sci-Fi"}
    psych_strong = [
        "psychological", "psychological thriller", "paranoia", "mindfuck", "identity",
        "hallucination", "unreliable narrator", "obsession", "mental illness",
        "disturbing", "ambiguous ending", "twist ending", "cerebral",
        "nightmare", "surrealism", "dreamlike",
    ]
    psych_very_strong = [
        "psychological thriller", "mindfuck", "paranoia", "unreliable narrator",
        "hallucination", "identity crisis", "ambiguous ending", "disturbing",
    ]

    if "Documentary" in genres:
        return "documentary_reflective"
    if "Animation" in genres and has([
        "psychological", "psychological thriller", "paranoia", "mindfuck", "identity",
        "surrealism", "dreamlike", "disturbing", "hallucination",
        "ambiguous ending", "dark", "horror", "tragedy", "adult animation",
    ]):
        return "animation_dark_surreal"
    if "Animation" in genres or "Children" in genres:
        return "animation_family"
    if {"Romance", "Comedy"}.issubset(genres):
        return "romantic_comedy_drama"
    if {"Romance", "Drama"}.issubset(genres) and not (genres & thriller_genres):
        return "romantic_comedy_drama"
    if "Sci-Fi" in genres and has([
        "time travel", "artificial intelligence", "identity", "dystopia",
        "existentialism", "philosophical", "mindfuck", "cerebral",
        "ambiguous ending", "paranoia", "memory", "technology",
    ]):
        return "sci_fi_reflective"
    if ("Crime" in genres or "Thriller" in genres) and has([
        "tense", "investigation", "murder", "serial killer", "neo-noir",
        "police", "conspiracy", "revenge", "suspense", "mystery", "crime",
    ]):
        return "crime_thriller"

    romance_comedy_drama = {"Comedy", "Drama", "Romance"}.issubset(genres) or ({"Drama", "Romance"}.issubset(genres) and not (genres & thriller_genres))
    if ((genres & thriller_genres) and has(psych_strong)) or has(psych_very_strong):
        if not romance_comedy_drama or has(psych_very_strong):
            return "psychological_thriller"
    if has(["surrealism", "dreamlike", "magical realism", "weird", "fantasy", "fairy tale", "absurd", "bizarre"]):
        return "surreal_fantasy"
    if "Comedy" in genres and has(["satire", "satirical", "dark comedy", "black comedy", "absurd", "witty", "quirky"]):
        return "comedy_satire"
    if "Drama" in genres and not (genres & thriller_genres) and has([
        "psychology", "psychological", "mental illness", "depression", "grief",
        "family", "character study", "trauma", "loneliness", "identity",
    ]):
        return "psychological_drama"
    if "Drama" in genres and has(["dark", "bleak", "depressing", "tragedy", "tragic", "disturbing", "moral", "existentialism"]):
        return "dark_drama"
    if pd.notna(year) and year < 1980 and "Adventure" in genres:
        return "classic_adventure"
    return "general"

def _normalized_entropy(distribution):
    values = np.array([float(value) for value in distribution if float(value) > 0], dtype=float)
    if len(values) <= 1:
        return 0.0, 0.0
    probabilities = values / values.sum()
    entropy = float(-(probabilities * np.log(probabilities)).sum())
    return entropy, float(entropy / np.log(len(probabilities)))



def _top_tags_by_movie_for_branch(liked_movies, tags_clean=None, discriminative_tag_profile=None, max_tags=8):
    if tags_clean is None or tags_clean.empty or "movieId" not in tags_clean.columns or "tag_clean" not in tags_clean.columns:
        return pd.Series(dtype=str)
    liked_ids = liked_movies["movieId"].dropna().astype(int).unique() if "movieId" in liked_movies.columns else []
    movie_tags = tags_clean[tags_clean["movieId"].isin(liked_ids)][["movieId", "tag_clean"]].dropna().drop_duplicates().copy()
    if movie_tags.empty:
        return pd.Series(dtype=str)
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty and "discriminative_weight" in discriminative_tag_profile.columns:
        movie_tags = movie_tags.merge(discriminative_tag_profile[["tag_clean", "discriminative_weight"]], on="tag_clean", how="left")
        movie_tags["tag_weight"] = movie_tags["discriminative_weight"].fillna(0.0)
    else:
        movie_tags["tag_weight"] = 1.0
    movie_tags = movie_tags.sort_values(["movieId", "tag_weight", "tag_clean"], ascending=[True, False, True])
    return movie_tags.groupby("movieId")["tag_clean"].apply(lambda values: ", ".join(values.astype(str).drop_duplicates().head(max_tags)))


def build_user_branch_profile(liked_movies, tags_clean=None, discriminative_tag_profile=None, assign_branch_func=None):
    assign_branch_func = assign_branch_func or assign_semantic_branch
    liked = liked_movies.copy() if liked_movies is not None and not liked_movies.empty else pd.DataFrame()
    for col in ["movieId", "genres", "year", "user_rating_5"]:
        if col not in liked.columns:
            liked[col] = "" if col == "genres" else np.nan
    if "title" not in liked.columns:
        liked["title"] = ""
    liked["movieId"] = pd.to_numeric(liked["movieId"], errors="coerce")
    liked = liked.dropna(subset=["movieId"]).copy()
    liked["movieId"] = liked["movieId"].astype(int) if not liked.empty else liked.get("movieId", pd.Series(dtype=int))
    liked["user_rating_5"] = pd.to_numeric(liked["user_rating_5"], errors="coerce")
    if not liked.empty and "semantic_branch" not in liked.columns:
        top_tags = _top_tags_by_movie_for_branch(liked, tags_clean, discriminative_tag_profile)
        liked["core_semantic_explanation_terms"] = liked["movieId"].map(top_tags).fillna("")
        liked["semantic_explanation_terms"] = liked["core_semantic_explanation_terms"]
        if "main_genre" not in liked.columns:
            liked["main_genre"] = liked["genres"].apply(main_genre_from_genres)
        liked["semantic_branch"] = liked.apply(assign_branch_func, axis=1)
    if liked.empty:
        profile = pd.DataFrame(columns=["semantic_branch", "branch_weight", "n_seed_movies", "example_liked_movies"])
    else:
        liked["branch_seed_weight"] = (liked["user_rating_5"].fillna(4.0) - 3.0).clip(lower=0.5)
        profile = liked.groupby("semantic_branch", as_index=False).agg(
            branch_weight=("branch_seed_weight", "sum"),
            n_seed_movies=("movieId", "nunique"),
            example_liked_movies=("title", lambda values: " | ".join(values.astype(str).drop_duplicates().head(5))),
        )
    candidate_branches = set(candidates_scored["semantic_branch"].dropna().astype(str)) if "candidates_scored" in globals() and "semantic_branch" in candidates_scored.columns else set()
    all_branches = sorted(set(SEMANTIC_BRANCHES) | set(profile.get("semantic_branch", pd.Series(dtype=str))) | candidate_branches)
    profile = pd.DataFrame({"semantic_branch": all_branches}).merge(profile, on="semantic_branch", how="left")
    profile["branch_weight"] = pd.to_numeric(profile["branch_weight"], errors="coerce").fillna(0.0)
    profile["n_seed_movies"] = pd.to_numeric(profile["n_seed_movies"], errors="coerce").fillna(0).astype(int)
    profile["example_liked_movies"] = profile["example_liked_movies"].fillna("")
    total_weight = profile["branch_weight"].sum()
    profile["branch_share"] = profile["branch_weight"] / total_weight if total_weight > 0 else 0.0
    profile = profile.sort_values(["branch_share", "branch_weight", "semantic_branch"], ascending=[False, False, True]).reset_index(drop=True)
    profile["branch_rank"] = np.where(profile["branch_share"] > 0, np.arange(1, len(profile) + 1), np.nan)
    def classify_branch(row):
        share = row["branch_share"]
        rank = row["branch_rank"]
        if share <= 0:
            return "unseen"
        if share >= 0.18 or (pd.notna(rank) and rank <= 2):
            return "strong"
        if share >= 0.08:
            return "medium"
        if share >= 0.03:
            return "weak"
        return "unseen"
    profile["branch_strength"] = profile.apply(classify_branch, axis=1)
    return profile[["semantic_branch", "branch_weight", "branch_share", "branch_rank", "branch_strength", "n_seed_movies", "example_liked_movies"]]


def add_branch_affinity_to_candidates(candidates, user_branch_profile, branch_col="semantic_branch"):
    scored = candidates.copy()
    drop_cols = ["branch_share", "branch_strength", "n_branch_seed_movies", "branch_affinity_score", "branch_is_profile_supported"]
    scored = scored.drop(columns=[col for col in drop_cols if col in scored.columns])
    profile_cols = ["semantic_branch", "branch_share", "branch_strength", "n_seed_movies"]
    scored = scored.merge(user_branch_profile[profile_cols], left_on=branch_col, right_on="semantic_branch", how="left", suffixes=("", "_profile"))
    if "semantic_branch_profile" in scored.columns:
        scored["semantic_branch"] = scored["semantic_branch"].fillna(scored["semantic_branch_profile"])
        scored = scored.drop(columns=["semantic_branch_profile"])
    scored["branch_share"] = scored["branch_share"].fillna(0.0)
    scored["branch_strength"] = scored["branch_strength"].fillna("unseen")
    scored["n_branch_seed_movies"] = scored["n_seed_movies"].fillna(0).astype(int)
    scored = scored.drop(columns=["n_seed_movies"])
    base_scores = {"strong": 1.00, "medium": 0.75, "weak": 0.45, "unseen": 0.15}
    scored["branch_affinity_score"] = scored["branch_strength"].map(base_scores).fillna(0.15)
    scored["branch_affinity_score"] = np.maximum(scored["branch_affinity_score"], np.sqrt(scored["branch_share"].clip(lower=0.0)))
    scored["branch_affinity_score"] = scored["branch_affinity_score"].clip(0.05, 1.0)
    scored["branch_is_profile_supported"] = scored["branch_strength"].isin(["strong", "medium", "weak"])
    return scored


def build_discriminative_semantic_profile(
    tags_clean,
    liked_movies,
    disliked_movies,
    positive_tag_profile=None,
    negative_tag_profile=None,
    smoothing=0.10,
    min_positive_seed_movies=1,
    beta_negative=0.75,
):
    required_cols = {"movieId", "tag_clean", "idf"}
    if tags_clean is None or tags_clean.empty or not required_cols.issubset(tags_clean.columns):
        return pd.DataFrame(columns=[
            "tag_clean", "semantic_category", "positive_weight", "negative_weight",
            "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
            "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
            "discriminative_weight_raw", "discriminative_weight",
            "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
        ])

    tag_cols = ["movieId", "tag_clean", "idf"] + (["semantic_category"] if "semantic_category" in tags_clean.columns else [])
    tag_features = tags_clean[tag_cols].copy()
    tag_features["movieId"] = pd.to_numeric(tag_features["movieId"], errors="coerce")
    tag_features["idf"] = pd.to_numeric(tag_features["idf"], errors="coerce").fillna(1.0)
    tag_features = tag_features.dropna(subset=["movieId", "tag_clean"])
    tag_features["movieId"] = tag_features["movieId"].astype(int)
    tag_features = tag_features.drop_duplicates(["movieId", "tag_clean"])

    positive_seed = liked_movies.loc[liked_movies["user_rating_5"] >= 4.0, ["movieId", "user_rating_5"]].dropna().copy()
    negative_seed = disliked_movies.loc[disliked_movies["user_rating_5"] <= 2.5, ["movieId", "user_rating_5"]].dropna().copy()
    positive_seed["movieId"] = positive_seed["movieId"].astype(int)
    negative_seed["movieId"] = negative_seed["movieId"].astype(int)

    positive_agg = pd.DataFrame(columns=["tag_clean", "positive_weight", "positive_n_seed_movies", "semantic_category"])
    if not positive_seed.empty:
        positive_tags = tag_features.merge(positive_seed, on="movieId", how="inner")
        positive_tags["positive_personal_weight"] = positive_tags["user_rating_5"] - 3.0
        positive_tags["contribution_positive"] = positive_tags["positive_personal_weight"] * positive_tags["idf"]
        agg_spec = {
            "positive_weight": ("contribution_positive", "sum"),
            "positive_n_seed_movies": ("movieId", "nunique"),
        }
        if "semantic_category" in positive_tags.columns:
            agg_spec["semantic_category"] = ("semantic_category", "first")
        positive_agg = positive_tags.groupby("tag_clean", as_index=False).agg(**agg_spec)
        if "semantic_category" not in positive_agg.columns:
            positive_agg["semantic_category"] = None

    negative_agg = pd.DataFrame(columns=["tag_clean", "negative_weight", "negative_n_seed_movies"])
    if not negative_seed.empty:
        negative_tags = tag_features.merge(negative_seed, on="movieId", how="inner")
        negative_tags["negative_personal_weight"] = 3.0 - negative_tags["user_rating_5"]
        negative_tags["contribution_negative"] = negative_tags["negative_personal_weight"] * negative_tags["idf"]
        negative_agg = negative_tags.groupby("tag_clean", as_index=False).agg(
            negative_weight=("contribution_negative", "sum"),
            negative_n_seed_movies=("movieId", "nunique"),
        )

    profile = positive_agg.merge(negative_agg, on="tag_clean", how="outer")
    if profile.empty:
        return pd.DataFrame(columns=[
            "tag_clean", "semantic_category", "positive_weight", "negative_weight",
            "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
            "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
            "discriminative_weight_raw", "discriminative_weight",
            "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
        ])

    for col in ["positive_weight", "negative_weight", "positive_n_seed_movies", "negative_n_seed_movies"]:
        profile[col] = pd.to_numeric(profile[col], errors="coerce").fillna(0.0)
    if "semantic_category" not in profile.columns:
        profile["semantic_category"] = None

    positive_total = profile["positive_weight"].sum()
    negative_total = profile["negative_weight"].sum()
    profile["positive_weight_norm"] = profile["positive_weight"] / positive_total if positive_total > 0 else 0.0
    profile["negative_weight_norm"] = profile["negative_weight"] / negative_total if negative_total > 0 else 0.0
    profile["tag_ambiguity"] = np.minimum(profile["positive_weight_norm"], profile["negative_weight_norm"])
    profile["discriminative_lift"] = np.log(
        (profile["positive_weight_norm"] + smoothing) / (profile["negative_weight_norm"] + smoothing)
    )
    profile["discriminative_weight_raw"] = (
        profile["positive_weight_norm"]
        * profile["discriminative_lift"].clip(lower=0.0)
        * (1 - profile["tag_ambiguity"])
    )

    many_positive_movies = len(positive_seed) >= 10
    profile.loc[many_positive_movies & (profile["positive_n_seed_movies"] == 1), "discriminative_weight_raw"] *= 0.70
    profile.loc[profile["positive_n_seed_movies"] >= 3, "discriminative_weight_raw"] *= 1.10
    profile.loc[profile["tag_ambiguity"] > 0.20, "discriminative_weight_raw"] *= 0.50
    profile.loc[profile["tag_ambiguity"] > 0.35, "discriminative_weight_raw"] *= 0.25
    profile.loc[profile["positive_n_seed_movies"] < min_positive_seed_movies, "discriminative_weight_raw"] = 0.0

    max_positive = profile["discriminative_weight_raw"].max()
    profile["discriminative_weight"] = profile["discriminative_weight_raw"] / max_positive if max_positive > 0 else 0.0

    profile["negative_discriminative_lift"] = np.log(
        (profile["negative_weight_norm"] + smoothing) / (profile["positive_weight_norm"] + smoothing)
    )
    profile["negative_discriminative_weight_raw"] = (
        profile["negative_weight_norm"]
        * profile["negative_discriminative_lift"].clip(lower=0.0)
        * (1 - profile["tag_ambiguity"])
    )
    max_negative = profile["negative_discriminative_weight_raw"].max()
    profile["negative_discriminative_weight"] = profile["negative_discriminative_weight_raw"] / max_negative if max_negative > 0 else 0.0

    profile["is_core_positive_tag"] = profile["discriminative_weight"] > 0
    profile["is_core_negative_tag"] = profile["negative_discriminative_weight"] > 0

    return profile[[
        "tag_clean", "semantic_category", "positive_weight", "negative_weight",
        "positive_weight_norm", "negative_weight_norm", "positive_n_seed_movies",
        "negative_n_seed_movies", "tag_ambiguity", "discriminative_lift",
        "discriminative_weight_raw", "discriminative_weight",
        "negative_discriminative_weight", "is_core_positive_tag", "is_core_negative_tag",
    ]].sort_values("discriminative_weight", ascending=False).reset_index(drop=True)


def compute_semantic_core_scores(candidates, tags_clean, discriminative_tag_profile):
    result = candidates[["movieId"]].copy()
    result["semantic_core_raw_score"] = 0.0
    result["semantic_core_negative_raw_score"] = 0.0
    result["semantic_core_score"] = 0.0
    result["semantic_core_negative_score"] = 0.0
    result["core_semantic_explanation_terms"] = ""
    result["core_negative_semantic_terms"] = ""

    if tags_clean is None or tags_clean.empty or discriminative_tag_profile is None or discriminative_tag_profile.empty:
        return result

    candidate_tags = tags_clean[tags_clean["movieId"].isin(result["movieId"].astype(int))][["movieId", "tag_clean"]].drop_duplicates()
    if candidate_tags.empty:
        return result

    profile_cols = ["tag_clean", "discriminative_weight", "negative_discriminative_weight"]
    weighted_tags = candidate_tags.merge(discriminative_tag_profile[profile_cols], on="tag_clean", how="inner")
    if weighted_tags.empty:
        return result

    positive_contrib = weighted_tags[weighted_tags["discriminative_weight"].fillna(0.0) > 0].copy()
    if not positive_contrib.empty:
        positive_contrib["contribution"] = positive_contrib["discriminative_weight"]
        raw_positive = positive_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_core_raw_score"] = result["movieId"].map(raw_positive).fillna(0.0)
        result["semantic_core_score"] = _rank_positive(result["semantic_core_raw_score"])
        result["core_semantic_explanation_terms"] = result["movieId"].map(
            _top_terms_from_weighted_tags(positive_contrib, value_col="contribution", max_terms=5)
        ).fillna("")

    negative_contrib = weighted_tags[weighted_tags["negative_discriminative_weight"].fillna(0.0) > 0].copy()
    if not negative_contrib.empty:
        negative_contrib["contribution"] = negative_contrib["negative_discriminative_weight"]
        raw_negative = negative_contrib.groupby("movieId")["contribution"].sum()
        result["semantic_core_negative_raw_score"] = result["movieId"].map(raw_negative).fillna(0.0)
        result["semantic_core_negative_score"] = _rank_positive(result["semantic_core_negative_raw_score"])
        result["core_negative_semantic_terms"] = result["movieId"].map(
            _top_terms_from_weighted_tags(negative_contrib, value_col="contribution", max_terms=5)
        ).fillna("")

    return result


def compute_semantic_net_score(df, beta=0.75):
    scored = df.copy()
    if {"semantic_core_raw_score", "semantic_core_negative_raw_score"}.issubset(scored.columns):
        semantic_raw = scored["semantic_core_raw_score"]
        negative_raw = scored["semantic_core_negative_raw_score"]
    else:
        semantic_raw = scored["semantic_raw_score"] if "semantic_raw_score" in scored.columns else scored.get("semantic_profile_score", 0.0)
        negative_raw = scored["negative_semantic_raw_score"] if "negative_semantic_raw_score" in scored.columns else scored.get("negative_semantic_score", 0.0)
    scored["semantic_net_raw_score"] = semantic_raw.fillna(0.0) - beta * negative_raw.fillna(0.0)
    scored["semantic_net_score"] = _rank_positive(scored["semantic_net_raw_score"])
    return scored


def _first_weight_column(df, preferred):
    if df is None or df.empty:
        return None
    for col in preferred:
        if col in df.columns:
            return col
    numeric_cols = [col for col in df.select_dtypes(include=[np.number]).columns if not col.lower().endswith("id")]
    return numeric_cols[0] if numeric_cols else None


def _normalized_tag_weights(df, preferred_weight_cols):
    if df is None or df.empty or "tag_clean" not in df.columns:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weight_col = _first_weight_column(df, preferred_weight_cols)
    if weight_col is None:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights = df[["tag_clean", weight_col]].dropna().copy()
    weights[weight_col] = pd.to_numeric(weights[weight_col], errors="coerce").fillna(0.0).clip(lower=0.0)
    weights = weights.groupby("tag_clean", as_index=False)[weight_col].sum()
    total = weights[weight_col].sum()
    if total <= 0:
        return pd.DataFrame(columns=["tag_clean", "weight_norm"])
    weights["weight_norm"] = weights[weight_col] / total
    return weights[["tag_clean", "weight_norm"]]


def _semantic_ambiguity(positive_tag_profile, negative_tag_profile):
    positive_weights = _normalized_tag_weights(positive_tag_profile, ["positive_weight", "tag_weight", "weight"])
    negative_weights = _normalized_tag_weights(negative_tag_profile, ["negative_weight", "tag_weight", "weight"])
    if positive_weights.empty or negative_weights.empty:
        return 1.0, 0
    overlap = positive_weights.merge(negative_weights, on="tag_clean", how="inner", suffixes=("_pos", "_neg"))
    if overlap.empty:
        return 0.0, 0
    ambiguity = np.minimum(overlap["weight_norm_pos"], overlap["weight_norm_neg"]).sum()
    return _clip01(ambiguity), len(overlap)


def extract_genre_weights_from_profile(df, rating_col="user_rating_5", positive=True):
    if df is None or df.empty or "genres" not in df.columns or rating_col not in df.columns:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    rows = []
    for _, row in df.dropna(subset=[rating_col]).iterrows():
        base_weight = row[rating_col] - 3.0 if positive else 3.0 - row[rating_col]
        if base_weight <= 0:
            continue
        for genre in split_genres(row.get("genres", "")):
            rows.append({"genre": genre, "weight": base_weight})
    if not rows:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights = pd.DataFrame(rows).groupby("genre", as_index=False)["weight"].sum()
    total = weights["weight"].sum()
    if total <= 0:
        return pd.DataFrame(columns=["genre", "genre_weight_norm"])
    weights["genre_weight_norm"] = weights["weight"] / total
    return weights[["genre", "genre_weight_norm"]]



def _valid_year_series(df, year_col="year"):
    if df is None or df.empty or year_col not in df.columns:
        return pd.Series(dtype=float)
    years = pd.to_numeric(df[year_col], errors="coerce")
    return years[(years >= 1874) & (years <= 2035)].dropna()



def build_movie_semantic_documents(movies_df, tags_clean, max_tags_per_movie=30):
    docs = movies_df[["movieId", "genres"]].copy()
    docs["movieId"] = pd.to_numeric(docs["movieId"], errors="coerce")
    docs = docs.dropna(subset=["movieId"]).copy()
    docs["movieId"] = docs["movieId"].astype(int)
    docs["genres_text"] = docs["genres"].fillna("").astype(str).str.replace("|", " ", regex=False)

    if tags_clean is not None and not tags_clean.empty and {"movieId", "tag_clean"}.issubset(tags_clean.columns):
        tag_cols = ["movieId", "tag_clean"]
        if "semantic_category" in tags_clean.columns:
            tag_cols.append("semantic_category")
        if "idf" in tags_clean.columns:
            tag_cols.append("idf")
        tags_doc = tags_clean[tag_cols].copy()
        tags_doc["movieId"] = pd.to_numeric(tags_doc["movieId"], errors="coerce")
        tags_doc = tags_doc.dropna(subset=["movieId", "tag_clean"]).copy()
        tags_doc["movieId"] = tags_doc["movieId"].astype(int)
        tags_doc["idf"] = pd.to_numeric(tags_doc["idf"], errors="coerce").fillna(1.0) if "idf" in tags_doc.columns else 1.0
        tags_doc = tags_doc.sort_values(["movieId", "idf", "tag_clean"], ascending=[True, False, True])
        tags_doc = tags_doc.groupby("movieId").head(max_tags_per_movie)
        tags_text = tags_doc.groupby("movieId")["tag_clean"].apply(lambda s: " ".join(s.astype(str))).rename("tags_text").reset_index()
        docs = docs.merge(tags_text, on="movieId", how="left")
        if "semantic_category" in tags_doc.columns:
            categories_text = tags_doc.dropna(subset=["semantic_category"]).groupby("movieId")["semantic_category"].apply(
                lambda s: " ".join(s.astype(str).drop_duplicates())
            ).rename("semantic_categories_text").reset_index()
            docs = docs.merge(categories_text, on="movieId", how="left")
        else:
            docs["semantic_categories_text"] = ""
    else:
        docs["tags_text"] = ""
        docs["semantic_categories_text"] = ""

    for col in ["tags_text", "semantic_categories_text"]:
        if col not in docs.columns:
            docs[col] = ""
        docs[col] = docs[col].fillna("").astype(str)
    docs["semantic_document"] = (
        docs["genres_text"] + " " + docs["tags_text"] + " " + docs["semantic_categories_text"]
    ).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    return docs[["movieId", "semantic_document"]]


def build_latent_semantic_space(movie_documents, n_components=64, max_features=8000, min_df=2, random_state=42):
    documents = movie_documents["semantic_document"].fillna("").astype(str)
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        min_df=min_df,
        ngram_range=(1, 2),
        sublinear_tf=True,
        stop_words="english",
    )
    tfidf_matrix = vectorizer.fit_transform(documents)
    n_movies, n_features = tfidf_matrix.shape
    movie_ids = movie_documents["movieId"].astype(int).tolist()
    movie_id_to_latent_idx = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}
    latent_idx_to_movie_id = {idx: movie_id for idx, movie_id in enumerate(movie_ids)}
    n_components_safe = min(n_components, n_features - 1, n_movies - 1)
    if n_components_safe >= 2:
        svd_model = TruncatedSVD(n_components=n_components_safe, random_state=random_state)
        latent_vectors = normalize(svd_model.fit_transform(tfidf_matrix))
    else:
        svd_model = None
        latent_vectors = normalize(tfidf_matrix)
    return latent_vectors, vectorizer, svd_model, movie_id_to_latent_idx, latent_idx_to_movie_id


def _latent_seed_indices(profile_df, movie_id_to_latent_idx, positive=True):
    if profile_df is None or profile_df.empty or "movieId" not in profile_df.columns:
        return [], np.array([], dtype=float), []
    work = profile_df.copy()
    work["movieId"] = pd.to_numeric(work["movieId"], errors="coerce")
    if "user_rating_5" in work.columns:
        work["user_rating_5"] = pd.to_numeric(work["user_rating_5"], errors="coerce")
        work = work[work["user_rating_5"] >= 4.0] if positive else work[work["user_rating_5"] <= 2.5]
        weights = (work["user_rating_5"] - 3.0) if positive else (3.0 - work["user_rating_5"])
        work["latent_weight"] = weights.fillna(1.0).clip(lower=0.05)
    else:
        work["latent_weight"] = 1.0
    work = work.dropna(subset=["movieId"]).copy()
    indices, seed_weights, titles = [], [], []
    for _, row in work.iterrows():
        movie_id = int(row["movieId"])
        if movie_id in movie_id_to_latent_idx:
            indices.append(movie_id_to_latent_idx[movie_id])
            seed_weights.append(float(row.get("latent_weight", 1.0)))
            titles.append(str(row.get("title", f"movieId={movie_id}")))
    return indices, np.array(seed_weights, dtype=float), titles


def compute_latent_user_similarity_scores(
    candidates,
    liked_movies,
    disliked_movies,
    latent_vectors,
    movie_id_to_latent_idx,
    top_k_positive=5,
    top_k_negative=5,
):
    result = candidates[["movieId"]].copy()
    result["latent_core_similarity_raw"] = 0.0
    result["latent_core_similarity_score"] = 0.0
    result["negative_latent_similarity_raw"] = 0.0
    result["negative_latent_similarity_score"] = 0.0
    result["nearest_liked_movies_latent"] = ""
    result["nearest_disliked_movies_latent"] = ""
    result["n_positive_latent_neighbors"] = 0
    result["n_negative_latent_neighbors"] = 0

    candidate_ids = pd.to_numeric(result["movieId"], errors="coerce").astype("Int64")
    candidate_positions = [pos for pos, movie_id in enumerate(candidate_ids) if pd.notna(movie_id) and int(movie_id) in movie_id_to_latent_idx]
    candidate_indices = [movie_id_to_latent_idx[int(candidate_ids.iloc[pos])] for pos in candidate_positions]
    if not candidate_indices:
        return result
    candidate_vectors = latent_vectors[candidate_indices]

    def fill_similarity(profile_df, positive=True):
        seed_indices, seed_weights, seed_titles = _latent_seed_indices(profile_df, movie_id_to_latent_idx, positive=positive)
        if not seed_indices:
            return
        sims = cosine_similarity(candidate_vectors, latent_vectors[seed_indices])
        top_k = min(top_k_positive if positive else top_k_negative, sims.shape[1])
        raw_scores, nearest_titles = [], []
        for row_sims in sims:
            order = np.argsort(row_sims)[::-1][:top_k]
            top_sims = row_sims[order]
            top_weights = seed_weights[order]
            denom = top_weights.sum()
            mean_topk = float(np.average(top_sims, weights=top_weights)) if denom > 0 else float(top_sims.mean())
            max_topk = float(top_sims.max()) if len(top_sims) else 0.0
            raw_scores.append(0.70 * mean_topk + 0.30 * max_topk)
            title_order = order[:3]
            nearest_titles.append(" | ".join(seed_titles[i] for i in title_order if i < len(seed_titles)))
        target_raw = "latent_core_similarity_raw" if positive else "negative_latent_similarity_raw"
        target_score = "latent_core_similarity_score" if positive else "negative_latent_similarity_score"
        target_titles = "nearest_liked_movies_latent" if positive else "nearest_disliked_movies_latent"
        target_n = "n_positive_latent_neighbors" if positive else "n_negative_latent_neighbors"
        raw_series = pd.Series(raw_scores, index=result.index[candidate_positions])
        result.loc[raw_series.index, target_raw] = raw_series
        result[target_score] = _rank_positive(result[target_raw])
        result.loc[raw_series.index, target_titles] = nearest_titles
        result.loc[raw_series.index, target_n] = len(seed_indices)

    fill_similarity(liked_movies, positive=True)
    fill_similarity(disliked_movies, positive=False)
    return result


def compute_positive_anchor_evidence(
    liked_movies,
    tags_clean,
    discriminative_tag_profile,
    user_branch_profile=None,
    assign_branch_func=None,
):
    output_cols = [
        "movieId", "title", "year", "genres", "user_rating_5",
        "semantic_branch", "branch_share", "branch_strength",
        "core_tag_weight_sum", "core_tag_count", "core_category_count",
        "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum",
        "centrality_score", "anchor_weight",
    ]
    if liked_movies is None or liked_movies.empty or "movieId" not in liked_movies.columns:
        return pd.DataFrame(columns=output_cols)

    liked = liked_movies.copy()
    liked["movieId"] = pd.to_numeric(liked["movieId"], errors="coerce")
    liked["user_rating_5"] = pd.to_numeric(liked.get("user_rating_5"), errors="coerce")
    liked = liked.dropna(subset=["movieId", "user_rating_5"]).copy()
    if liked.empty:
        return pd.DataFrame(columns=output_cols)
    liked["movieId"] = liked["movieId"].astype(int)

    positive_seed = liked[liked["user_rating_5"] >= 4.0].copy()
    if len(positive_seed) < 8:
        positive_seed = liked[liked["user_rating_5"] >= 3.5].copy()
    if positive_seed.empty:
        return pd.DataFrame(columns=output_cols)

    base_cols = [col for col in ["movieId", "title", "year", "genres", "user_rating_5", "semantic_branch"] if col in positive_seed.columns]
    movie_base = positive_seed[base_cols].drop_duplicates("movieId").copy()
    for col in ["title", "year", "genres", "semantic_branch"]:
        if col not in movie_base.columns:
            movie_base[col] = "" if col in ["title", "genres", "semantic_branch"] else np.nan

    if tags_clean is None or tags_clean.empty or not {"movieId", "tag_clean"}.issubset(tags_clean.columns):
        evidence = movie_base.copy()
        for col in ["core_tag_weight_sum", "core_tag_count", "core_category_count", "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum"]:
            evidence[col] = 0.0
    else:
        tag_cols = ["movieId", "tag_clean"]
        for col in ["semantic_category", "idf"]:
            if col in tags_clean.columns:
                tag_cols.append(col)
        movie_tags = tags_clean[tag_cols].copy()
        movie_tags["movieId"] = pd.to_numeric(movie_tags["movieId"], errors="coerce")
        movie_tags = movie_tags.dropna(subset=["movieId", "tag_clean"]).copy()
        movie_tags["movieId"] = movie_tags["movieId"].astype(int)
        if "semantic_category" not in movie_tags.columns:
            movie_tags["semantic_category"] = np.nan
        if "idf" not in movie_tags.columns:
            movie_tags["idf"] = 1.0
        movie_tags["idf"] = pd.to_numeric(movie_tags["idf"], errors="coerce").fillna(1.0)
        movie_tags = movie_tags.drop_duplicates(["movieId", "tag_clean"])
        movie_tags = movie_tags.merge(movie_base[["movieId"]], on="movieId", how="inner")

        profile_cols = ["tag_clean"]
        if discriminative_tag_profile is not None and not discriminative_tag_profile.empty:
            for col in ["discriminative_weight", "negative_discriminative_weight", "tag_ambiguity", "semantic_category"]:
                if col in discriminative_tag_profile.columns:
                    profile_cols.append(col)
            weighted_tags = movie_tags.merge(discriminative_tag_profile[profile_cols].drop_duplicates("tag_clean"), on="tag_clean", how="left", suffixes=("", "_profile"))
        else:
            weighted_tags = movie_tags.copy()
        if "semantic_category_profile" in weighted_tags.columns:
            weighted_tags["semantic_category"] = weighted_tags["semantic_category"].fillna(weighted_tags["semantic_category_profile"])
            weighted_tags = weighted_tags.drop(columns=["semantic_category_profile"])
        for col in ["discriminative_weight", "negative_discriminative_weight", "tag_ambiguity"]:
            if col not in weighted_tags.columns:
                weighted_tags[col] = 0.0
            weighted_tags[col] = pd.to_numeric(weighted_tags[col], errors="coerce").fillna(0.0)

        if weighted_tags.empty:
            evidence = movie_base.copy()
            for col in ["core_tag_weight_sum", "core_tag_count", "core_category_count", "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum"]:
                evidence[col] = 0.0
        else:
            useful_tags = weighted_tags[weighted_tags["discriminative_weight"] > 0].copy()
            category_counts = useful_tags.dropna(subset=["semantic_category"]).groupby("movieId")["semantic_category"].nunique()
            top_tags = weighted_tags.sort_values(["movieId", "discriminative_weight", "idf", "tag_clean"], ascending=[True, False, False, True]).groupby("movieId")["tag_clean"].apply(
                lambda values: ", ".join(values.astype(str).drop_duplicates().head(8))
            )
            grouped = weighted_tags.groupby("movieId").agg(
                core_tag_weight_sum=("discriminative_weight", "sum"),
                core_tag_weight_mean=("discriminative_weight", "mean"),
                core_tag_count=("discriminative_weight", lambda values: int((pd.to_numeric(values, errors="coerce").fillna(0.0) > 0).sum())),
                mean_idf=("idf", "mean"),
                ambiguity_mean=("tag_ambiguity", "mean"),
                negative_tag_overlap_sum=("negative_discriminative_weight", "sum"),
            ).reset_index()
            grouped["core_category_count"] = grouped["movieId"].map(category_counts).fillna(0).astype(int)
            grouped["top_tags"] = grouped["movieId"].map(top_tags).fillna("")
            evidence = movie_base.merge(grouped, on="movieId", how="left")

    for col in ["core_tag_weight_sum", "core_tag_weight_mean", "core_tag_count", "core_category_count", "mean_idf", "ambiguity_mean", "negative_tag_overlap_sum"]:
        if col not in evidence.columns:
            evidence[col] = 0.0
        fill_value = 1.0 if col == "mean_idf" else 0.0
        evidence[col] = pd.to_numeric(evidence[col], errors="coerce").fillna(fill_value)
    if "top_tags" not in evidence.columns:
        evidence["top_tags"] = ""

    evidence["core_tag_weight_sum_norm"] = rank_normalize_positive(evidence["core_tag_weight_sum"])
    evidence["core_tag_count_norm"] = rank_normalize_positive(evidence["core_tag_count"])
    evidence["core_category_count_norm"] = rank_normalize_positive(evidence["core_category_count"])
    evidence["mean_idf_norm"] = rank_normalize_positive(evidence["mean_idf"])
    evidence["negative_tag_overlap_norm"] = rank_normalize_positive(evidence["negative_tag_overlap_sum"])

    if "semantic_branch" not in evidence.columns or evidence["semantic_branch"].fillna("").eq("").all():
        evidence["semantic_branch"] = "general"
    if assign_branch_func is not None:
        evidence["core_semantic_explanation_terms"] = evidence["top_tags"].fillna("")
        evidence["semantic_explanation_terms"] = evidence["top_tags"].fillna("")
        if "main_genre" not in evidence.columns:
            evidence["main_genre"] = evidence["genres"].apply(main_genre_from_genres)
        missing_branch = evidence["semantic_branch"].isna() | evidence["semantic_branch"].astype(str).str.strip().eq("") | evidence["semantic_branch"].eq("general")
        evidence.loc[missing_branch, "semantic_branch"] = evidence.loc[missing_branch].apply(assign_branch_func, axis=1)
    evidence["semantic_branch"] = evidence["semantic_branch"].fillna("general").astype(str)

    if user_branch_profile is not None and not user_branch_profile.empty and "semantic_branch" in user_branch_profile.columns:
        branch_cols = [col for col in ["semantic_branch", "branch_share", "branch_strength"] if col in user_branch_profile.columns]
        evidence = evidence.drop(columns=[col for col in ["branch_share", "branch_strength"] if col in evidence.columns]).merge(
            user_branch_profile[branch_cols].drop_duplicates("semantic_branch"), on="semantic_branch", how="left"
        )
    if "branch_share" not in evidence.columns:
        evidence["branch_share"] = 0.5
    if "branch_strength" not in evidence.columns:
        evidence["branch_strength"] = "unknown"
    evidence["branch_share"] = pd.to_numeric(evidence["branch_share"], errors="coerce").fillna(0.5).clip(0.0, 1.0)
    evidence["branch_strength"] = evidence["branch_strength"].fillna("unknown").astype(str)

    evidence["rating_weight"] = (pd.to_numeric(evidence["user_rating_5"], errors="coerce").fillna(3.5) - 3.0).clip(lower=0.5)
    evidence["centrality_score"] = (
        0.30 * evidence["core_tag_weight_sum_norm"]
        + 0.20 * evidence["core_category_count_norm"]
        + 0.15 * evidence["mean_idf_norm"]
        + 0.15 * evidence["core_tag_count_norm"]
        + 0.15 * evidence["branch_share"]
        - 0.15 * evidence["negative_tag_overlap_norm"]
        - 0.10 * evidence["ambiguity_mean"].clip(0.0, 1.0)
    ).clip(0.0, 1.0)
    evidence["anchor_weight"] = evidence["rating_weight"] * (0.30 + 0.70 * evidence["centrality_score"])
    sparse_anchor_mask = (evidence["core_tag_count"] <= 1) & (evidence["core_category_count"] <= 1)
    evidence.loc[sparse_anchor_mask, "anchor_weight"] *= 0.60
    evidence.loc[evidence["branch_strength"].eq("unseen"), "anchor_weight"] *= 0.40

    return evidence[output_cols].sort_values("anchor_weight", ascending=False).reset_index(drop=True)


def select_core_positive_anchors(positive_anchor_evidence, top_n_anchors=30, min_anchors=12):
    output_cols = [
        "movieId", "title", "year", "genres", "user_rating_5", "semantic_branch",
        "branch_share", "branch_strength", "centrality_score", "anchor_weight",
        "is_core_anchor", "anchor_rank", "branch_anchor_limit", "selected_reason",
    ]
    if positive_anchor_evidence is None or positive_anchor_evidence.empty:
        return pd.DataFrame(columns=output_cols)
    ordered = positive_anchor_evidence.sort_values("anchor_weight", ascending=False).reset_index(drop=True).copy()

    def branch_limit(row):
        share = float(row.get("branch_share", 0.0) or 0.0)
        strength = str(row.get("branch_strength", "unknown"))
        expected_slots = top_n_anchors * share
        if strength == "strong":
            limit = max(5, int(np.ceil(expected_slots + 2)))
        elif strength == "medium":
            limit = min(7, max(3, int(np.ceil(expected_slots + 1))))
        elif strength == "weak":
            limit = min(3, max(1, int(np.ceil(expected_slots))))
        elif strength == "unseen":
            limit = 1
        else:
            limit = 3
        return int(max(1, min(top_n_anchors, limit)))

    ordered["branch_anchor_limit"] = ordered.apply(branch_limit, axis=1)
    selected_indices = []
    selected_reasons = {}
    branch_counts = {}
    for idx, row in ordered.iterrows():
        branch = row.get("semantic_branch", "general")
        if branch_counts.get(branch, 0) >= int(row.get("branch_anchor_limit", 1)):
            continue
        selected_indices.append(idx)
        selected_reasons[idx] = "core_anchor"
        branch_counts[branch] = branch_counts.get(branch, 0) + 1
        if len(selected_indices) >= top_n_anchors:
            break
    min_target = min(min_anchors, len(ordered), top_n_anchors)
    if len(selected_indices) < min_target:
        for idx, _ in ordered.iterrows():
            if idx in selected_indices:
                continue
            selected_indices.append(idx)
            selected_reasons[idx] = "minimum_fill"
            if len(selected_indices) >= min_target:
                break
    selected = ordered.loc[selected_indices].copy()
    selected["is_core_anchor"] = True
    selected["anchor_rank"] = np.arange(1, len(selected) + 1)
    selected["selected_reason"] = selected.index.map(selected_reasons).fillna("core_anchor")
    return selected[output_cols].sort_values("anchor_rank").reset_index(drop=True)


def _dense_latent_rows(latent_vectors, indices):
    rows = latent_vectors[indices]
    return rows.toarray() if hasattr(rows, "toarray") else np.asarray(rows)


def compute_latent_core_preference_scores(candidates, core_positive_anchors, disliked_movies, latent_vectors, movie_id_to_latent_idx, user_branch_profile=None, top_k_positive=5, top_k_negative=5):
    result = candidates[["movieId"]].copy()
    defaults = {
        "positive_centroid_similarity": 0.0, "topk_anchor_similarity": 0.0,
        "branch_prototype_similarity": 0.0, "negative_latent_preference_raw": 0.0,
        "negative_latent_preference_score": 0.0, "latent_core_preference_raw": 0.0,
        "latent_core_preference_score": 0.0, "nearest_core_anchor_movies": "",
        "nearest_negative_movies_latent": "", "n_core_anchors_used": 0, "n_negative_anchors_used": 0,
    }
    for col, default in defaults.items():
        result[col] = default
    if candidates is None or candidates.empty or core_positive_anchors is None or core_positive_anchors.empty:
        return result

    candidate_ids = pd.to_numeric(result["movieId"], errors="coerce").astype("Int64")
    candidate_positions = [pos for pos, movie_id in enumerate(candidate_ids) if pd.notna(movie_id) and int(movie_id) in movie_id_to_latent_idx]
    if not candidate_positions:
        return result
    candidate_indices = [movie_id_to_latent_idx[int(candidate_ids.iloc[pos])] for pos in candidate_positions]
    candidate_vectors = _dense_latent_rows(latent_vectors, candidate_indices)

    anchors = core_positive_anchors.copy()
    anchors["movieId"] = pd.to_numeric(anchors["movieId"], errors="coerce")
    anchors = anchors.dropna(subset=["movieId"]).copy()
    anchors["movieId"] = anchors["movieId"].astype(int)
    anchors = anchors[anchors["movieId"].isin(movie_id_to_latent_idx)].copy()
    if anchors.empty:
        return result
    anchor_indices = [movie_id_to_latent_idx[int(movie_id)] for movie_id in anchors["movieId"]]
    anchor_vectors = _dense_latent_rows(latent_vectors, anchor_indices)
    anchor_weights = pd.to_numeric(anchors.get("anchor_weight", 1.0), errors="coerce").fillna(1.0).clip(lower=0.05).to_numpy(dtype=float)
    anchor_titles = anchors.get("title", pd.Series([""] * len(anchors))).fillna("").astype(str).tolist()

    positive_centroid = np.average(anchor_vectors, axis=0, weights=anchor_weights)
    positive_centroid = normalize(np.asarray(positive_centroid).reshape(1, -1))[0]
    result.loc[result.index[candidate_positions], "positive_centroid_similarity"] = cosine_similarity(candidate_vectors, positive_centroid.reshape(1, -1)).ravel()
    result.loc[result.index[candidate_positions], "n_core_anchors_used"] = len(anchors)

    anchor_sims = cosine_similarity(candidate_vectors, anchor_vectors)
    topk_scores, nearest_anchor_titles = [], []
    top_k_pos = min(top_k_positive, anchor_sims.shape[1])
    for row_sims in anchor_sims:
        order = np.argsort(row_sims)[::-1][:top_k_pos]
        top_sims = row_sims[order]
        top_weights = anchor_weights[order]
        mean_topk = float(np.average(top_sims, weights=top_weights)) if top_weights.sum() > 0 else float(top_sims.mean())
        max_topk = float(top_sims.max()) if len(top_sims) else 0.0
        topk_scores.append(0.70 * mean_topk + 0.30 * max_topk)
        nearest_anchor_titles.append(" | ".join(anchor_titles[i] for i in order[:3] if i < len(anchor_titles)))
    result.loc[result.index[candidate_positions], "topk_anchor_similarity"] = topk_scores
    result.loc[result.index[candidate_positions], "nearest_core_anchor_movies"] = nearest_anchor_titles

    branch_prototypes = {}
    for branch, group in anchors.groupby("semantic_branch", dropna=False):
        branch = str(branch) if pd.notna(branch) else "general"
        positions = [anchors.index.get_loc(idx) for idx in group.index]
        prototype = np.average(anchor_vectors[positions], axis=0, weights=anchor_weights[positions])
        branch_prototypes[branch] = normalize(np.asarray(prototype).reshape(1, -1))[0]
    branch_strength_lookup = {}
    if user_branch_profile is not None and not user_branch_profile.empty and "semantic_branch" in user_branch_profile.columns:
        branch_strength_lookup = user_branch_profile.set_index("semantic_branch").get("branch_strength", pd.Series(dtype=str)).to_dict()
    strong_medium_branches = [b for b in branch_prototypes if branch_strength_lookup.get(b) in ["strong", "medium"]] or list(branch_prototypes)

    meta_cols = [col for col in ["movieId", "semantic_branch", "semantic_core_score"] if col in candidates.columns]
    candidate_meta = candidates[meta_cols].copy() if meta_cols else candidates[["movieId"]].copy()
    if "semantic_branch" not in candidate_meta.columns:
        candidate_meta["semantic_branch"] = "general"
    candidate_meta["movieId"] = pd.to_numeric(candidate_meta["movieId"], errors="coerce")
    result_meta = result[["movieId"]].copy()
    result_meta["movieId"] = pd.to_numeric(result_meta["movieId"], errors="coerce")
    result_meta = result_meta.merge(candidate_meta.drop_duplicates("movieId"), on="movieId", how="left")
    result_meta["semantic_branch"] = result_meta["semantic_branch"].fillna("general").astype(str)

    branch_scores = []
    for pos, cand_idx in enumerate(candidate_positions):
        candidate_vector = candidate_vectors[pos].reshape(1, -1)
        branch = result_meta.iloc[cand_idx].get("semantic_branch", "general")
        if branch in branch_prototypes:
            branch_scores.append(float(cosine_similarity(candidate_vector, branch_prototypes[branch].reshape(1, -1))[0, 0]))
        elif strong_medium_branches:
            sims = [float(cosine_similarity(candidate_vector, branch_prototypes[b].reshape(1, -1))[0, 0]) for b in strong_medium_branches]
            branch_scores.append(max(sims) * 0.65 if sims else 0.0)
        else:
            branch_scores.append(0.0)
    result.loc[result.index[candidate_positions], "branch_prototype_similarity"] = branch_scores

    negative_indices, negative_weights, negative_titles = [], [], []
    if disliked_movies is not None and not disliked_movies.empty and "movieId" in disliked_movies.columns:
        negatives = disliked_movies.copy()
        negatives["movieId"] = pd.to_numeric(negatives["movieId"], errors="coerce")
        negatives["user_rating_5"] = pd.to_numeric(negatives.get("user_rating_5"), errors="coerce")
        negatives = negatives[(negatives["user_rating_5"] <= 2.5) & negatives["movieId"].notna()].copy()
        for _, row in negatives.iterrows():
            movie_id = int(row["movieId"])
            if movie_id in movie_id_to_latent_idx:
                negative_indices.append(movie_id_to_latent_idx[movie_id])
                negative_weights.append(max(3.0 - float(row.get("user_rating_5", 2.5)), 0.05))
                negative_titles.append(str(row.get("title", f"movieId={movie_id}")))
    if negative_indices:
        negative_vectors = _dense_latent_rows(latent_vectors, negative_indices)
        negative_sims = cosine_similarity(candidate_vectors, negative_vectors)
        negative_weights = np.array(negative_weights, dtype=float)
        top_k_neg = min(top_k_negative, negative_sims.shape[1])
        raw_scores, nearest_titles = [], []
        for row_sims in negative_sims:
            order = np.argsort(row_sims)[::-1][:top_k_neg]
            top_sims = row_sims[order]
            top_weights = negative_weights[order]
            mean_topk = float(np.average(top_sims, weights=top_weights)) if top_weights.sum() > 0 else float(top_sims.mean())
            max_topk = float(top_sims.max()) if len(top_sims) else 0.0
            raw_scores.append(0.70 * mean_topk + 0.30 * max_topk)
            nearest_titles.append(" | ".join(negative_titles[i] for i in order[:3] if i < len(negative_titles)))
        result.loc[result.index[candidate_positions], "negative_latent_preference_raw"] = raw_scores
        result.loc[result.index[candidate_positions], "nearest_negative_movies_latent"] = nearest_titles
        result.loc[result.index[candidate_positions], "n_negative_anchors_used"] = len(negative_indices)
    result["negative_latent_preference_score"] = rank_normalize_positive(result["negative_latent_preference_raw"])

    if user_branch_profile is not None and not user_branch_profile.empty and {"semantic_branch", "branch_share", "branch_strength"}.issubset(user_branch_profile.columns):
        branch_meta = result_meta.merge(user_branch_profile[["semantic_branch", "branch_share", "branch_strength"]].drop_duplicates("semantic_branch"), on="semantic_branch", how="left")
    else:
        branch_meta = result_meta.copy()
        branch_meta["branch_share"] = 0.0
        branch_meta["branch_strength"] = "unseen"
    branch_multiplier = branch_meta["branch_strength"].fillna("unseen").map({"strong":1.00, "medium":0.88, "weak":0.68, "unseen":0.45, "unknown":0.70}).fillna(0.70)
    if "semantic_core_score" in result_meta.columns:
        semantic_support_multiplier = 0.65 + 0.35 * pd.to_numeric(result_meta["semantic_core_score"], errors="coerce").fillna(0.0).clip(0.0, 1.0)
    else:
        semantic_support_multiplier = pd.Series(1.0, index=result.index)

    raw = 0.42 * result["positive_centroid_similarity"] + 0.38 * result["topk_anchor_similarity"] + 0.20 * result["branch_prototype_similarity"] - 0.50 * result["negative_latent_preference_raw"]
    result["latent_core_preference_raw"] = (raw * branch_multiplier.reindex(result.index).fillna(0.70) * semantic_support_multiplier.reindex(result.index).fillna(1.0)).fillna(0.0)
    result["latent_core_preference_score"] = rank_normalize_positive(result["latent_core_preference_raw"])
    return result


def compute_semantic_relevance_scores(df):
    scored = df.copy()

    if "semantic_net_score" in scored.columns:
        semantic_fallback = scored["semantic_net_score"]
    elif "semantic_core_score" in scored.columns:
        semantic_fallback = scored["semantic_core_score"]
    else:
        semantic_fallback = scored.get("semantic_profile_score", 0.0)

    if "latent_core_preference_score" in scored.columns and pd.to_numeric(scored["latent_core_preference_score"], errors="coerce").fillna(0.0).var() > 0:
        latent_core = pd.to_numeric(scored["latent_core_preference_score"], errors="coerce").fillna(0.0)
        if "semantic_net_score" in scored.columns:
            semantic_net = pd.to_numeric(scored["semantic_net_score"], errors="coerce").fillna(0.0)
            scored["semantic_relevance_raw"] = 0.82 * latent_core + 0.18 * semantic_net
        else:
            scored["semantic_relevance_raw"] = latent_core
    else:
        scored["semantic_relevance_raw"] = pd.Series(semantic_fallback, index=scored.index).fillna(0.0)
    scored["semantic_relevance_score"] = rank_normalize_positive(scored["semantic_relevance_raw"])

    if "semantic_core_negative_score" in scored.columns:
        negative_existing = scored["semantic_core_negative_score"]
    else:
        negative_existing = scored.get("negative_semantic_score", 0.0)

    if "negative_latent_preference_score" in scored.columns:
        negative_existing_series = pd.Series(negative_existing, index=scored.index).fillna(0.0)
        scored["negative_semantic_relevance_raw"] = (
            0.65 * pd.to_numeric(scored["negative_latent_preference_score"], errors="coerce").fillna(0.0)
            + 0.35 * negative_existing_series
        )
    else:
        scored["negative_semantic_relevance_raw"] = pd.Series(negative_existing, index=scored.index).fillna(0.0)
    scored["negative_semantic_relevance_score"] = rank_normalize_positive(scored["negative_semantic_relevance_raw"])

    if "branch_strength" in scored.columns:
        branch_multiplier = scored["branch_strength"].fillna("unknown").map({
            "strong": 1.00,
            "medium": 0.82,
            "weak": 0.62,
            "unseen": 0.40,
            "unknown": 0.70,
        }).fillna(0.70)
        if "latent_core_preference_score" in scored.columns:
            latent_core = pd.to_numeric(scored["latent_core_preference_score"], errors="coerce").fillna(0.0)
            p90 = latent_core.quantile(0.90) if len(latent_core) else np.inf
            branch_multiplier = branch_multiplier.mask(latent_core >= p90, np.maximum(branch_multiplier, 0.75))
        scored["branch_support_multiplier"] = branch_multiplier
        scored["semantic_relevance_adjusted_score"] = rank_normalize_positive(scored["semantic_relevance_score"].fillna(0.0) * branch_multiplier)
    else:
        scored["branch_support_multiplier"] = 1.0
        scored["semantic_relevance_adjusted_score"] = scored["semantic_relevance_score"].fillna(0.0).clip(0.0, 1.0)
    return scored

def analyze_temporal_profile(
    user_movies,
    liked_movies=None,
    watched_movies=None,
    year_col="year",
    rating_col="user_rating_5",
):
    liked_years = _valid_year_series(liked_movies, year_col)
    user_years = _valid_year_series(user_movies, year_col)
    watched_years = _valid_year_series(watched_movies, year_col)

    profile_years = liked_years if len(liked_years) >= 5 else user_years
    if len(profile_years) < 5 and len(watched_years) > len(profile_years):
        profile_years = watched_years

    liked_bounds_years = liked_years if not liked_years.empty else profile_years

    if profile_years.empty:
        return {
            "n_temporal_movies": 0,
            "min_year": np.nan,
            "max_year": np.nan,
            "q10_year": np.nan,
            "q25_year": np.nan,
            "median_year": np.nan,
            "q75_year": np.nan,
            "q90_year": np.nan,
            "mean_year": np.nan,
            "std_year": np.nan,
            "earliest_liked_year": liked_bounds_years.min() if not liked_bounds_years.empty else np.nan,
            "latest_liked_year": liked_bounds_years.max() if not liked_bounds_years.empty else np.nan,
            "pct_before_1960": 0.0,
            "pct_before_1970": 0.0,
            "pct_before_1980": 0.0,
            "pct_before_1990": 0.0,
            "pct_after_2000": 0.0,
            "pct_after_2010": 0.0,
            "temporal_profile_strength": 0.0,
            "classic_tolerance": 0.0,
            "temporal_strictness": 0.0,
        }

    n_temporal_movies = len(profile_years)
    q10_year = float(profile_years.quantile(0.10))
    q90_year = float(profile_years.quantile(0.90))
    pct_before_1960 = float((profile_years < 1960).mean())
    pct_before_1970 = float((profile_years < 1970).mean())
    pct_before_1980 = float((profile_years < 1980).mean())
    pct_before_1990 = float((profile_years < 1990).mean())
    pct_after_2000 = float((profile_years >= 2000).mean())
    pct_after_2010 = float((profile_years >= 2010).mean())

    if n_temporal_movies < 5:
        temporal_profile_strength = min(0.4, n_temporal_movies / 10)
    else:
        temporal_profile_strength = _clip01(
            min(1.0, n_temporal_movies / 30) * 0.60
            + min(1.0, max(1.0, q90_year - q10_year) / 40) * 0.10
            + 0.30
        )

    classic_tolerance = _clip01(
        0.50 * pct_before_1980
        + 0.30 * pct_before_1970
        + 0.20 * pct_before_1960
    )
    temporal_strictness = _clip01(temporal_profile_strength * (1 - classic_tolerance))

    return {
        "n_temporal_movies": int(n_temporal_movies),
        "min_year": float(profile_years.min()),
        "max_year": float(profile_years.max()),
        "q10_year": q10_year,
        "q25_year": float(profile_years.quantile(0.25)),
        "median_year": float(profile_years.median()),
        "q75_year": float(profile_years.quantile(0.75)),
        "q90_year": q90_year,
        "mean_year": float(profile_years.mean()),
        "std_year": float(profile_years.std(ddof=0)) if n_temporal_movies > 1 else 0.0,
        "earliest_liked_year": float(liked_bounds_years.min()) if not liked_bounds_years.empty else np.nan,
        "latest_liked_year": float(liked_bounds_years.max()) if not liked_bounds_years.empty else np.nan,
        "pct_before_1960": pct_before_1960,
        "pct_before_1970": pct_before_1970,
        "pct_before_1980": pct_before_1980,
        "pct_before_1990": pct_before_1990,
        "pct_after_2000": pct_after_2000,
        "pct_after_2010": pct_after_2010,
        "temporal_profile_strength": temporal_profile_strength,
        "classic_tolerance": classic_tolerance,
        "temporal_strictness": temporal_strictness,
    }


def compute_year_affinity_scores(candidates, temporal_metrics, year_col="year"):
    scored = candidates.copy()
    temporal_profile_strength = float(temporal_metrics.get("temporal_profile_strength", 0.0) or 0.0)
    temporal_strictness = float(temporal_metrics.get("temporal_strictness", 0.0) or 0.0)
    classic_tolerance = float(temporal_metrics.get("classic_tolerance", 0.0) or 0.0)

    scored["year_affinity_score"] = 1.0
    scored["temporal_mismatch_penalty"] = 0.0
    scored["temporal_distance_from_profile"] = 0.0
    scored["is_temporal_outlier"] = False

    years = pd.to_numeric(scored.get(year_col), errors="coerce")
    invalid_year_mask = years.isna()
    if invalid_year_mask.any():
        scored.loc[invalid_year_mask, "year_affinity_score"] = 0.5
        scored.loc[invalid_year_mask, "temporal_mismatch_penalty"] = 0.5 * temporal_strictness
        scored.loc[invalid_year_mask, "temporal_distance_from_profile"] = np.nan
        scored.loc[invalid_year_mask, "is_temporal_outlier"] = True

    if temporal_profile_strength < 0.25:
        scored.loc[~invalid_year_mask, "year_affinity_score"] = 1.0
        scored.loc[~invalid_year_mask, "temporal_mismatch_penalty"] = 0.0
        scored.loc[~invalid_year_mask, "temporal_distance_from_profile"] = 0.0
        scored.loc[~invalid_year_mask, "is_temporal_outlier"] = False
        return scored

    q10_year = temporal_metrics.get("q10_year")
    q90_year = temporal_metrics.get("q90_year")
    if pd.isna(q10_year) or pd.isna(q90_year):
        return scored

    preferred_low = float(q10_year) - 5
    preferred_high = float(q90_year) + 5
    valid_year_mask = ~invalid_year_mask
    before_mask = valid_year_mask & (years < preferred_low)
    after_mask = valid_year_mask & (years > preferred_high)
    inside_mask = valid_year_mask & ~(before_mask | after_mask)

    scored.loc[inside_mask, "year_affinity_score"] = 1.0
    scored.loc[inside_mask, "temporal_distance_from_profile"] = 0.0

    if after_mask.any():
        distance = years.loc[after_mask] - preferred_high
        scored.loc[after_mask, "year_affinity_score"] = np.maximum(np.exp(-distance / 35), 0.65)
        scored.loc[after_mask, "temporal_distance_from_profile"] = distance

    if before_mask.any():
        distance = preferred_low - years.loc[before_mask]
        decay = 15 + 35 * classic_tolerance
        floor = 0.35 if classic_tolerance >= 0.30 else 0.05
        scored.loc[before_mask, "year_affinity_score"] = np.maximum(np.exp(-distance / decay), floor)
        scored.loc[before_mask, "temporal_distance_from_profile"] = distance

    scored["year_affinity_score"] = scored["year_affinity_score"].clip(0.0, 1.0)
    scored["temporal_mismatch_penalty"] = (1 - scored["year_affinity_score"]) * temporal_strictness
    scored.loc[invalid_year_mask, "temporal_mismatch_penalty"] = 0.5 * temporal_strictness
    scored["is_temporal_outlier"] = (
        (years < preferred_low - 20)
        & (classic_tolerance < 0.20)
        & (temporal_profile_strength >= 0.50)
    ).fillna(False)
    scored.loc[invalid_year_mask, "is_temporal_outlier"] = True
    return scored


def apply_temporal_outlier_allowance(candidates, min_candidates_after_filter=100):
    scored = candidates.copy()
    if "is_temporal_outlier" not in scored.columns:
        scored["allow_temporal_outlier"] = True
        return scored

    collab_p80 = scored["item_item_collab_score"].quantile(0.80) if "item_item_collab_score" in scored.columns else np.inf
    semantic_reference = "semantic_net_score" if "semantic_net_score" in scored.columns else "semantic_profile_score"
    semantic_p90 = scored[semantic_reference].quantile(0.90) if semantic_reference in scored.columns else np.inf
    rating_p95 = scored["rating_score"].quantile(0.95) if "rating_score" in scored.columns else np.inf

    scored["allow_temporal_outlier"] = (
        ~scored["is_temporal_outlier"].fillna(False)
        | (scored.get("item_item_collab_score", 0) >= collab_p80)
        | (scored.get(semantic_reference, 0) >= semantic_p90)
        | (scored.get("rating_score", 0) >= rating_p95)
    )

    filtered = scored[scored["allow_temporal_outlier"]].copy()
    if len(filtered) >= min_candidates_after_filter:
        return filtered

    warnings.warn(
        "No se aplica el filtro suave de outliers temporales porque dejar?a menos de "
        f"{min_candidates_after_filter} candidatos."
    )
    scored["allow_temporal_outlier"] = True
    return scored

def analyze_user_profile_for_adaptive_weights(
    liked_movies,
    neutral_movies,
    disliked_movies,
    positive_tag_profile,
    negative_tag_profile,
    trakt_ratings_profile,
    movie_id_to_col=None,
    candidates_scored=None,
    discriminative_tag_profile=None,
):
    n_liked = len(liked_movies)
    n_neutral = len(neutral_movies)
    n_disliked = len(disliked_movies)
    n_total_ratings = len(trakt_ratings_profile)
    profile_size_score = _clip01(n_total_ratings / 80)
    liked_size_score = _clip01(n_liked / 30)
    disliked_size_score = _clip01(n_disliked / 15)

    n_positive_semantic_tags = len(positive_tag_profile) if positive_tag_profile is not None else 0
    n_negative_semantic_tags = len(negative_tag_profile) if negative_tag_profile is not None else 0
    semantic_ambiguity, n_shared_semantic_tags = _semantic_ambiguity(positive_tag_profile, negative_tag_profile)

    if positive_tag_profile is not None and "semantic_category" in positive_tag_profile.columns:
        category_strength = _clip01(positive_tag_profile["semantic_category"].dropna().nunique() / 4)
    else:
        category_strength = 0.5
    semantic_profile_strength = _clip01(
        0.45 * min(1.0, n_positive_semantic_tags / 40)
        + 0.35 * liked_size_score
        + 0.20 * category_strength
    )
    semantic_reliability = _clip01(semantic_profile_strength * (1 - semantic_ambiguity))

    n_core_positive_tags = 0
    n_core_negative_tags = 0
    mean_core_tag_ambiguity = semantic_ambiguity
    core_semantic_strength = semantic_profile_strength
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty:
        core_positive = discriminative_tag_profile[discriminative_tag_profile["is_core_positive_tag"]].copy()
        core_negative = discriminative_tag_profile[discriminative_tag_profile["is_core_negative_tag"]].copy()
        n_core_positive_tags = len(core_positive)
        n_core_negative_tags = len(core_negative)
        mean_core_tag_ambiguity = core_positive["tag_ambiguity"].mean() if not core_positive.empty else 1.0
        if not core_positive.empty and "semantic_category" in core_positive.columns:
            n_core_categories = core_positive["semantic_category"].dropna().nunique()
        else:
            n_core_categories = 0
        core_semantic_strength = _clip01(
            min(1.0, n_core_positive_tags / 30) * 0.45
            + min(1.0, n_core_categories / 4) * 0.25
            + liked_size_score * 0.30
        )
        semantic_reliability = _clip01(core_semantic_strength * (1 - mean_core_tag_ambiguity))

    negative_profile_strength = _clip01(0.50 * disliked_size_score + 0.50 * min(1.0, max(n_negative_semantic_tags, n_core_negative_tags) / 30))
    negative_semantic_reliability = _clip01(negative_profile_strength * (1 - mean_core_tag_ambiguity))

    positive_genres = extract_genre_weights_from_profile(liked_movies, positive=True)
    negative_genres = extract_genre_weights_from_profile(disliked_movies, positive=False)
    n_positive_genres = len(positive_genres)
    n_negative_genres = len(negative_genres)
    if positive_genres.empty or negative_genres.empty:
        genre_ambiguity = 1.0
    else:
        genre_overlap = positive_genres.merge(negative_genres, on="genre", how="inner", suffixes=("_pos", "_neg"))
        genre_ambiguity = _clip01(np.minimum(genre_overlap["genre_weight_norm_pos"], genre_overlap["genre_weight_norm_neg"]).sum()) if not genre_overlap.empty else 0.0
    genre_profile_strength = _clip01(0.50 * min(1.0, n_positive_genres / 6) + 0.50 * liked_size_score)
    genre_reliability = _clip01(genre_profile_strength * (1 - genre_ambiguity))

    if movie_id_to_col is not None:
        rated_ids = set(trakt_ratings_profile["movieId"].dropna().astype(int)) if "movieId" in trakt_ratings_profile.columns else set()
        collab_coverage = len(rated_ids & set(movie_id_to_col.keys())) / len(rated_ids) if rated_ids else 0.0
    else:
        collab_coverage = 0.5
    if candidates_scored is not None and "n_collab_evidence" in candidates_scored.columns and len(candidates_scored):
        collab_candidate_evidence = (candidates_scored["n_collab_evidence"].fillna(0) > 0).mean()
        collab_reliability = _clip01(0.70 * collab_coverage + 0.30 * collab_candidate_evidence)
    else:
        collab_candidate_evidence = np.nan
        collab_reliability = _clip01(collab_coverage)

    profile_weakness = _clip01(1 - profile_size_score)
    return {
        "n_liked": n_liked,
        "n_neutral": n_neutral,
        "n_disliked": n_disliked,
        "n_total_ratings": n_total_ratings,
        "profile_size_score": profile_size_score,
        "liked_size_score": liked_size_score,
        "disliked_size_score": disliked_size_score,
        "n_positive_semantic_tags": n_positive_semantic_tags,
        "n_negative_semantic_tags": n_negative_semantic_tags,
        "n_shared_semantic_tags": n_shared_semantic_tags,
        "semantic_ambiguity": semantic_ambiguity,
        "semantic_profile_strength": semantic_profile_strength,
        "semantic_reliability": semantic_reliability,
        "negative_profile_strength": negative_profile_strength,
        "negative_semantic_reliability": negative_semantic_reliability,
        "n_core_positive_tags": n_core_positive_tags,
        "n_core_negative_tags": n_core_negative_tags,
        "mean_core_tag_ambiguity": _clip01(mean_core_tag_ambiguity),
        "core_semantic_strength": core_semantic_strength,
        "n_positive_genres": n_positive_genres,
        "n_negative_genres": n_negative_genres,
        "genre_ambiguity": genre_ambiguity,
        "genre_profile_strength": genre_profile_strength,
        "genre_reliability": genre_reliability,
        "collab_coverage": _clip01(collab_coverage),
        "collab_candidate_evidence": collab_candidate_evidence,
        "collab_reliability": collab_reliability,
        "profile_weakness": profile_weakness,
    }


def _bounded_normalize(raw_weights, target_total, min_bounds=None, max_bounds=None):
    min_bounds = min_bounds or {}
    max_bounds = max_bounds or {}
    keys = list(raw_weights)
    weights = {key: max(float(raw_weights[key]), 0.0) for key in keys}
    raw_total = sum(weights.values())
    if raw_total <= 0:
        return {key: target_total / len(keys) for key in keys}
    weights = {key: value / raw_total * target_total for key, value in weights.items()}
    fixed = {}
    free = set(keys)
    for _ in range(10):
        changed = False
        for key in list(free):
            lower = min_bounds.get(key, 0.0)
            upper = max_bounds.get(key, float("inf"))
            if weights[key] < lower:
                fixed[key] = lower
                free.remove(key)
                changed = True
            elif weights[key] > upper:
                fixed[key] = upper
                free.remove(key)
                changed = True
        remaining = target_total - sum(fixed.values())
        if remaining <= 0 or not free:
            break
        free_raw_total = sum(raw_weights[key] for key in free)
        if free_raw_total <= 0:
            for key in free:
                weights[key] = remaining / len(free)
        else:
            for key in free:
                weights[key] = raw_weights[key] / free_raw_total * remaining
        if not changed:
            break
    weights.update(fixed)
    return weights


def derive_adaptive_weights(profile_metrics):
    genre_reliability = profile_metrics.get("genre_reliability", 0.0)
    semantic_reliability = profile_metrics.get("semantic_reliability", 0.0)
    collab_reliability = profile_metrics.get("collab_reliability", 0.0)
    negative_semantic_reliability = profile_metrics.get("negative_semantic_reliability", 0.0)
    negative_profile_strength = profile_metrics.get("negative_profile_strength", 0.0)
    profile_weakness = profile_metrics.get("profile_weakness", 1.0)
    temporal_profile_strength = profile_metrics.get("temporal_profile_strength", 0.0)
    temporal_strictness = profile_metrics.get("temporal_strictness", 0.0)

    raw_positive = {
        "genre": 0.06 + 0.12 * genre_reliability,
        "semantic": 0.10 + 0.35 * semantic_reliability,
        "collab": 0.10 + 0.22 * collab_reliability,
        "rating": 0.14 + 0.14 * profile_weakness,
        "popularity": 0.02 + 0.08 * profile_weakness,
        "year_affinity": 0.04 + 0.08 * temporal_profile_strength,
    }
    raw_negative = {
        "negative_genre": 0.10 + 0.15 * genre_reliability * negative_profile_strength,
        "negative_semantic": 0.15 + 0.35 * negative_semantic_reliability,
        "negative_collab": 0.08 + 0.20 * collab_reliability * negative_profile_strength,
        "temporal_mismatch": 0.08 + 0.22 * temporal_strictness,
    }

    positive_min = {"rating": 0.12}
    if collab_reliability >= 0.70:
        positive_min["collab"] = 0.18
    positive_max = {"semantic": 0.30, "popularity": 0.06, "collab": 0.25, "rating": 0.25, "year_affinity": 0.10}
    negative_max = {"negative_semantic": 0.35, "temporal_mismatch": 0.22}

    weights = _bounded_normalize(raw_positive, 0.75, min_bounds=positive_min, max_bounds=positive_max)
    weights.update(_bounded_normalize(raw_negative, 0.60, max_bounds=negative_max))
    return weights


discriminative_tag_profile = build_discriminative_semantic_profile(
    tags_clean=tags_clean,
    liked_movies=liked_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
)

print("Top 30 tags por discriminative_weight")
display(discriminative_tag_profile.sort_values("discriminative_weight", ascending=False).head(30))
print("Top 30 tags por negative_discriminative_weight")
display(discriminative_tag_profile.sort_values("negative_discriminative_weight", ascending=False).head(30))
print("Media de tag_ambiguity:", discriminative_tag_profile["tag_ambiguity"].mean() if not discriminative_tag_profile.empty else np.nan)
print("Número de core positive tags:", int(discriminative_tag_profile["is_core_positive_tag"].sum()) if not discriminative_tag_profile.empty else 0)
print("Número de core negative tags:", int(discriminative_tag_profile["is_core_negative_tag"].sum()) if not discriminative_tag_profile.empty else 0)
if not discriminative_tag_profile.empty and "semantic_category" in discriminative_tag_profile.columns:
    display(discriminative_tag_profile.loc[discriminative_tag_profile["is_core_positive_tag"], "semantic_category"].value_counts(dropna=False))

discriminative_tag_profile.to_csv(REPORTS_RESULTADOS / "discriminative_tag_profile.csv", index=False)

semantic_core_scores = compute_semantic_core_scores(
    candidates=candidates_scored,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
)
candidates_scored = candidates_scored.drop(columns=[col for col in semantic_core_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
    semantic_core_scores,
    on="movieId",
    how="left",
)
for col in ["semantic_core_raw_score", "semantic_core_negative_raw_score", "semantic_core_score", "semantic_core_negative_score"]:
    candidates_scored[col] = candidates_scored[col].fillna(0.0)
for col in ["core_semantic_explanation_terms", "core_negative_semantic_terms"]:
    candidates_scored[col] = candidates_scored[col].fillna("")

if "main_genre" not in candidates_scored.columns:
    candidates_scored["main_genre"] = candidates_scored["genres"].apply(main_genre_from_genres)
candidates_scored["semantic_branch"] = candidates_scored.apply(assign_semantic_branch, axis=1)

user_branch_profile = build_user_branch_profile(
    liked_movies=liked_movies,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
    assign_branch_func=assign_semantic_branch,
)
candidates_scored = add_branch_affinity_to_candidates(
    candidates_scored,
    user_branch_profile,
    branch_col="semantic_branch",
)


temporal_metrics = analyze_temporal_profile(
    user_movies=trakt_ratings_profile,
    liked_movies=liked_movies,
    watched_movies=trakt_watched.merge(movies[["movieId", "year"]], on="movieId", how="left") if "year" not in trakt_watched.columns else trakt_watched,
)
temporal_metrics_df = pd.DataFrame(temporal_metrics.items(), columns=["metric", "value"])
display(temporal_metrics_df)
temporal_metrics_df.to_csv(REPORTS_RESULTADOS / "temporal_profile_metrics.csv", index=False)

candidates_scored = compute_year_affinity_scores(candidates_scored, temporal_metrics, year_col="year")

profile_metrics = analyze_user_profile_for_adaptive_weights(
    liked_movies=liked_movies,
    neutral_movies=neutral_movies,
    disliked_movies=disliked_movies,
    positive_tag_profile=positive_tag_profile,
    negative_tag_profile=negative_tag_profile,
    trakt_ratings_profile=trakt_ratings,
    movie_id_to_col=movie_id_to_col,
    candidates_scored=candidates_scored,
    discriminative_tag_profile=discriminative_tag_profile,
)
profile_metrics.update(temporal_metrics)
profile_metrics_df = pd.DataFrame(profile_metrics.items(), columns=["metric", "value"])
display(profile_metrics_df.sort_values("metric"))
profile_metrics_df.to_csv(REPORTS_RESULTADOS / "adaptive_profile_metrics.csv", index=False)

semantic_negative_beta = float(np.clip(0.50 + 0.75 * profile_metrics["negative_profile_strength"], 0.50, 1.25))
candidates_scored = compute_semantic_net_score(candidates_scored, beta=semantic_negative_beta)

positive_anchor_evidence = pd.DataFrame()
core_positive_anchors = pd.DataFrame()
try:
    positive_anchor_evidence = compute_positive_anchor_evidence(
        liked_movies=liked_movies,
        tags_clean=tags_clean,
        discriminative_tag_profile=discriminative_tag_profile,
        user_branch_profile=user_branch_profile if "user_branch_profile" in globals() else None,
        assign_branch_func=assign_semantic_branch if "assign_semantic_branch" in globals() else None,
    )
    positive_anchor_evidence.to_csv(REPORTS_RESULTADOS / "positive_anchor_evidence.csv", index=False)
    print("Top 25 positive_anchor_evidence por anchor_weight:")
    display(positive_anchor_evidence[[
        "title", "year", "user_rating_5", "semantic_branch", "branch_strength",
        "centrality_score", "anchor_weight"
    ]].head(25))

    core_positive_anchors = select_core_positive_anchors(positive_anchor_evidence, top_n_anchors=30, min_anchors=12)
    core_positive_anchors.to_csv(REPORTS_RESULTADOS / "core_positive_anchors_selected.csv", index=False)
    print("Distribucion de ramas de core_positive_anchors_selected:")
    display(core_positive_anchors["semantic_branch"].value_counts(dropna=False))
    display(core_positive_anchors[[
        "title", "year", "user_rating_5", "semantic_branch", "branch_strength",
        "centrality_score", "anchor_weight", "selected_reason"
    ]].head(30))
except Exception as exc:
    warnings.warn(f"No se pudo calcular la evidencia de anclas positivas; se usara fallback semantico. Detalle: {exc}")
    positive_anchor_evidence = pd.DataFrame()
    core_positive_anchors = pd.DataFrame()

latent_semantic_diagnostics = {
    "n_movie_documents": 0,
    "tfidf_n_features": 0,
    "latent_n_components": 0,
    "n_liked_movies_latent": 0,
    "n_disliked_movies_latent": 0,
}
try:
    movie_semantic_documents = build_movie_semantic_documents(movies, tags_clean, max_tags_per_movie=30)
    latent_vectors, latent_vectorizer, latent_svd_model, movie_id_to_latent_idx, latent_idx_to_movie_id = build_latent_semantic_space(
        movie_semantic_documents,
        n_components=64,
        max_features=8000,
        min_df=2,
        random_state=RANDOM_STATE,
    )
    latent_similarity_scores = compute_latent_user_similarity_scores(
        candidates=candidates_scored,
        liked_movies=liked_movies,
        disliked_movies=disliked_movies,
        latent_vectors=latent_vectors,
        movie_id_to_latent_idx=movie_id_to_latent_idx,
        top_k_positive=5,
        top_k_negative=5,
    )
    candidates_scored = candidates_scored.drop(columns=[col for col in latent_similarity_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
        latent_similarity_scores,
        on="movieId",
        how="left",
    )

    latent_core_preference_scores = compute_latent_core_preference_scores(
        candidates=candidates_scored,
        core_positive_anchors=core_positive_anchors,
        disliked_movies=disliked_movies,
        latent_vectors=latent_vectors,
        movie_id_to_latent_idx=movie_id_to_latent_idx,
        user_branch_profile=user_branch_profile if "user_branch_profile" in globals() else None,
        top_k_positive=5,
        top_k_negative=5,
    )
    candidates_scored = candidates_scored.drop(columns=[col for col in latent_core_preference_scores.columns if col != "movieId" and col in candidates_scored.columns]).merge(
        latent_core_preference_scores,
        on="movieId",
        how="left",
    )
    latent_semantic_diagnostics.update({
        "n_movie_documents": len(movie_semantic_documents),
        "tfidf_n_features": len(getattr(latent_vectorizer, "vocabulary_", {})),
        "latent_n_components": int(latent_vectors.shape[1]) if len(latent_vectors.shape) > 1 else 0,
        "n_liked_movies_latent": int(candidates_scored["n_positive_latent_neighbors"].max()) if "n_positive_latent_neighbors" in candidates_scored.columns else 0,
        "n_disliked_movies_latent": int(candidates_scored["n_negative_latent_neighbors"].max()) if "n_negative_latent_neighbors" in candidates_scored.columns else 0,
        "n_positive_anchor_evidence": int(len(positive_anchor_evidence)),
        "n_core_anchors_selected": int(len(core_positive_anchors)),
        "n_core_anchors_used": int(candidates_scored["n_core_anchors_used"].max()) if "n_core_anchors_used" in candidates_scored.columns else 0,
        "n_negative_anchors_used": int(candidates_scored["n_negative_anchors_used"].max()) if "n_negative_anchors_used" in candidates_scored.columns else 0,
    })
except Exception as exc:
    warnings.warn(f"No se pudo calcular la similitud semantica latente; se usan scores latentes en 0. Detalle: {exc}")
    for col, default in {
        "latent_core_similarity_raw": 0.0,
        "latent_core_similarity_score": 0.0,
        "negative_latent_similarity_raw": 0.0,
        "negative_latent_similarity_score": 0.0,
        "nearest_liked_movies_latent": "",
        "nearest_disliked_movies_latent": "",
        "n_positive_latent_neighbors": 0,
        "n_negative_latent_neighbors": 0,
        "positive_centroid_similarity": 0.0,
        "topk_anchor_similarity": 0.0,
        "branch_prototype_similarity": 0.0,
        "negative_latent_preference_raw": 0.0,
        "negative_latent_preference_score": 0.0,
        "latent_core_preference_raw": 0.0,
        "latent_core_preference_score": 0.0,
        "nearest_core_anchor_movies": "",
        "nearest_negative_movies_latent": "",
        "n_core_anchors_used": 0,
        "n_negative_anchors_used": 0,
    }.items():
        candidates_scored[col] = default

candidates_scored = compute_semantic_relevance_scores(candidates_scored)
print("Peliculas con documentos semanticos:", latent_semantic_diagnostics["n_movie_documents"])
print("Dimensiones TF-IDF:", latent_semantic_diagnostics["tfidf_n_features"])
print("Dimensiones latentes:", latent_semantic_diagnostics["latent_n_components"])
print("Peliculas positivas usadas para similitud latente:", latent_semantic_diagnostics["n_liked_movies_latent"])
print("Peliculas negativas usadas para similitud latente:", latent_semantic_diagnostics["n_disliked_movies_latent"])
if "latent_core_similarity_score" in candidates_scored.columns:
    display(candidates_scored.sort_values("latent_core_similarity_score", ascending=False)[[
        "title", "year", "latent_core_similarity_score", "semantic_net_score", "semantic_relevance_score", "nearest_liked_movies_latent"
    ]].head(20))
if "latent_core_preference_score" in candidates_scored.columns:
    display(candidates_scored.sort_values("latent_core_preference_score", ascending=False)[[
        "title", "year", "genres", "semantic_branch", "branch_strength",
        "latent_core_preference_score", "semantic_net_score", "semantic_relevance_score",
        "nearest_core_anchor_movies"
    ]].head(30))
temporal_outlier_candidate_count = int(candidates_scored["is_temporal_outlier"].sum()) if "is_temporal_outlier" in candidates_scored.columns else 0
candidates_scored = apply_temporal_outlier_allowance(candidates_scored)
temporal_outlier_allowed_count = int((candidates_scored["is_temporal_outlier"] & candidates_scored["allow_temporal_outlier"]).sum()) if {"is_temporal_outlier", "allow_temporal_outlier"}.issubset(candidates_scored.columns) else 0
temporal_outlier_removed_count = max(0, temporal_outlier_candidate_count - temporal_outlier_allowed_count)
candidates_scored["semantic_negative_beta"] = semantic_negative_beta

adaptive_weights = derive_adaptive_weights(profile_metrics)
adaptive_weights_df = pd.DataFrame(adaptive_weights.items(), columns=["component", "weight"])
display(adaptive_weights_df)
adaptive_weights_df.to_csv(REPORTS_RESULTADOS / "adaptive_weights.csv", index=False)

print(f"semantic_negative_beta: {semantic_negative_beta:.3f}")
print("Pesos adaptativos calculados:")
print(adaptive_weights)

Top 30 tags por discriminative_weight


,tag_clean,semantic_category,positive_weight,negative_weight,positive_weight_norm,negative_weight_norm,positive_n_seed_movies,negative_n_seed_movies,tag_ambiguity,discriminative_lift,discriminative_weight_raw,discriminative_weight,negative_discriminative_weight,is_core_positive_tag,is_core_negative_tag
0,tense,tone_atmosphere,64.307744,0.000000,0.010534,0.000000,12.0,0.0,0.000000,0.100153,0.001161,1.000000,0.0,True,False
1,dark,tone_atmosphere,73.126727,2.358927,0.011979,0.002606,14.0,1.0,0.002606,0.087415,0.001149,0.989921,0.0,True,False
2,psychology,theme,54.468484,0.000000,0.008922,0.000000,10.0,0.0,0.000000,0.085465,0.000839,0.722775,0.0,True,False
3,cinematography,visual_style,104.322034,10.866879,0.017089,0.012004,21.0,3.0,0.012004,0.044401,0.000825,0.710552,0.0,True,False
4,nostalgic,tone_atmosphere,53.348209,0.000000,0.008739,0.000000,7.0,0.0,0.000000,0.083778,0.000805,0.693942,0.0,True,False
5,existentialism,theme,52.521941,0.000000,0.008603,0.000000,8.0,0.0,0.000000,0.082533,0.000781,0.673038,0.0,True,False
6,ambiguous ending,narrative_structure,52.335802,0.000000,0.008573,0.000000,6.0,0.0,0.000000,0.082252,0.000776,0.668371,0.0,True,False
7,dreamlike,visual_style,50.012098,0.000000,0.008192,0.000000,8.0,0.0,0.000000,0.078740,0.000710,0.611424,0.0,True,False
8,surrealism,visual_style,49.732575,0.000000,0.008147,0.000000,10.0,0.0,0.000000,0.078317,0.000702,0.604738,0.0,True,False
9,atmospheric,tone_atmosphere,111.034884,12.811717,0.018188,0.014152,22.0,4.0,0.014152,0.034750,0.000685,0.590599,0.0,True,False


Top 30 tags por negative_discriminative_weight


,tag_clean,semantic_category,positive_weight,negative_weight,positive_weight_norm,negative_weight_norm,positive_n_seed_movies,negative_n_seed_movies,tag_ambiguity,discriminative_lift,discriminative_weight_raw,discriminative_weight,negative_discriminative_weight,is_core_positive_tag,is_core_negative_tag
210,long takes,NaN,0.000000,22.442782,0.000000,0.024790,0.0,2.0,0.000000,-0.221464,0.0,0.0,1.000000,False,True
277,surprise ending,narrative_structure,67.747161,24.039315,0.011097,0.026554,14.0,5.0,0.011097,-0.130259,0.0,0.0,0.623025,False,True
204,twist ending,narrative_structure,60.228121,22.585545,0.009866,0.024948,9.0,3.0,0.009866,-0.128638,0.0,0.0,0.578780,False,True
223,robots,theme,17.900399,17.900399,0.002932,0.019773,3.0,3.0,0.002932,-0.151525,0.0,0.0,0.544118,False,True
211,visually appealing,visual_style,79.191596,23.997453,0.012972,0.026507,14.0,6.0,0.012972,-0.113161,0.0,0.0,0.539275,False,True
272,technology,theme,16.684003,16.684003,0.002733,0.018429,3.0,3.0,0.002733,-0.142181,0.0,0.0,0.475965,False,True
267,dark humor,tone_atmosphere,22.942751,17.207063,0.003758,0.019007,3.0,3.0,0.003758,-0.137118,0.0,0.0,0.472919,False,True
260,shaky camera,NaN,0.000000,14.091218,0.000000,0.015565,0.0,1.0,0.000000,-0.144664,0.0,0.0,0.410137,False,True
276,stupid ending,narrative_structure,7.170772,14.341545,0.001175,0.015842,1.0,1.0,0.001175,-0.135376,0.0,0.0,0.390164,False,True
212,unexpected ending,narrative_structure,14.386490,14.386490,0.002357,0.015891,2.0,1.0,0.002357,-0.124189,0.0,0.0,0.358621,False,True


Media de tag_ambiguity: 0.0014676250137088621
Número de core positive tags: 199
Número de core negative tags: 88


...

Peliculas con documentos semanticos: 80505
Dimensiones TF-IDF: 7660
Dimensiones latentes: 64
Peliculas positivas usadas para similitud latente: 46
Peliculas negativas usadas para similitud latente: 20


,title,year,latent_core_similarity_score,semantic_net_score,semantic_relevance_score,nearest_liked_movies_latent
1496,Solaris (Solyaris) (1972),1972.0,1.000000,0.994043,0.999294,Arrival (2016) | Donnie Darko (2001) | Gattaca...
3427,In Bruges (2008),2008.0,0.999794,0.968354,0.997174,"Fargo (1996) | Big Lebowski, The (1998) | Uncu..."
2735,3 Women (Three Women) (1977),1977.0,0.999589,0.891288,0.987756,Parasite (2019) | Once Upon a Time in Hollywoo...
2096,Silent Running (1972),1972.0,0.999383,0.144453,0.902049,"Truman Show, The (1998) | Arrival (2016) | Gat..."
2891,"Holy Mountain, The (Montaña sagrada, La) (1973)",1973.0,0.999177,0.000000,0.796091,Parasite (2019) | Monty Python's Life of Brian...
4764,Promising Young Woman (2020),2020.0,0.998972,0.357409,0.932658,Parasite (2019) | Once Upon a Time in Hollywoo...
3965,"Hunt, The (Jagten) (2012)",2012.0,0.998766,0.916605,0.986814,Ex Machina (2015) | Arrival (2016) | Parasite ...
3306,Sunshine (2007),2007.0,0.998561,0.834698,0.964445,Ex Machina (2015) | Arrival (2016) | Donnie Da...
3743,Monsters (2010),2010.0,0.998355,0.000000,0.848128,Gattaca (1997) | Blade Runner 2049 (2017) | Ar...
4026,Oblivion (2013),2013.0,0.998149,0.000000,0.858017,Arrival (2016) | Gattaca (1997) | Ex Machina (...


,title,year,genres,semantic_branch,branch_strength,latent_core_preference_score,semantic_net_score,semantic_relevance_score,nearest_core_anchor_movies
2226,Narc (2002),2002.0,Crime|Drama|Thriller,crime_thriller,strong,1.000000,0.928891,0.995762,"Departed, The (2006) | No Country for Old Men ..."
2963,Old Boy (2003),2003.0,Mystery|Thriller,crime_thriller,strong,0.999761,0.963142,0.998587,No Country for Old Men (2007) | Once Upon a Ti...
139,Léon: The Professional (a.k.a. The Professiona...,1994.0,Action|Crime|Drama|Thriller,crime_thriller,strong,0.999521,0.950856,0.997645,Nightcrawler (2014) | Uncut Gems (2019) | Depa...
2568,The Butterfly Effect (2004),2004.0,Drama|Sci-Fi|Thriller,sci_fi_reflective,strong,0.999282,0.986597,0.999529,Interstellar (2014) | Arrival (2016) | Donnie ...
4112,Enemy (2013),2013.0,Mystery|Thriller,crime_thriller,strong,0.999043,0.997394,0.999765,Shutter Island (2010) | Donnie Darko (2001) | ...
3754,Black Swan (2010),2010.0,Drama|Thriller,crime_thriller,strong,0.998804,1.000000,1.000000,"Shutter Island (2010) | Shining, The (1980) | ..."
532,Brazil (1985),1985.0,Fantasy|Sci-Fi,sci_fi_reflective,strong,0.998564,0.840283,0.987050,"Truman Show, The (1998) | Blade Runner 2049 (2..."
4061,New World (Shin-sae-gye) (2013),2013.0,Thriller,crime_thriller,strong,0.998325,0.893522,0.992230,No Country for Old Men (2007) | Once Upon a Ti...
150,"Shawshank Redemption, The (1994)",1994.0,Crime|Drama,crime_thriller,strong,0.998086,0.969471,0.998116,"Departed, The (2006) | Arrival (2016) | No Cou..."
4764,Promising Young Woman (2020),2020.0,Crime|Drama|Thriller,crime_thriller,strong,0.997846,0.357409,0.932658,Parasite (2019) | Once Upon a Time in Hollywoo...


,component,weight
0,genre,0.061278
1,semantic,0.300000
2,collab,0.180000
3,rating,0.120000
4,popularity,0.012755
5,year_affinity,0.075967
6,negative_genre,0.074043
7,negative_semantic,0.255017
8,negative_collab,0.122562
9,temporal_mismatch,0.148379


semantic_negative_beta: 1.250
Pesos adaptativos calculados:
{'genre': 0.061278422059187716, 'semantic': 0.3, 'collab': 0.18, 'rating': 0.12, 'popularity': 0.01275468343024904, 'year_affinity': 0.07596689451056327, 'negative_genre': 0.07404290840570853, 'negative_semantic': 0.2550166372476469, 'negative_collab': 0.12256185191463556, 'temporal_mismatch': 0.148378602432009}


## Análisis y configuración de pesos del score híbrido

In [16]:
WEIGHT_CONFIGS = {
    "personalizado_semantico": {
        "genre": 0.18,
        "semantic": 0.35,
        "collab": 0.12,
        "rating": 0.15,
        "popularity": 0.03,
        "negative_genre": 0.20,
        "negative_semantic": 0.30,
        "negative_collab": 0.15,
    },
    "equilibrado_semantico": {
        "genre": 0.20,
        "semantic": 0.30,
        "collab": 0.15,
        "rating": 0.17,
        "popularity": 0.04,
        "negative_genre": 0.20,
        "negative_semantic": 0.25,
        "negative_collab": 0.15,
    },
    "descubrimiento_semantico": {
        "genre": 0.18,
        "semantic": 0.40,
        "collab": 0.08,
        "rating": 0.14,
        "popularity": 0.00,
        "negative_genre": 0.20,
        "negative_semantic": 0.35,
        "negative_collab": 0.20,
    },
}

ACTIVE_WEIGHT_CONFIG = "adaptive"


def compute_hybrid_score(df, weights):
    scored = df.copy()
    scored["contrib_genre"] = weights["genre"] * scored["genre_profile_score"]
    if "year_affinity_score" not in scored.columns:
        scored["year_affinity_score"] = 1.0
    if "temporal_mismatch_penalty" not in scored.columns:
        scored["temporal_mismatch_penalty"] = 0.0
    if "semantic_relevance_adjusted_score" in scored.columns:
        semantic_signal = scored["semantic_relevance_adjusted_score"]
    elif "semantic_relevance_score" in scored.columns:
        semantic_signal = scored["semantic_relevance_score"]
    elif "semantic_net_score" in scored.columns:
        semantic_signal = scored["semantic_net_score"]
    elif "semantic_core_score" in scored.columns:
        semantic_signal = scored["semantic_core_score"]
    else:
        semantic_signal = scored["semantic_profile_score"]
    if "negative_semantic_relevance_score" in scored.columns:
        negative_semantic_signal = scored["negative_semantic_relevance_score"]
    elif "semantic_core_negative_score" in scored.columns:
        negative_semantic_signal = scored["semantic_core_negative_score"]
    else:
        negative_semantic_signal = scored["negative_semantic_score"]
    scored["contrib_semantic"] = weights["semantic"] * semantic_signal
    scored["contrib_collab"] = weights["collab"] * scored["item_item_collab_score"]
    scored["contrib_rating"] = weights["rating"] * scored["rating_score"]
    scored["contrib_popularity"] = weights["popularity"] * scored["popularity_score"]
    scored["contrib_year_affinity"] = weights.get("year_affinity", 0.0) * scored["year_affinity_score"]
    scored["penalty_negative_genre"] = weights["negative_genre"] * scored["negative_genre_score"]
    scored["penalty_negative_semantic"] = weights["negative_semantic"] * negative_semantic_signal
    scored["penalty_negative_collab"] = weights["negative_collab"] * scored["item_item_negative_collab_score"]
    scored["penalty_temporal_mismatch"] = weights.get("temporal_mismatch", 0.0) * scored["temporal_mismatch_penalty"]

    scored["hybrid_score"] = (
        scored["contrib_genre"]
        + scored["contrib_semantic"]
        + scored["contrib_collab"]
        + scored["contrib_rating"]
        + scored["contrib_popularity"]
        + scored["contrib_year_affinity"]
        - scored["penalty_negative_genre"]
        - scored["penalty_negative_semantic"]
        - scored["penalty_negative_collab"]
        - scored["penalty_temporal_mismatch"]
    )

    contribution_cols = {
        "semantic": "contrib_semantic",
        "genre": "contrib_genre",
        "collab": "contrib_collab",
        "rating": "contrib_rating",
        "popularity": "contrib_popularity",
        "year_affinity": "contrib_year_affinity",
    }
    scored["dominant_signal"] = scored[list(contribution_cols.values())].idxmax(axis=1)
    scored["dominant_signal"] = scored["dominant_signal"].replace({value: key for key, value in contribution_cols.items()})
    return scored


candidates_scored_base = candidates_scored.copy()
print(f"Configuración activa de pesos: {ACTIVE_WEIGHT_CONFIG}")

Configuración activa de pesos: adaptive


## 14. Hybrid score inicial

In [17]:
weights = adaptive_weights if ACTIVE_WEIGHT_CONFIG == "adaptive" else WEIGHT_CONFIGS[ACTIVE_WEIGHT_CONFIG]

pre_latent_hybrid_input = candidates_scored.drop(columns=["semantic_relevance_score", "semantic_relevance_adjusted_score", "negative_semantic_relevance_score"], errors="ignore")
pre_latent_hybrid_scored = compute_hybrid_score(pre_latent_hybrid_input, weights).sort_values("hybrid_score", ascending=False).reset_index(drop=True)
pre_latent_top20_titles = set(pre_latent_hybrid_scored.head(20)["title"].astype(str))

candidates_scored = compute_hybrid_score(candidates_scored, weights)
post_latent_hybrid_top20 = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
post_latent_top20_titles = set(post_latent_hybrid_top20["title"].astype(str))
latent_ranking_overlap_top20 = len(pre_latent_top20_titles & post_latent_top20_titles) / 20 if pre_latent_top20_titles else np.nan
n_titles_changed_after_latent = 20 - len(pre_latent_top20_titles & post_latent_top20_titles) if pre_latent_top20_titles else np.nan
overlap_top20_before_after_semantic_adjustment = latent_ranking_overlap_top20
semantic_adjustment_changed_count = n_titles_changed_after_latent
print(f"Overlap top 20 antes/despues de semantic_relevance_adjusted_score: {overlap_top20_before_after_semantic_adjustment:.3f}")
print(f"Titulos cambiados tras aplicar semantic_relevance_adjusted_score: {semantic_adjustment_changed_count}")
if pd.notna(latent_ranking_overlap_top20) and latent_ranking_overlap_top20 > 0.85:
    warnings.warn("El nuevo semantic_relevance_adjusted_score apenas modifica el ranking base.")
if {"latent_core_preference_score", "semantic_relevance_score", "semantic_net_score"}.issubset(candidates_scored.columns):
    relevance_delta = (candidates_scored["semantic_relevance_score"].fillna(0.0) - candidates_scored["semantic_net_score"].fillna(0.0)).abs().mean()
    if relevance_delta < 1e-6:
        warnings.warn("semantic_relevance_score no cambia respecto a semantic_net_score pese a existir latent_core_preference_score.")

semantic_diagnostic_cols = [
    col for col in [
        "semantic_net_score", "latent_core_similarity_score", "latent_core_preference_score",
        "semantic_relevance_score", "semantic_relevance_adjusted_score",
        "negative_semantic_relevance_score", "hybrid_score"
    ] if col in candidates_scored.columns
]
print("Columnas semanticas disponibles:", semantic_diagnostic_cols)
if len(semantic_diagnostic_cols) >= 2:
    display(candidates_scored[semantic_diagnostic_cols].corr(numeric_only=True))

def _top20_titles_by_score(df, score_col):
    if score_col not in df.columns:
        return set()
    return set(df.sort_values(score_col, ascending=False).head(20)["title"].astype(str))

semantic_net_top20_titles = _top20_titles_by_score(candidates_scored, "semantic_net_score")
latent_preference_top20_titles = _top20_titles_by_score(candidates_scored, "latent_core_preference_score")
semantic_relevance_top20_titles = _top20_titles_by_score(candidates_scored, "semantic_relevance_score")
hybrid_pre_rerank_top20_titles = _top20_titles_by_score(candidates_scored, "hybrid_score")
overlap_top20_semantic_net_vs_latent_preference = (
    len(semantic_net_top20_titles & latent_preference_top20_titles) / 20
    if semantic_net_top20_titles and latent_preference_top20_titles else np.nan
)
overlap_top20_latent_preference_vs_hybrid = (
    len(latent_preference_top20_titles & hybrid_pre_rerank_top20_titles) / 20
    if latent_preference_top20_titles and hybrid_pre_rerank_top20_titles else np.nan
)
print("Overlap semantic_net vs latent_core_preference:", overlap_top20_semantic_net_vs_latent_preference)
print("Overlap latent_core_preference vs hybrid pre-rerank:", overlap_top20_latent_preference_vs_hybrid)

for score_col in ["semantic_net_score", "latent_core_preference_score", "semantic_relevance_score", "hybrid_score"]:
    if score_col in candidates_scored.columns:
        print(f"Top 20 por {score_col} antes del re-ranking:")
        display(candidates_scored.sort_values(score_col, ascending=False)[["title", "year", "genres", score_col]].head(20))

if "latent_core_preference_score" in candidates_scored.columns and pd.notna(overlap_top20_latent_preference_vs_hybrid) and overlap_top20_latent_preference_vs_hybrid > 0.85:
    warnings.warn("latent_core_preference_score existe pero apenas cambia el ranking base.")

print("Top 30 candidatos por latent_core_preference_score antes del re-ranking:")
if "latent_core_preference_score" in candidates_scored.columns:
    display(candidates_scored.sort_values("latent_core_preference_score", ascending=False)[[
        "title", "year", "genres", "semantic_branch", "branch_strength",
        "latent_core_preference_score", "semantic_net_score", "semantic_relevance_score",
        "nearest_core_anchor_movies"
    ]].head(30))
print("Top 20 por hybrid_score tras semantic_relevance_score y antes del re-ranking:")
display(post_latent_hybrid_top20[["title", "year", "genres", "hybrid_score", "semantic_relevance_score", "dominant_signal"]])

candidates_scored = candidates_scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
candidates_scored["hybrid_rank_raw"] = np.arange(1, len(candidates_scored) + 1)

AUDIT_COLUMNS = [
    "title",
    "genres",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "negative_semantic_relevance_score",
    "semantic_relevance_score",
    "negative_semantic_relevance_raw",
    "semantic_relevance_raw",
    "branch_support_multiplier",
    "negative_latent_similarity_score",
    "latent_core_similarity_score",
    "latent_core_preference_score",
    "nearest_core_anchor_movies",
    "semantic_branch",
    "branch_strength",
    "branch_share",
    "n_negative_anchors_used",
    "n_core_anchors_used",
    "nearest_negative_movies_latent",
    "latent_core_preference_raw",
    "negative_latent_preference_score",
    "branch_prototype_similarity",
    "topk_anchor_similarity",
    "positive_centroid_similarity",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_negative_beta",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "hybrid_score",
]

display(candidates_scored[AUDIT_COLUMNS].head(20))
display(candidates_scored.head(50)["dominant_signal"].value_counts())

Overlap top 20 antes/despues de semantic_relevance_adjusted_score: 0.250
Titulos cambiados tras aplicar semantic_relevance_adjusted_score: 15
Columnas semanticas disponibles: ['semantic_net_score', 'latent_core_similarity_score', 'latent_core_preference_score', 'semantic_relevance_score', 'semantic_relevance_adjusted_score', 'negative_semantic_relevance_score', 'hybrid_score']


,semantic_net_score,latent_core_similarity_score,latent_core_preference_score,semantic_relevance_score,semantic_relevance_adjusted_score,negative_semantic_relevance_score,hybrid_score
semantic_net_score,1.000000,0.452978,0.584650,0.700736,0.592213,0.290437,0.438765
latent_core_similarity_score,0.452978,1.000000,0.800380,0.789665,0.708922,0.835427,0.191301
latent_core_preference_score,0.584650,0.800380,1.000000,0.987777,0.972365,0.655342,0.503090
semantic_relevance_score,0.700736,0.789665,0.987777,1.000000,0.965529,0.629146,0.524881
semantic_relevance_adjusted_score,0.592213,0.708922,0.972365,0.965529,1.000000,0.569171,0.562133
negative_semantic_relevance_score,0.290437,0.835427,0.655342,0.629146,0.569171,1.000000,-0.072988
hybrid_score,0.438765,0.191301,0.503090,0.524881,0.562133,-0.072988,1.000000


Overlap semantic_net vs latent_core_preference: 0.2
Overlap latent_core_preference vs hybrid pre-rerank: 0.05
Top 20 por semantic_net_score antes del re-ranking:


,title,year,genres,semantic_net_score
3754,Black Swan (2010),2010.0,Drama|Thriller,1.000000
809,Pi (1998),1998.0,Drama|Sci-Fi|Thriller,0.999628
1947,Mulholland Drive (2001),2001.0,Crime|Drama|Film-Noir|Mystery|Thriller,0.999255
679,Lost Highway (1997),1997.0,Crime|Drama|Fantasy|Film-Noir|Mystery|Romance,0.998883
2586,Persona (1966),1966.0,Drama,0.998511
2936,Neon Genesis Evangelion: The End of Evangelion...,1997.0,Action|Animation|Drama|Fantasy|Sci-Fi,0.998138
559,"Seventh Seal, The (Sjunde inseglet, Det) (1957)",1957.0,Drama,0.997766
4112,Enemy (2013),2013.0,Mystery|Thriller,0.997394
4548,Mother! (2017),2017.0,Drama|Horror|Mystery|Thriller,0.997022
3581,Antichrist (2009),2009.0,Drama|Fantasy,0.996649


Top 20 por latent_core_preference_score antes del re-ranking:


...

,title,year,genres,semantic_branch,branch_strength,latent_core_preference_score,semantic_net_score,semantic_relevance_score,nearest_core_anchor_movies
2226,Narc (2002),2002.0,Crime|Drama|Thriller,crime_thriller,strong,1.000000,0.928891,0.995762,"Departed, The (2006) | No Country for Old Men ..."
2963,Old Boy (2003),2003.0,Mystery|Thriller,crime_thriller,strong,0.999761,0.963142,0.998587,No Country for Old Men (2007) | Once Upon a Ti...
139,Léon: The Professional (a.k.a. The Professiona...,1994.0,Action|Crime|Drama|Thriller,crime_thriller,strong,0.999521,0.950856,0.997645,Nightcrawler (2014) | Uncut Gems (2019) | Depa...
2568,The Butterfly Effect (2004),2004.0,Drama|Sci-Fi|Thriller,sci_fi_reflective,strong,0.999282,0.986597,0.999529,Interstellar (2014) | Arrival (2016) | Donnie ...
4112,Enemy (2013),2013.0,Mystery|Thriller,crime_thriller,strong,0.999043,0.997394,0.999765,Shutter Island (2010) | Donnie Darko (2001) | ...
3754,Black Swan (2010),2010.0,Drama|Thriller,crime_thriller,strong,0.998804,1.000000,1.000000,"Shutter Island (2010) | Shining, The (1980) | ..."
532,Brazil (1985),1985.0,Fantasy|Sci-Fi,sci_fi_reflective,strong,0.998564,0.840283,0.987050,"Truman Show, The (1998) | Blade Runner 2049 (2..."
4061,New World (Shin-sae-gye) (2013),2013.0,Thriller,crime_thriller,strong,0.998325,0.893522,0.992230,No Country for Old Men (2007) | Once Upon a Ti...
150,"Shawshank Redemption, The (1994)",1994.0,Crime|Drama,crime_thriller,strong,0.998086,0.969471,0.998116,"Departed, The (2006) | Arrival (2016) | No Cou..."
4764,Promising Young Woman (2020),2020.0,Crime|Drama|Thriller,crime_thriller,strong,0.997846,0.357409,0.932658,Parasite (2019) | Once Upon a Time in Hollywoo...


Top 20 por hybrid_score tras semantic_relevance_score y antes del re-ranking:


,title,year,genres,hybrid_score,semantic_relevance_score,dominant_signal
1,Heat (1995),1995.0,Action|Crime|Thriller,0.468213,0.980692,semantic
1688,"Emperor's New Groove, The (2000)",2000.0,Adventure|Animation|Children|Comedy|Fantasy,0.466480,0.746880,semantic
317,Wallace & Gromit: The Best of Aardman Animatio...,1996.0,Adventure|Animation|Comedy,0.452443,0.631976,semantic
75,Crimson Tide (1995),1995.0,Drama|Thriller|War,0.451982,0.878502,semantic
568,"Femme Nikita, La (Nikita) (1990)",1990.0,Action|Crime|Romance|Thriller,0.449655,0.886037,semantic
3438,"Counterfeiters, The (Die Fälscher) (2007)",2007.0,Crime|Drama|War,0.448234,0.699317,semantic
3789,"Lincoln Lawyer, The (2011)",2011.0,Crime|Drama|Thriller,0.444949,0.656699,semantic
3025,Knockin' on Heaven's Door (1997),1997.0,Action|Comedy|Crime|Drama,0.444002,0.858253,semantic
4101,American Hustle (2013),2013.0,Crime|Drama,0.443252,0.799623,semantic
219,In the Line of Fire (1993),1993.0,Action|Thriller,0.438192,0.810690,semantic


,title,genres,genre_profile_score,semantic_profile_score,semantic_core_score,semantic_core_negative_score,semantic_net_score,negative_semantic_relevance_score,semantic_relevance_score,negative_semantic_relevance_raw,...,contrib_year_affinity,penalty_negative_genre,penalty_negative_semantic,penalty_negative_collab,penalty_temporal_mismatch,dominant_signal,semantic_explanation_terms,core_semantic_explanation_terms,core_negative_semantic_terms,hybrid_score
0,Heat (1995),Action|Crime|Thriller,0.218045,0.869340,0.930774,0.101845,0.958675,0.497018,0.980692,0.396534,...,0.075967,0.019559,0.126748,0.0000,0.0,semantic,"tense, neo-noir, atmospheric, bleak","tense, atmospheric, neo-noir, witty, bleak","violent, philosophy",0.468213
1,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.368421,0.000000,0.447499,0.000000,0.493671,0.178285,0.746880,0.129385,...,0.075967,0.019559,0.045466,0.0000,0.0,semantic,,"witty, quirky",,0.466480
2,Wallace & Gromit: The Best of Aardman Animatio...,Adventure|Animation|Comedy,0.263158,0.000000,0.356166,0.000000,0.388682,0.213243,0.631976,0.152509,...,0.075967,0.013970,0.054380,0.0000,0.0,semantic,,witty,,0.452443
3,Crimson Tide (1995),Drama|Thriller|War,0.315789,0.727964,0.770215,0.000000,0.820551,0.287477,0.878502,0.214394,...,0.075967,0.018161,0.073311,0.0000,0.0,semantic,"tense, morality","tense, morality, moral dilemma",,0.451982
4,"Femme Nikita, La (Nikita) (1990)",Action|Crime|Romance|Thriller,0.270677,0.804355,0.899651,0.097942,0.931124,0.441291,0.886037,0.348653,...,0.075967,0.023750,0.112537,0.0000,0.0,semantic,"tense, bittersweet, atmospheric","tense, atmospheric, stylized, bittersweet",secret identity,0.449655
5,"Counterfeiters, The (Die Fälscher) (2007)",Crime|Drama|War,0.300752,0.401832,0.515125,0.138219,0.547282,0.283775,0.699317,0.213717,...,0.075967,0.013970,0.072367,0.0000,0.0,semantic,"cinematography, thought-provoking",cinematography,thought-provoking,0.448234
6,"Lincoln Lawyer, The (2011)",Crime|Drama|Thriller,0.375940,0.000000,0.401687,0.000000,0.438943,0.303516,0.656699,0.230033,...,0.075967,0.020956,0.077402,0.0000,0.0,semantic,,"witty, upper class, serial killer",,0.444949
7,Knockin' on Heaven's Door (1997),Action|Comedy|Crime|Drama,0.496241,0.425510,0.658813,0.000000,0.718541,0.361917,0.858253,0.279354,...,0.075967,0.030735,0.092295,0.0000,0.0,semantic,existentialism,"existentialism, sad ending, existential",,0.444002
8,American Hustle (2013),Crime|Drama,0.285714,0.473903,0.580279,0.000000,0.638496,0.272671,0.799623,0.204102,...,0.075967,0.013970,0.069536,0.0000,0.0,semantic,"bittersweet, politics","stylized, bittersweet, politics",,0.443252
9,In the Line of Fire (1993),Action|Thriller,0.142857,0.636882,0.680047,0.000000,0.738645,0.264446,0.810690,0.196350,...,0.075967,0.016764,0.067438,0.0000,0.0,semantic,"tense, suspenseful","tense, suspenseful",,0.438192


dominant_signal
semantic    50
Name: count, dtype: int64

## 15. Diversidad

In [18]:
def analyze_user_diversity_profile(
    liked_movies,
    tags_clean=None,
    discriminative_tag_profile=None,
    temporal_metrics=None,
    user_branch_profile=None,
):
    genre_rows = []
    if liked_movies is not None and not liked_movies.empty:
        for _, row in liked_movies.dropna(subset=["user_rating_5"]).iterrows():
            movie_weight = max(float(row.get("user_rating_5", 0)) - 3.0, 0.0)
            if movie_weight <= 0:
                continue
            for genre in split_genres(row.get("genres", "")):
                genre_rows.append({"genre": genre, "weight": movie_weight})

    if genre_rows:
        genre_profile = pd.DataFrame(genre_rows).groupby("genre", as_index=False)["weight"].sum()
        genre_total = genre_profile["weight"].sum()
        genre_profile["share"] = genre_profile["weight"] / genre_total if genre_total > 0 else 0.0
        genre_profile = genre_profile.sort_values("share", ascending=False)
        n_positive_genres = int(len(genre_profile))
        top_genre = str(genre_profile.iloc[0]["genre"])
        top_genre_share = float(genre_profile.iloc[0]["share"])
        genre_entropy, genre_diversity_score = _normalized_entropy(genre_profile["share"])
    else:
        n_positive_genres = 0
        top_genre = ""
        top_genre_share = 1.0
        genre_entropy = 0.0
        genre_diversity_score = 0.0

    semantic_category_diversity_score = 0.5
    semantic_category_entropy = 0.0
    n_positive_semantic_categories = 0
    top_semantic_category = ""
    top_semantic_category_share = 0.5
    if discriminative_tag_profile is not None and not discriminative_tag_profile.empty and "semantic_category" in discriminative_tag_profile.columns:
        core_tags = discriminative_tag_profile[
            (discriminative_tag_profile["discriminative_weight"].fillna(0) > 0)
            & discriminative_tag_profile["semantic_category"].notna()
        ].copy()
        if not core_tags.empty:
            category_profile = core_tags.groupby("semantic_category", as_index=False)["discriminative_weight"].sum()
            category_total = category_profile["discriminative_weight"].sum()
            category_profile["share"] = category_profile["discriminative_weight"] / category_total if category_total > 0 else 0.0
            category_profile = category_profile.sort_values("share", ascending=False)
            n_positive_semantic_categories = int(len(category_profile))
            top_semantic_category = str(category_profile.iloc[0]["semantic_category"])
            top_semantic_category_share = float(category_profile.iloc[0]["share"])
            semantic_category_entropy, semantic_category_diversity_score = _normalized_entropy(category_profile["share"])

    branch_positive = user_branch_profile[user_branch_profile["branch_share"].fillna(0) > 0].copy() if user_branch_profile is not None and not user_branch_profile.empty else pd.DataFrame()
    if branch_positive.empty:
        n_user_branches_supported = 0
        top_branch = ""
        top_branch_share = 1.0
        branch_entropy = 0.0
        branch_diversity_score = 0.0
    else:
        branch_positive = branch_positive.sort_values("branch_share", ascending=False)
        n_user_branches_supported = int(len(branch_positive))
        top_branch = str(branch_positive.iloc[0]["semantic_branch"])
        top_branch_share = float(branch_positive.iloc[0]["branch_share"])
        branch_entropy, branch_diversity_score = _normalized_entropy(branch_positive["branch_share"])
    branch_profile_concentration = top_branch_share

    temporal_metrics = temporal_metrics or {}
    q10_year = temporal_metrics.get("q10_year", np.nan)
    q90_year = temporal_metrics.get("q90_year", np.nan)
    classic_tolerance = float(temporal_metrics.get("classic_tolerance", 0.0) or 0.0)
    if pd.isna(q10_year) or pd.isna(q90_year):
        temporal_span = 0.0
        temporal_diversity_score = 0.0
    else:
        temporal_span = max(0.0, float(q90_year) - float(q10_year))
        if temporal_span < 10:
            temporal_diversity_score = 0.0
        elif temporal_span <= 20:
            temporal_diversity_score = 0.3
        elif temporal_span <= 35:
            temporal_diversity_score = 0.6
        else:
            temporal_diversity_score = 1.0
        temporal_diversity_score = _clip01(temporal_diversity_score + 0.25 * classic_tolerance)

    profile_concentration_score = _clip01(
        0.40 * top_branch_share
        + 0.30 * top_genre_share
        + 0.20 * top_semantic_category_share
        + 0.10 * (1 - temporal_diversity_score)
    )
    overall_user_diversity_score = _clip01(
        0.30 * genre_diversity_score
        + 0.35 * branch_diversity_score
        + 0.20 * semantic_category_diversity_score
        + 0.15 * temporal_diversity_score
    )

    if overall_user_diversity_score < 0.35 or profile_concentration_score > 0.70:
        diversity_level = "concentrated"
    elif overall_user_diversity_score > 0.65 and profile_concentration_score <= 0.70:
        diversity_level = "diverse"
    else:
        diversity_level = "medium"

    return {
        "n_positive_genres": n_positive_genres,
        "top_genre": top_genre,
        "top_genre_share": top_genre_share,
        "genre_entropy": genre_entropy,
        "genre_diversity_score": genre_diversity_score,
        "n_positive_semantic_categories": n_positive_semantic_categories,
        "top_semantic_category": top_semantic_category,
        "top_semantic_category_share": top_semantic_category_share,
        "semantic_category_entropy": semantic_category_entropy,
        "semantic_category_diversity_score": semantic_category_diversity_score,
        "n_user_branches_supported": n_user_branches_supported,
        "top_branch": top_branch,
        "top_branch_share": top_branch_share,
        "branch_entropy": branch_entropy,
        "branch_diversity_score": branch_diversity_score,
        "branch_profile_concentration": branch_profile_concentration,
        "temporal_span": temporal_span,
        "temporal_diversity_score": temporal_diversity_score,
        "profile_concentration_score": profile_concentration_score,
        "overall_user_diversity_score": overall_user_diversity_score,
        "diversity_level": diversity_level,
    }


def derive_adaptive_rerank_config(diversity_metrics):
    level = diversity_metrics.get("diversity_level", "medium")
    configs = {
        "concentrated": {
            "max_per_genre": 8,
            "max_per_branch": 8,
            "max_per_signal": 18,
            "diversity_penalty_strength": 0.03,
            "branch_penalty_strength": 0.03,
            "genre_penalty_strength": 0.03,
            "signal_penalty_strength": 0.00,
            "exploration_slots": 1,
            "score_floor_ratio": 0.90,
        },
        "medium": {
            "max_per_genre": 6,
            "max_per_branch": 6,
            "max_per_signal": 17,
            "diversity_penalty_strength": 0.06,
            "branch_penalty_strength": 0.06,
            "genre_penalty_strength": 0.05,
            "signal_penalty_strength": 0.01,
            "exploration_slots": 2,
            "score_floor_ratio": 0.87,
        },
        "diverse": {
            "max_per_genre": 4,
            "max_per_branch": 4,
            "max_per_signal": 16,
            "diversity_penalty_strength": 0.10,
            "branch_penalty_strength": 0.10,
            "genre_penalty_strength": 0.08,
            "signal_penalty_strength": 0.02,
            "exploration_slots": 3,
            "score_floor_ratio": 0.84,
        },
    }
    config = dict(configs.get(level, configs["medium"]))
    config["diversity_level"] = level
    return config


def select_diverse_recommendations(df, top_n=20, max_per_main_genre=3, max_drama_total=5):
    work = df.sort_values("hybrid_score", ascending=False).copy()
    if "main_genre" not in work.columns:
        work["main_genre"] = work["genres"].apply(main_genre_from_genres)
    return work.head(top_n).reset_index(drop=True)


def _branch_slot_limits(user_branch_profile, top_n, diversity_level):
    limits = {}
    for _, row in user_branch_profile.iterrows():
        branch = row["semantic_branch"]
        share = float(row.get("branch_share", 0.0) or 0.0)
        strength = row.get("branch_strength", "unseen")
        expected_slots = top_n * share
        if strength == "strong":
            limit = max(4, int(np.ceil(expected_slots + 2)))
        elif strength == "medium":
            limit = min(3, max(2, int(np.ceil(expected_slots))))
        elif strength == "weak":
            limit = min(2, max(1, int(np.ceil(expected_slots))))
        else:
            limit = 1
        limits[branch] = int(min(top_n, limit))
    return limits


def branch_aware_adaptive_rerank(
    candidates,
    adaptive_rerank_config,
    user_branch_profile,
    top_n=20,
    score_col="hybrid_score",
    genre_col="main_genre",
    branch_col="semantic_branch",
    signal_col="dominant_signal",
    candidate_pool_size=300,
):
    enriched = candidates.copy()
    if "branch_affinity_score" not in enriched.columns or "branch_strength" not in enriched.columns or "branch_share" not in enriched.columns:
        enriched = add_branch_affinity_to_candidates(enriched, user_branch_profile, branch_col=branch_col)
    work = enriched.sort_values(score_col, ascending=False).head(candidate_pool_size).copy()
    if genre_col not in work.columns:
        work[genre_col] = work["genres"].apply(main_genre_from_genres)
    if signal_col not in work.columns:
        work[signal_col] = "unknown"

    work["pre_rerank_rank"] = np.arange(1, len(work) + 1)
    pre_top = work.head(top_n).copy()
    pre_min = float(pre_top[score_col].min()) if not pre_top.empty else float(work[score_col].max())
    pre_avg = float(pre_top[score_col].mean()) if not pre_top.empty else float(work[score_col].mean())
    pool_q75 = float(work[score_col].quantile(0.75)) if len(work) else pre_min
    score_floor_ratio = float(adaptive_rerank_config.get("score_floor_ratio", 0.87))
    score_floor = max(pre_min * score_floor_ratio, pool_q75)

    collab_p80 = work["item_item_collab_score"].quantile(0.80) if "item_item_collab_score" in work.columns else np.inf
    rating_p90 = work["rating_score"].quantile(0.90) if "rating_score" in work.columns else np.inf
    semantic_p90 = work["semantic_relevance_adjusted_score"].quantile(0.90) if "semantic_relevance_adjusted_score" in work.columns else (work["semantic_net_score"].quantile(0.90) if "semantic_net_score" in work.columns else np.inf)

    diversity_level = adaptive_rerank_config.get("diversity_level", "medium")
    branch_limits = _branch_slot_limits(user_branch_profile, top_n, diversity_level)
    max_per_genre = int(adaptive_rerank_config.get("max_per_genre", 6))
    max_per_signal = int(adaptive_rerank_config.get("max_per_signal", 17))
    exploration_budget = int(adaptive_rerank_config.get("exploration_slots", 2))
    genre_penalty_strength = float(adaptive_rerank_config.get("genre_penalty_strength", 0.05))
    branch_penalty_strength = float(adaptive_rerank_config.get("branch_penalty_strength", 0.06))
    signal_penalty_strength = 0.0

    selected_indices = []
    selected_set = set()
    selection_scores = {}
    reasons = {}
    genre_counts = {}
    branch_counts = {}
    signal_counts = {}
    exploration_count = 0

    def is_unseen(row):
        return row.get("branch_strength", "unseen") == "unseen"

    def is_weak(row):
        return row.get("branch_strength", "unseen") == "weak"

    def is_exploration(row):
        return is_unseen(row) or float(row.get("branch_share", 0.0) or 0.0) < 0.03

    def has_strong_unseen_signal(row):
        return (
            float(row.get("item_item_collab_score", 0.0) or 0.0) >= collab_p80
            or float(row.get("rating_score", 0.0) or 0.0) >= rating_p90
            or float(row.get("semantic_net_score", 0.0) or 0.0) >= semantic_p90
        )

    def unseen_is_allowed(row):
        if not is_unseen(row):
            return True
        was_original_top = bool(row["pre_rerank_rank"] <= top_n)
        if branch_counts.get(row.get(branch_col, "general"), 0) >= 1:
            return False
        if not (float(row[score_col]) >= pre_min or was_original_top):
            return False
        if not was_original_top and not has_strong_unseen_signal(row):
            return False
        return True

    def slot_limit_for(row, relaxed=False):
        strength = row.get("branch_strength", "unseen")
        base_limit = branch_limits.get(row.get(branch_col, "general"), 1)
        if strength == "unseen":
            return 1
        if strength == "medium":
            return min(3, base_limit)
        if strength == "weak":
            return min(2, base_limit)
        return base_limit + (1 if relaxed else 0)

    def selection_score_for(row, relaxed=False):
        branch_limit = slot_limit_for(row, relaxed=relaxed)
        branch_soft_start = max(1, int(np.floor(branch_limit / 2)))
        genre_count = genre_counts.get(row.get(genre_col, "Unknown"), 0)
        branch_count = branch_counts.get(row.get(branch_col, "general"), 0)
        signal_count = signal_counts.get(row.get(signal_col, "unknown"), 0)
        strength = row.get("branch_strength", "unseen")
        unsupported_penalty = 0.0 if strength in ["strong", "medium"] else (0.03 if strength == "weak" else 0.08)
        if relaxed and strength == "weak":
            unsupported_penalty *= 0.5
        return (
            float(row[score_col])
            + 0.06 * float(row.get("branch_affinity_score", 0.15) or 0.15)
            - genre_penalty_strength * max(0, genre_count - 1)
            - branch_penalty_strength * max(0, branch_count - branch_soft_start)
            - unsupported_penalty
        )

    def add_candidate(idx, row, reason, adjusted_score):
        nonlocal exploration_count
        selected_indices.append(idx)
        selected_set.add(idx)
        selection_scores[idx] = adjusted_score
        if reason.startswith("fallback_fill"):
            reasons[idx] = reason
        elif is_unseen(row):
            reasons[idx] = "selected_exploration_branch"
        elif is_weak(row):
            reasons[idx] = "selected_relaxed_weak_profile_branch" if "relaxed" in reason else "selected_weak_profile_branch"
        elif row["pre_rerank_rank"] <= top_n:
            reasons[idx] = "kept_from_original_top20"
        else:
            reasons[idx] = reason
        genre = row.get(genre_col, "Unknown")
        branch = row.get(branch_col, "general")
        signal = row.get(signal_col, "unknown")
        genre_counts[genre] = genre_counts.get(genre, 0) + 1
        branch_counts[branch] = branch_counts.get(branch, 0) + 1
        signal_counts[signal] = signal_counts.get(signal, 0) + 1
        if is_exploration(row):
            exploration_count += 1

    def reason_for(row, relaxed=False):
        branch = row.get(branch_col, "general")
        if is_unseen(row):
            return "selected_exploration_branch"
        if is_weak(row):
            return "selected_relaxed_weak_profile_branch" if relaxed else "selected_weak_profile_branch"
        if row.get("branch_strength", "unseen") == "medium" and branch_counts.get(branch, 0) >= slot_limit_for(row, relaxed=False):
            return "kept_high_score_medium_branch"
        return "selected_relaxed_profile_branch" if relaxed else "selected_profile_branch"

    def candidate_allowed(row, relaxed=False):
        branch = row.get(branch_col, "general")
        genre = row.get(genre_col, "Unknown")
        signal = row.get(signal_col, "unknown")
        if is_unseen(row):
            if not unseen_is_allowed(row):
                return False
        current_branch_count = branch_counts.get(branch, 0)
        base_branch_limit = slot_limit_for(row, relaxed=False)
        if current_branch_count >= slot_limit_for(row, relaxed=relaxed):
            if row.get("branch_strength", "unseen") == "medium":
                high_score_medium = (
                    current_branch_count < 4
                    and row["pre_rerank_rank"] <= top_n
                    and float(row[score_col]) >= score_floor
                    and branch != "animation_family"
                )
                if not high_score_medium:
                    return False
            else:
                return False
        if branch == "animation_family" and row.get("branch_strength", "unseen") == "medium" and current_branch_count >= 3:
            return False
        if is_weak(row) and branch_counts.get(branch, 0) >= 2:
            if not relaxed:
                return False
            if not (row["pre_rerank_rank"] <= top_n or float(row[score_col]) >= pre_min):
                return False
        if genre_counts.get(genre, 0) >= max_per_genre + (1 if relaxed else 0):
            return False
        if is_exploration(row) and exploration_count >= exploration_budget + (0 if is_unseen(row) else (1 if relaxed else 0)):
            return False
        return True

    def run_pass(current_score_floor, relaxed=False):
        while len(selected_indices) < top_n:
            best_idx = None
            best_score = -np.inf
            best_row = None
            for idx, row in work.iterrows():
                if idx in selected_set or float(row[score_col]) < current_score_floor:
                    continue
                if not candidate_allowed(row, relaxed=relaxed):
                    continue
                adjusted_score = selection_score_for(row, relaxed=relaxed)
                if adjusted_score > best_score:
                    best_idx = idx
                    best_score = adjusted_score
                    best_row = row
            if best_idx is None:
                break
            add_candidate(best_idx, best_row, reason_for(best_row, relaxed=relaxed), best_score)

    run_pass(score_floor, relaxed=False)
    if len(selected_indices) < top_n:
        run_pass(pre_min * 0.80, relaxed=True)
    if len(selected_indices) < top_n:
        for idx, row in work.iterrows():
            if idx in selected_set:
                continue
            branch = row.get(branch_col, "general")
            if is_unseen(row) and branch_counts.get(branch, 0) >= 1:
                fallback_reason = "fallback_fill_unseen_branch"
            elif branch == "animation_family" and row.get("branch_strength", "unseen") == "medium" and branch_counts.get(branch, 0) >= 3:
                fallback_reason = "fallback_fill_animation_family_over_cap"
            else:
                fallback_reason = "fallback_fill"
            add_candidate(idx, row, fallback_reason, float(row[score_col]))
            if len(selected_indices) >= top_n:
                break

    selected = work.loc[selected_indices].copy()
    selected_index_series = pd.Series(selected.index, index=selected.index)
    selected["final_rank"] = np.arange(1, len(selected) + 1)
    selected["was_in_pre_rerank_top20"] = selected["pre_rerank_rank"] <= top_n
    selected["entered_by_rerank"] = ~selected["was_in_pre_rerank_top20"]
    selected["selected_by_rerank"] = True
    selected["rerank_reason"] = selected_index_series.map(reasons).fillna("fallback_fill")
    selected["rerank_selection_score"] = selected_index_series.map(selection_scores).fillna(selected[score_col])
    selected["branch_slot_limit"] = selected.apply(lambda row: branch_limits.get(row.get(branch_col, "general"), 1), axis=1)
    selected["rerank_score_floor"] = score_floor
    selected["rerank_score_floor_ratio"] = score_floor_ratio
    selected.attrs["rerank_summary"] = {
        "score_floor": score_floor,
        "score_floor_ratio": score_floor_ratio,
        "pre_rerank_top_n_min_score": pre_min,
        "pre_rerank_top_n_avg_score": pre_avg,
        "candidate_pool_q75_score": pool_q75,
        "exploration_budget": exploration_budget,
        "collab_p80": collab_p80,
        "rating_p90": rating_p90,
        "semantic_p90": semantic_p90,
    }
    return selected.sort_values("final_rank").reset_index(drop=True)


def compare_weight_configs(base_df, weight_configs, top_n=20):
    rows = []
    for config_name, config_weights in weight_configs.items():
        scored = compute_hybrid_score(base_df, config_weights)
        scored = scored.sort_values("hybrid_score", ascending=False).reset_index(drop=True)
        if "main_genre" not in scored.columns:
            scored["main_genre"] = scored["genres"].apply(main_genre_from_genres)
        selected = scored.head(top_n).copy()
        rows.append(
            {
                "config": config_name,
                "avg_genre_profile_score": selected["genre_profile_score"].mean(),
                "avg_semantic_profile_score": selected["semantic_profile_score"].mean(),
                "avg_negative_genre_score": selected["negative_genre_score"].mean(),
                "avg_negative_semantic_score": selected["negative_semantic_score"].mean(),
                "avg_item_item_collab_score": selected["item_item_collab_score"].mean(),
                "avg_item_item_negative_collab_score": selected["item_item_negative_collab_score"].mean(),
                "avg_rating_mean": selected["rating_mean"].mean(),
                "avg_rating_count": selected["rating_count"].mean(),
                "avg_popularity_score": selected["popularity_score"].mean(),
                "avg_year_affinity_score": selected["year_affinity_score"].mean() if "year_affinity_score" in selected.columns else np.nan,
                "avg_temporal_mismatch_penalty": selected["temporal_mismatch_penalty"].mean() if "temporal_mismatch_penalty" in selected.columns else np.nan,
                "n_temporal_outliers": selected["is_temporal_outlier"].sum() if "is_temporal_outlier" in selected.columns else 0,
                "n_distinct_main_genres": selected["main_genre"].nunique() if "main_genre" in selected.columns else selected["main_genre"].nunique(),
                "dominant_signal_counts": selected["dominant_signal"].value_counts().to_dict(),
                "top_titles": "; ".join(selected["title"].head(5).astype(str)),
            }
        )
    return pd.DataFrame(rows)


if "main_genre" not in candidates_scored.columns:
    candidates_scored["main_genre"] = candidates_scored["genres"].apply(main_genre_from_genres)
candidates_scored["semantic_branch"] = candidates_scored.apply(assign_semantic_branch, axis=1)

user_branch_profile = build_user_branch_profile(
    liked_movies=liked_movies,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
    assign_branch_func=assign_semantic_branch,
)
display(user_branch_profile.sort_values("branch_share", ascending=False))
user_branch_profile.to_csv(REPORTS_RESULTADOS / "user_branch_profile.csv", index=False)

candidates_scored = add_branch_affinity_to_candidates(
    candidates_scored,
    user_branch_profile,
    branch_col="semantic_branch",
)

pre_semantic_adjustment_top20 = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy() if "hybrid_score" in candidates_scored.columns else pd.DataFrame()
pre_semantic_adjustment_titles = set(pre_semantic_adjustment_top20["title"].astype(str)) if not pre_semantic_adjustment_top20.empty else set()
candidates_scored = compute_semantic_relevance_scores(candidates_scored)
candidates_scored = compute_hybrid_score(candidates_scored, weights)
post_semantic_adjustment_top20 = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
post_semantic_adjustment_titles = set(post_semantic_adjustment_top20["title"].astype(str))
overlap_top20_before_after_semantic_adjustment = (
    len(pre_semantic_adjustment_titles & post_semantic_adjustment_titles) / 20
    if pre_semantic_adjustment_titles else np.nan
)
print("Top 20 por hybrid_score antes de recalcular con semantic_relevance_adjusted_score:")
if not pre_semantic_adjustment_top20.empty:
    display(pre_semantic_adjustment_top20[["title", "year", "genres", "hybrid_score"]])
print("Top 20 por hybrid_score despues de recalcular con semantic_relevance_adjusted_score:")
display(post_semantic_adjustment_top20[["title", "year", "genres", "hybrid_score", "semantic_relevance_adjusted_score", "dominant_signal"]])
print("Overlap top20 antes/despues de semantic_relevance_adjusted_score:", overlap_top20_before_after_semantic_adjustment)
if pd.notna(overlap_top20_before_after_semantic_adjustment) and overlap_top20_before_after_semantic_adjustment > 0.85:
    warnings.warn("El nuevo semantic_relevance_adjusted_score apenas modifica el ranking base.")

branch_assignment_cols = [
    "movieId", "title", "year", "genres", "semantic_branch", "branch_strength", "branch_share",
    "latent_core_preference_score", "semantic_net_score", "semantic_relevance_score",
    "semantic_relevance_adjusted_score", "hybrid_score", "core_semantic_explanation_terms",
    "semantic_explanation_terms", "nearest_core_anchor_movies"
]
branch_assignment_diagnostics = candidates_scored.sort_values("hybrid_score", ascending=False).head(150).copy()
branch_assignment_diagnostics["branch_assignment_warning"] = branch_assignment_diagnostics.apply(_branch_warning_for_row, axis=1)
branch_assignment_available_cols = [col for col in branch_assignment_cols + ["branch_assignment_warning"] if col in branch_assignment_diagnostics.columns]
display(branch_assignment_diagnostics[branch_assignment_available_cols])
branch_assignment_diagnostics[branch_assignment_available_cols].to_csv(REPORTS_RESULTADOS / "branch_assignment_diagnostics.csv", index=False)
branch_warning_counts = branch_assignment_diagnostics["branch_assignment_warning"].value_counts()
if any(idx for idx in branch_warning_counts.index if idx):
    warnings.warn(f"Avisos de asignacion de ramas: {branch_warning_counts.to_dict()}")

print("Distribucion de semantic_branch en candidatos:")
display(candidates_scored["semantic_branch"].value_counts(dropna=False))

diversity_metrics = analyze_user_diversity_profile(
    liked_movies=liked_movies,
    tags_clean=tags_clean,
    discriminative_tag_profile=discriminative_tag_profile,
    temporal_metrics=temporal_metrics,
    user_branch_profile=user_branch_profile,
)
diversity_metrics_df = pd.DataFrame(diversity_metrics.items(), columns=["metric", "value"])
display(diversity_metrics_df.sort_values("metric"))
diversity_metrics_df.to_csv(REPORTS_RESULTADOS / "diversity_profile_metrics.csv", index=False)

adaptive_rerank_config = derive_adaptive_rerank_config(diversity_metrics)
adaptive_rerank_config_df = pd.DataFrame(adaptive_rerank_config.items(), columns=["parameter", "value"])
display(adaptive_rerank_config_df)
adaptive_rerank_config_df.to_csv(REPORTS_RESULTADOS / "adaptive_rerank_config.csv", index=False)

pre_rerank_top = candidates_scored.sort_values("hybrid_score", ascending=False).head(20).copy()
pre_rerank_titles = set(pre_rerank_top["title"].astype(str))

weight_configs_for_comparison = dict(WEIGHT_CONFIGS)
weight_configs_for_comparison["adaptive"] = adaptive_weights
display(compare_weight_configs(candidates_scored_base, weight_configs_for_comparison, top_n=20))

recommendations = branch_aware_adaptive_rerank(
    candidates=candidates_scored,
    adaptive_rerank_config=adaptive_rerank_config,
    user_branch_profile=user_branch_profile,
    top_n=20,
    score_col="hybrid_score",
    genre_col="main_genre",
    branch_col="semantic_branch",
    signal_col="dominant_signal",
    candidate_pool_size=300,
)
rerank_summary = recommendations.attrs.get("rerank_summary", {})
recommendations = recommendations.sort_values("final_rank").reset_index(drop=True)
print("Top 20 final despues del branch-aware reranking:")
display(recommendations[["title", "year", "genres", "semantic_branch", "branch_strength", "hybrid_score", "final_rank", "rerank_reason", "dominant_signal"]])

post_rerank_titles = set(recommendations["title"].astype(str))
rerank_entered_titles = sorted(post_rerank_titles - pre_rerank_titles)
rerank_removed_titles = sorted(pre_rerank_titles - post_rerank_titles)
post_decades = ((pd.to_numeric(recommendations["year"], errors="coerce") // 10) * 10).astype("Int64")
unseen_mask = recommendations["branch_strength"] == "unseen"
weak_mask = recommendations["branch_strength"] == "weak"
animation_family_mask = recommendations["semantic_branch"] == "animation_family"
animation_dark_surreal_mask = recommendations["semantic_branch"] == "animation_dark_surreal"
psychological_thriller_mask = recommendations["semantic_branch"] == "psychological_thriller"
psychological_drama_mask = recommendations["semantic_branch"] == "psychological_drama"
exploration_mask = unseen_mask | (recommendations["branch_share"] < 0.03)
unseen_branch_counts = recommendations.loc[unseen_mask, "semantic_branch"].value_counts()
weak_branch_counts = recommendations.loc[weak_mask, "semantic_branch"].value_counts()
unseen_titles = " | ".join(recommendations.loc[unseen_mask, "title"].astype(str))
weak_titles = " | ".join(recommendations.loc[weak_mask, "title"].astype(str))
rerank_diagnostic_rows = [
    {"metric": "diversity_level", "value": diversity_metrics["diversity_level"]},
    {"metric": "overall_user_diversity_score", "value": diversity_metrics["overall_user_diversity_score"]},
    {"metric": "profile_concentration_score", "value": diversity_metrics["profile_concentration_score"]},
    {"metric": "branch_diversity_score", "value": diversity_metrics["branch_diversity_score"]},
    {"metric": "top_branch", "value": diversity_metrics["top_branch"]},
    {"metric": "top_branch_share", "value": diversity_metrics["top_branch_share"]},
    {"metric": "score_floor", "value": rerank_summary.get("score_floor")},
    {"metric": "score_floor_ratio", "value": adaptive_rerank_config["score_floor_ratio"]},
    {"metric": "exploration_budget", "value": rerank_summary.get("exploration_budget")},
    {"metric": "avg_hybrid_score_pre_rerank_top20", "value": pre_rerank_top["hybrid_score"].mean()},
    {"metric": "avg_hybrid_score_post_rerank_top20", "value": recommendations["hybrid_score"].mean()},
    {"metric": "min_hybrid_score_pre_rerank_top20", "value": pre_rerank_top["hybrid_score"].min()},
    {"metric": "min_hybrid_score_post_rerank_top20", "value": recommendations["hybrid_score"].min()},
    {"metric": "n_entered_by_rerank", "value": int(recommendations["entered_by_rerank"].sum())},
    {"metric": "n_exploration_branches_top20", "value": int(exploration_mask.sum())},
    {"metric": "n_fallback_fill", "value": int(recommendations["rerank_reason"].astype(str).str.startswith("fallback_fill").sum())},

    {"metric": "n_unseen_branches_top20", "value": int(unseen_branch_counts.size)},
    {"metric": "n_unseen_movies_top20", "value": int(unseen_mask.sum())},
    {"metric": "unseen_branch_distribution_top20", "value": unseen_branch_counts.to_dict()},
    {"metric": "n_weak_movies_top20", "value": int(weak_mask.sum())},
    {"metric": "weak_branch_distribution_top20", "value": weak_branch_counts.to_dict()},
    {"metric": "n_selected_exploration_branch", "value": int((recommendations["rerank_reason"] == "selected_exploration_branch").sum())},
    {"metric": "n_fallback_fill_unseen_branch", "value": int((recommendations["rerank_reason"] == "fallback_fill_unseen_branch").sum())},
    {"metric": "max_unseen_branch_count_top20", "value": int(unseen_branch_counts.max()) if not unseen_branch_counts.empty else 0},
    {"metric": "max_weak_branch_count_top20", "value": int(weak_branch_counts.max()) if not weak_branch_counts.empty else 0},
    {"metric": "titles_unseen_branch_top20", "value": unseen_titles},
    {"metric": "titles_weak_branch_top20", "value": weak_titles},
    {"metric": "titles_entered_by_rerank", "value": "; ".join(rerank_entered_titles)},
    {"metric": "titles_removed_by_rerank", "value": "; ".join(rerank_removed_titles)},
    {"metric": "semantic_branch_distribution_top20", "value": recommendations["semantic_branch"].value_counts().to_dict()},
    {"metric": "branch_strength_distribution_top20", "value": recommendations["branch_strength"].value_counts().to_dict()},
    {"metric": "main_genre_distribution_top20", "value": recommendations["main_genre"].value_counts().to_dict()},
    {"metric": "dominant_signal_distribution_top20", "value": recommendations["dominant_signal"].value_counts().to_dict()},
    {"metric": "decade_distribution_top20", "value": post_decades.value_counts().sort_index().to_dict()},
]
rerank_diagnostics_df = pd.DataFrame(rerank_diagnostic_rows)
display(rerank_diagnostics_df)
rerank_diagnostics_df.to_csv(REPORTS_RESULTADOS / "diagnostico_reranking_diversificado.csv", index=False)

latent_semantic_diagnostics.update({
    "mean_latent_core_similarity_top20": recommendations["latent_core_similarity_score"].mean() if "latent_core_similarity_score" in recommendations.columns else 0.0,
    "mean_semantic_relevance_top20": recommendations["semantic_relevance_score"].mean() if "semantic_relevance_score" in recommendations.columns else 0.0,
    "mean_negative_latent_similarity_top20": recommendations["negative_latent_similarity_score"].mean() if "negative_latent_similarity_score" in recommendations.columns else 0.0,
})
latent_semantic_diagnostics_df = pd.DataFrame(latent_semantic_diagnostics.items(), columns=["metric", "value"])
display(latent_semantic_diagnostics_df)
latent_semantic_diagnostics_df.to_csv(REPORTS_RESULTADOS / "latent_semantic_diagnostics.csv", index=False)

anchor_branch_distribution = core_positive_anchors["semantic_branch"].value_counts().to_dict() if "core_positive_anchors" in globals() and core_positive_anchors is not None and not core_positive_anchors.empty and "semantic_branch" in core_positive_anchors.columns else {}
top20_branch_distribution = recommendations["semantic_branch"].value_counts().to_dict() if "semantic_branch" in recommendations.columns else {}
top20_titles_by_latent_core_preference = ""
if "latent_core_preference_score" in candidates_scored.columns:
    top20_titles_by_latent_core_preference = "; ".join(candidates_scored.sort_values("latent_core_preference_score", ascending=False).head(20)["title"].astype(str))
liked_available = 0
negative_available = 0
if "movie_id_to_latent_idx" in globals():
    latent_movie_ids = set(movie_id_to_latent_idx.keys())
    liked_available = len(set(liked_movies["movieId"].dropna().astype(int)) & latent_movie_ids) if "movieId" in liked_movies.columns else 0
    negative_seed = disliked_movies[pd.to_numeric(disliked_movies.get("user_rating_5"), errors="coerce") <= 2.5] if disliked_movies is not None and not disliked_movies.empty else pd.DataFrame()
    negative_available = len(set(negative_seed["movieId"].dropna().astype(int)) & latent_movie_ids) if "movieId" in negative_seed.columns else 0

latent_core_preference_diagnostics = {
    "n_liked_movies_available": int(liked_available),
    "n_positive_anchor_evidence": int(len(positive_anchor_evidence)) if "positive_anchor_evidence" in globals() and positive_anchor_evidence is not None else 0,
    "n_core_anchors_selected": int(len(core_positive_anchors)) if "core_positive_anchors" in globals() and core_positive_anchors is not None else 0,
    "n_negative_movies_available": int(negative_available),
    "mean_anchor_weight": float(core_positive_anchors["anchor_weight"].mean()) if "core_positive_anchors" in globals() and core_positive_anchors is not None and not core_positive_anchors.empty and "anchor_weight" in core_positive_anchors.columns else 0.0,
    "mean_centrality_score_anchors": float(core_positive_anchors["centrality_score"].mean()) if "core_positive_anchors" in globals() and core_positive_anchors is not None and not core_positive_anchors.empty and "centrality_score" in core_positive_anchors.columns else 0.0,
    "mean_latent_core_preference_top20": float(recommendations["latent_core_preference_score"].mean()) if "latent_core_preference_score" in recommendations.columns else 0.0,
    "mean_latent_core_similarity_old_top20": float(recommendations["latent_core_similarity_score"].mean()) if "latent_core_similarity_score" in recommendations.columns else 0.0,
    "mean_semantic_net_score_top20": float(recommendations["semantic_net_score"].mean()) if "semantic_net_score" in recommendations.columns else 0.0,
    "mean_semantic_relevance_score_top20": float(recommendations["semantic_relevance_score"].mean()) if "semantic_relevance_score" in recommendations.columns else 0.0,
    "mean_negative_latent_preference_top20": float(recommendations["negative_latent_preference_score"].mean()) if "negative_latent_preference_score" in recommendations.columns else 0.0,
    "top_anchor_branch_distribution": anchor_branch_distribution,
    "top20_branch_distribution": top20_branch_distribution,
    "top20_titles_by_latent_core_preference": top20_titles_by_latent_core_preference,
    "overlap_top20_before_after_latent": latent_ranking_overlap_top20 if "latent_ranking_overlap_top20" in globals() else np.nan,
    "n_titles_changed_after_latent": n_titles_changed_after_latent if "n_titles_changed_after_latent" in globals() else np.nan,
}
latent_core_preference_diagnostics_df = pd.DataFrame(latent_core_preference_diagnostics.items(), columns=["metric", "value"])
display(latent_core_preference_diagnostics_df)
latent_core_preference_diagnostics_df.to_csv(REPORTS_RESULTADOS / "latent_core_preference_diagnostics.csv", index=False)
if pd.notna(latent_core_preference_diagnostics.get("overlap_top20_before_after_latent")) and latent_core_preference_diagnostics["overlap_top20_before_after_latent"] > 0.85:
    warnings.warn("La senal latent_core_preference_score apenas esta modificando el ranking base.")
if "pre_latent_top20_titles" in globals() and post_rerank_titles == pre_latent_top20_titles:
    warnings.warn("El top final es identico al ranking anterior al cambio latente.")
medium_branch_counts = recommendations.loc[recommendations["branch_strength"] == "medium", "semantic_branch"].value_counts() if "branch_strength" in recommendations.columns else pd.Series(dtype=int)
if not medium_branch_counts.empty and int(medium_branch_counts.max()) > 3:
    warnings.warn("Hay una rama medium con mas de 3 peliculas en el top 20.")
if int(animation_family_mask.sum()) > 3:
    warnings.warn("animation_family supera 3 peliculas en el top 20.")

overlap_top20_pre_rerank_vs_final = len(pre_rerank_titles & post_rerank_titles) / 20 if pre_rerank_titles else np.nan
final_model_stability_diagnostics = {
    "n_candidates": int(len(candidates_scored)),
    "n_recommendations": int(len(recommendations)),
    "overlap_top20_semantic_net_vs_latent_preference": overlap_top20_semantic_net_vs_latent_preference if "overlap_top20_semantic_net_vs_latent_preference" in globals() else np.nan,
    "overlap_top20_before_after_semantic_adjustment": overlap_top20_before_after_semantic_adjustment if "overlap_top20_before_after_semantic_adjustment" in globals() else np.nan,
    "overlap_top20_pre_rerank_vs_final": overlap_top20_pre_rerank_vs_final,
    "mean_hybrid_score_pre_rerank_top20": float(pre_rerank_top["hybrid_score"].mean()),
    "mean_hybrid_score_final_top20": float(recommendations["hybrid_score"].mean()),
    "min_hybrid_score_pre_rerank_top20": float(pre_rerank_top["hybrid_score"].min()),
    "min_hybrid_score_final_top20": float(recommendations["hybrid_score"].min()),
    "mean_latent_core_preference_final_top20": float(recommendations["latent_core_preference_score"].mean()) if "latent_core_preference_score" in recommendations.columns else 0.0,
    "mean_semantic_relevance_adjusted_final_top20": float(recommendations["semantic_relevance_adjusted_score"].mean()) if "semantic_relevance_adjusted_score" in recommendations.columns else 0.0,
    "n_animation_family_top20": int(animation_family_mask.sum()),
    "n_animation_dark_surreal_top20": int(animation_dark_surreal_mask.sum()),
    "n_psychological_thriller_top20": int(psychological_thriller_mask.sum()),
    "n_psychological_drama_top20": int(psychological_drama_mask.sum()),
    "n_unseen_branch_top20": int(unseen_mask.sum()),
    "n_fallback_fill": int(recommendations["rerank_reason"].astype(str).str.startswith("fallback_fill").sum()),
    "branch_distribution_top20": recommendations["semantic_branch"].value_counts().to_dict(),
    "branch_strength_distribution_top20": recommendations["branch_strength"].value_counts().to_dict(),
    "dominant_signal_distribution_top20": recommendations["dominant_signal"].value_counts().to_dict(),
    "titles_entered_by_rerank": "; ".join(rerank_entered_titles),
    "titles_removed_by_rerank": "; ".join(rerank_removed_titles),
}
final_model_stability_diagnostics_df = pd.DataFrame(final_model_stability_diagnostics.items(), columns=["metric", "value"])
display(final_model_stability_diagnostics_df)
final_model_stability_diagnostics_df.to_csv(REPORTS_RESULTADOS / "final_model_stability_diagnostics.csv", index=False)

print("Media latent_core_similarity_score top 20:", latent_semantic_diagnostics["mean_latent_core_similarity_top20"])
print("Media semantic_net_score top 20:", recommendations["semantic_net_score"].mean() if "semantic_net_score" in recommendations.columns else np.nan)
print("Media semantic_relevance_score top 20:", latent_semantic_diagnostics["mean_semantic_relevance_top20"])

print("Perfil de ramas del usuario:")
display(user_branch_profile.sort_values("branch_share", ascending=False))
print(f"Nivel de diversidad detectado: {diversity_metrics['diversity_level']}")
print(f"Branch diversity score: {diversity_metrics['branch_diversity_score']:.3f}")
print(f"Top branch: {diversity_metrics['top_branch']} ({diversity_metrics['top_branch_share']:.3f})")
print("Configuracion de re-ranking aplicada:")
print(adaptive_rerank_config)
print(f"Peliculas que entraron por rerank: {len(rerank_entered_titles)}")
print(f"Peliculas que salieron por rerank: {len(rerank_removed_titles)}")
print(f"Media hybrid_score antes/despues: {pre_rerank_top['hybrid_score'].mean():.3f} / {recommendations['hybrid_score'].mean():.3f}")
print(f"Exploraciones usadas: {int(exploration_mask.sum())}")
print("Distribucion de semantic_branch antes del rerank:")
display(pre_rerank_top["semantic_branch"].value_counts(dropna=False))
print("Distribucion de semantic_branch despues del rerank:")
display(recommendations["semantic_branch"].value_counts(dropna=False))
print("Distribucion de branch_strength despues del rerank:")
display(recommendations["branch_strength"].value_counts(dropna=False))
print("Distribucion de main_genre despues del rerank:")
display(recommendations["main_genre"].value_counts(dropna=False))
print("Distribucion de dominant_signal despues del rerank:")
display(recommendations["dominant_signal"].value_counts(dropna=False))
print("Sanity branch_strength top 20:")
display(recommendations["branch_strength"].value_counts(dropna=False))
print("Sanity semantic_branch top 20:")
display(recommendations["semantic_branch"].value_counts(dropna=False))
print(f"Peliculas unseen en top 20: {int(unseen_mask.sum())}")
max_unseen_branch_count = int(unseen_branch_counts.max()) if not unseen_branch_counts.empty else 0
print(f"Max peliculas por rama unseen: {max_unseen_branch_count}")
if max_unseen_branch_count > 1:
    warnings.warn("Hay una rama unseen con mas de una pelicula en el top 20.")
unseen_low_score = recommendations[unseen_mask & (recommendations["hybrid_score"] < rerank_summary.get("pre_rerank_top_n_min_score", -np.inf)) & (~recommendations["was_in_pre_rerank_top20"])]
if not unseen_low_score.empty:
    warnings.warn("Hay peliculas unseen fuera del top 20 original con hybrid_score inferior al minimo original.")

final_display_cols = [
    "title", "year", "main_genre", "semantic_branch", "branch_strength",
    "branch_share", "hybrid_score", "semantic_relevance_score", "semantic_relevance_adjusted_score",
    "latent_core_preference_score", "latent_core_similarity_score", "semantic_net_score",
    "negative_latent_preference_score", "negative_latent_similarity_score",
    "nearest_core_anchor_movies", "nearest_liked_movies_latent", "pre_rerank_rank", "final_rank",
    "rerank_reason", "dominant_signal",
]
display(recommendations[final_display_cols])


,semantic_branch,branch_weight,branch_share,branch_rank,branch_strength,n_seed_movies,example_liked_movies
0,crime_thriller,13.5,0.254717,1.0,strong,12,City of God (Cidade de Deus) (2002) | Nightcra...
1,sci_fi_reflective,11.0,0.207547,2.0,strong,9,Blade Runner 2049 (2017) | Mr. Nobody (2009) |...
2,general,6.0,0.113208,3.0,medium,6,Cinema Paradiso (Nuovo cinema Paradiso) (1989)...
3,psychological_thriller,5.5,0.103774,4.0,medium,5,Fear and Loathing in Las Vegas (1998) | Once U...
4,animation_family,4.5,0.084906,5.0,medium,4,"Toy Story 3 (2010) | Monsters, Inc. (2001) | T..."
5,romantic_comedy_drama,4.0,0.075472,6.0,weak,4,Edward Scissorhands (1990) | Annie Hall (1977)...
6,animation_dark_surreal,3.0,0.056604,7.0,weak,2,Shrek 2 (2004) | Toy Story 2 (1999)
7,surreal_fantasy,2.5,0.047170,8.0,weak,2,"Great Beauty, The (Grande Bellezza, La) (2013)..."
8,psychological_drama,2.0,0.037736,9.0,weak,2,Whiplash (2014) | Intouchables (2011)
9,comedy_satire,1.0,0.018868,10.0,unseen,1,Triangle of Sadness (2022)


Top 20 por hybrid_score antes de recalcular con semantic_relevance_adjusted_score:


,title,year,genres,hybrid_score
0,Heat (1995),1995.0,Action|Crime|Thriller,0.468213
1,"Emperor's New Groove, The (2000)",2000.0,Adventure|Animation|Children|Comedy|Fantasy,0.466480
2,Wallace & Gromit: The Best of Aardman Animatio...,1996.0,Adventure|Animation|Comedy,0.452443
3,Crimson Tide (1995),1995.0,Drama|Thriller|War,0.451982
4,"Femme Nikita, La (Nikita) (1990)",1990.0,Action|Crime|Romance|Thriller,0.449655
5,"Counterfeiters, The (Die Fälscher) (2007)",2007.0,Crime|Drama|War,0.448234
6,"Lincoln Lawyer, The (2011)",2011.0,Crime|Drama|Thriller,0.444949
7,Knockin' on Heaven's Door (1997),1997.0,Action|Comedy|Crime|Drama,0.444002
8,American Hustle (2013),2013.0,Crime|Drama,0.443252
9,In the Line of Fire (1993),1993.0,Action|Thriller,0.438192


Top 20 por hybrid_score despues de recalcular con semantic_relevance_adjusted_score:


,title,year,genres,hybrid_score,semantic_relevance_adjusted_score,dominant_signal
0,Heat (1995),1995.0,Action|Crime|Thriller,0.473036,0.982174,semantic
1,"Emperor's New Groove, The (2000)",2000.0,Adventure|Animation|Children|Comedy|Fantasy,0.466870,0.750439,semantic
3,Crimson Tide (1995),1995.0,Drama|Thriller|War,0.455069,0.921918,semantic
4,"Femme Nikita, La (Nikita) (1990)",1990.0,Action|Crime|Romance|Thriller,0.453098,0.923927,semantic
2,Wallace & Gromit: The Best of Aardman Animatio...,1996.0,Adventure|Animation|Comedy,0.450839,0.656540,semantic
5,"Counterfeiters, The (Die Fälscher) (2007)",2007.0,Crime|Drama|War,0.449271,0.802912,semantic
7,Knockin' on Heaven's Door (1997),1997.0,Action|Comedy|Crime|Drama,0.447245,0.911624,semantic
6,"Lincoln Lawyer, The (2011)",2011.0,Crime|Drama|Thriller,0.446300,0.778559,semantic
8,American Hustle (2013),2013.0,Crime|Drama,0.444808,0.876977,semantic
9,In the Line of Fire (1993),1993.0,Action|Thriller,0.439730,0.884258,semantic


...

branch_strength
strong    11
medium     5
weak       4
Name: count, dtype: int64

Sanity semantic_branch top 20:


semantic_branch
crime_thriller            8
sci_fi_reflective         3
animation_family          2
general                   2
animation_dark_surreal    2
psychological_thriller    1
romantic_comedy_drama     1
surreal_fantasy           1
Name: count, dtype: int64

Peliculas unseen en top 20: 0
Max peliculas por rama unseen: 0


,title,year,main_genre,semantic_branch,branch_strength,branch_share,hybrid_score,semantic_relevance_score,semantic_relevance_adjusted_score,latent_core_preference_score,latent_core_similarity_score,semantic_net_score,negative_latent_preference_score,negative_latent_similarity_score,nearest_core_anchor_movies,nearest_liked_movies_latent,pre_rerank_rank,final_rank,rerank_reason,dominant_signal
0,Heat (1995),1995.0,Action,crime_thriller,strong,0.254717,0.473036,0.979413,0.982174,0.954774,0.718898,0.958675,0.555213,0.555213,Nightcrawler (2014) | Blade Runner (1982) | Un...,Nightcrawler (2014) | Blade Runner (1982) | Un...,1,1,kept_from_original_top20,semantic
1,Crimson Tide (1995),1995.0,Drama,crime_thriller,strong,0.254717,0.455069,0.871203,0.921918,0.815985,0.445610,0.820551,0.329838,0.329838,Dunkirk (2017) | Nightcrawler (2014) | Forrest...,Dunkirk (2017) | Nightcrawler (2014) | Forrest...,3,2,kept_from_original_top20,semantic
2,"Femme Nikita, La (Nikita) (1990)",1990.0,Action,crime_thriller,strong,0.254717,0.453098,0.878735,0.923927,0.802824,0.437384,0.931124,0.483652,0.483652,Nightcrawler (2014) | Blade Runner (1982) | Lo...,Nightcrawler (2014) | Blade Runner (1982) | Lo...,4,3,kept_from_original_top20,semantic
3,"Emperor's New Groove, The (2000)",2000.0,Adventure,animation_family,medium,0.084906,0.466870,0.733618,0.750439,0.731754,0.940572,0.493671,0.199054,0.199054,Toy Story 2 (1999) | Toy Story 3 (2010) | Mont...,"Monsters, Inc. (2001) | Toy Story 2 (1999) | S...",2,4,kept_from_original_top20,semantic
4,"Counterfeiters, The (Die Fälscher) (2007)",2007.0,Crime,crime_thriller,strong,0.254717,0.449271,0.684409,0.802912,0.665231,0.320790,0.547282,0.254370,0.254370,Nightcrawler (2014) | Uncut Gems (2019) | City...,"Hate (Haine, La) (1995) | Nightcrawler (2014) ...",6,5,kept_from_original_top20,semantic
5,"Lincoln Lawyer, The (2011)",2011.0,Crime,crime_thriller,strong,0.254717,0.446300,0.640974,0.778559,0.646088,0.440263,0.438943,0.353897,0.353897,City of God (Cidade de Deus) (2002) | Nightcra...,City of God (Cidade de Deus) (2002) | Nightcra...,8,6,kept_from_original_top20,semantic
6,Wallace & Gromit: The Best of Aardman Animatio...,1996.0,Adventure,animation_family,medium,0.084906,0.450839,0.616119,0.656540,0.624791,0.402838,0.388682,0.234629,0.234629,Toy Story 3 (2010) | Toy Story 2 (1999) | Mont...,"Toy Story 3 (2010) | Monsters, Inc. (2001) | S...",5,7,kept_from_original_top20,semantic
7,Schindler's List (1993),1993.0,Drama,psychological_thriller,medium,0.103774,0.427043,0.918654,0.848858,0.874850,0.614641,0.848474,0.557680,0.557680,Nightcrawler (2014) | Blade Runner (1982) | On...,"Hate (Haine, La) (1995) | Nightcrawler (2014) ...",15,8,kept_from_original_top20,semantic
8,"Good Morning, Vietnam (1987)",1987.0,Comedy,general,medium,0.113208,0.383356,0.677379,0.712277,0.692510,0.567345,0.400596,0.484063,0.484063,Once Upon a Time in Hollywood (2019) | Dunkirk...,Once Upon a Time in Hollywood (2019) | Dunkirk...,50,9,selected_profile_branch,semantic
9,Down by Law (1986),1986.0,Comedy,general,medium,0.113208,0.381954,0.865177,0.820236,0.801388,0.569196,0.838049,0.373432,0.373432,Uncut Gems (2019) | Nightcrawler (2014) | Blad...,Uncut Gems (2019) | Nightcrawler (2014) | Blad...,52,10,selected_profile_branch,semantic


## 16. Explicaciones

In [19]:
def _clean_explanation_terms(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple, set)):
        value = ", ".join(str(item) for item in value if item)
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()


def build_explanation(row):
    dominant_signal = row.get("dominant_signal", "")
    core_semantic_terms = _clean_explanation_terms(row.get("core_semantic_explanation_terms", ""))
    semantic_terms = core_semantic_terms or _clean_explanation_terms(row.get("semantic_explanation_terms", ""))
    core_negative_terms = _clean_explanation_terms(row.get("core_negative_semantic_terms", ""))
    negative_semantic_terms = core_negative_terms or _clean_explanation_terms(row.get("negative_semantic_terms", ""))
    similar_liked = _clean_explanation_terms(row.get("similar_liked_movies", ""))
    nearest_liked_latent = _clean_explanation_terms(row.get("nearest_liked_movies_latent", ""))
    nearest_core_anchors = _clean_explanation_terms(row.get("nearest_core_anchor_movies", ""))
    semantic_branch = _clean_explanation_terms(row.get("semantic_branch", ""))
    branch_explanation = {
        "psychological_thriller": "encaja con un perfil de peliculas psicologicas o de tension",
        "animation_family": "encaja con una rama de animacion o fantasia presente en tu perfil",
        "documentary_reflective": "encaja con una rama mas documental o reflexiva",
        "crime_thriller": "encaja con peliculas de crimen, tension o investigacion",
        "surreal_fantasy": "encaja con rasgos surrealistas, fantasticos o poco convencionales",
    }

    if dominant_signal == "semantic" and semantic_terms:
        explanation = f"Recomendada porque comparte rasgos semánticos especialmente representativos de tus películas mejor valoradas: {semantic_terms}."
    elif dominant_signal == "genre":
        explanation = "Recomendada porque encaja con los géneros que mejor aparecen en tu perfil."
    elif dominant_signal == "collab" and similar_liked:
        explanation = "Recomendada porque presenta similitud colaborativa con películas que valoraste positivamente: " + similar_liked + "."
    elif dominant_signal == "rating":
        explanation = "Recomendada porque destaca por su valoración media en MovieLens."
    elif dominant_signal == "popularity":
        explanation = "Recomendada porque combina afinidad con una señal alta de popularidad en MovieLens."
    elif semantic_terms:
        explanation = f"Recomendada porque comparte algunos rasgos semánticos con tus películas mejor valoradas: {semantic_terms}."
    else:
        explanation = "Recomendada porque mantiene un equilibrio razonable entre afinidad, calidad y popularidad."

    if semantic_branch in branch_explanation:
        explanation += " También " + branch_explanation[semantic_branch] + "."

    negative_semantic_signal = max(row.get("semantic_core_negative_score", 0), row.get("negative_semantic_score", 0))
    if row.get("rating_score", 0) >= 0.65:
        explanation += " Además, tiene buena valoración media."
    if negative_semantic_signal > 0.35 and negative_semantic_terms:
        explanation += f" Se controla la similitud con rasgos presentes en películas peor valoradas: {negative_semantic_terms}."
    if row.get("item_item_negative_collab_score", 0) > 0.35:
        explanation += " También se aplica una penalización colaborativa por similitud con películas peor valoradas."
    return explanation


recommendations["explanation"] = recommendations.apply(build_explanation, axis=1)

## 17. Resultados

In [20]:
RESULT_COLUMNS = [
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "dominant_signal",
    "hybrid_score",
    "explanation",
]

display(recommendations[RESULT_COLUMNS].head(20))

,title,year,genres,rating_mean,rating_count,genre_profile_score,semantic_raw_score,negative_semantic_raw_score,semantic_profile_score,negative_genre_score,...,contrib_semantic,contrib_collab,contrib_rating,contrib_popularity,penalty_negative_genre,penalty_negative_semantic,penalty_negative_collab,dominant_signal,hybrid_score,explanation
0,Heat (1995),1995.0,Action|Crime|Thriller,3.868277,29490,0.218045,161.257373,7.946345,0.869340,0.264151,...,0.294652,0.168087,0.052097,0.010022,0.019559,0.121592,0.000000,semantic,0.473036,Recomendada porque comparte rasgos semánticos ...
1,Crimson Tide (1995),1995.0,Drama|Thriller|War,3.744935,25270,0.315789,105.450723,0.000000,0.727964,0.245283,...,0.276575,0.115866,0.044696,0.009685,0.018161,0.068910,0.000000,semantic,0.455069,Recomendada porque comparte rasgos semánticos ...
2,"Femme Nikita, La (Nikita) (1990)",1990.0,Action|Crime|Romance|Thriller,3.863397,9868,0.270677,129.693677,6.269553,0.804355,0.320755,...,0.277178,0.155358,0.051804,0.007629,0.023750,0.107675,0.000000,semantic,0.453098,Recomendada porque comparte rasgos semánticos ...
3,"Emperor's New Groove, The (2000)",2000.0,Adventure|Animation|Children|Comedy|Fantasy,3.650783,11437,0.368421,0.000000,1.290081,0.000000,0.264151,...,0.225132,0.158296,0.039047,0.007952,0.019559,0.042540,0.000000,semantic,0.466870,Recomendada porque comparte rasgos semánticos ...
4,"Counterfeiters, The (Die Fälscher) (2007)",2007.0,Crime|Drama|War,3.902542,1416,0.300752,49.615084,6.533137,0.401832,0.188679,...,0.240874,0.138441,0.054153,0.003387,0.013970,0.068008,0.000000,semantic,0.449271,Recomendada porque comparte rasgos semánticos ...
5,"Lincoln Lawyer, The (2011)",2011.0,Crime|Drama|Thriller,3.760992,4094,0.375940,0.000000,1.290081,0.000000,0.283019,...,0.233568,0.156228,0.045660,0.005706,0.020956,0.072910,0.000000,semantic,0.446300,Recomendada porque comparte rasgos semánticos ...
6,Wallace & Gromit: The Best of Aardman Animatio...,1996.0,Adventure|Animation|Comedy,4.094810,8612,0.263158,0.000000,1.290081,0.000000,0.188679,...,0.196962,0.153726,0.065689,0.007332,0.013970,0.050992,0.000000,semantic,0.450839,Recomendada porque comparte rasgos semánticos ...
7,Schindler's List (1993),1993.0,Drama|War,4.236990,73849,0.225564,196.558623,17.556262,0.922226,0.150943,...,0.254657,0.176954,0.074219,0.012029,0.011176,0.169429,0.000000,semantic,0.427043,Recomendada porque comparte rasgos semánticos ...
8,"Good Morning, Vietnam (1987)",1987.0,Comedy|Drama|War,3.675327,14376,0.383459,16.948676,0.000000,0.035430,0.245283,...,0.213683,0.138114,0.040520,0.008452,0.018161,0.098716,0.000000,semantic,0.383356,Recomendada porque comparte rasgos semánticos ...
9,Down by Law (1986),1986.0,Comedy|Drama|Film-Noir,3.999235,3267,0.368421,120.868609,3.202929,0.783616,0.245283,...,0.246071,0.124243,0.059954,0.005213,0.018161,0.076573,0.057336,semantic,0.381954,Recomendada porque comparte rasgos semánticos ...


## 18. Sanity checks finales

In [21]:
n_recommendations = len(recommendations)
already_watched = recommendations["movieId"].isin(watched_movie_ids).sum()
already_rated = recommendations["movieId"].isin(rated_movie_ids).sum()
mean_rating_top = recommendations["rating_mean"].mean()
mean_count_top = recommendations["rating_count"].mean()
distinct_main_genres = recommendations["main_genre"].nunique()
high_negative = (recommendations["negative_similarity_score"] > 0.35).sum()
high_negative_collab = (recommendations["item_item_negative_collab_score"] > 0.35).sum()
high_semantic = (recommendations["semantic_profile_score"] > 0.5).sum()
high_negative_semantic = (recommendations["negative_semantic_score"] > 0.35).sum()
high_core_semantic = (recommendations["semantic_core_score"] > 0.5).sum() if "semantic_core_score" in recommendations.columns else 0
high_core_negative_semantic = (recommendations["semantic_core_negative_score"] > 0.35).sum() if "semantic_core_negative_score" in recommendations.columns else 0
low_rating = (recommendations["rating_mean"] < 3.2).sum()

def print_top_mean(column):
    if column in recommendations.columns:
        print(f"Media {column} top 20: {recommendations[column].mean():.3f}")


print(f"Número de recomendaciones: {n_recommendations}")
print(f"Ya vistas: {already_watched}")
print(f"Ya valoradas: {already_rated}")
print(f"rating_mean medio top 20: {mean_rating_top:.3f}")
print(f"rating_count medio top 20: {mean_count_top:.1f}")
print(f"Géneros principales distintos: {distinct_main_genres}")
print(f"Recomendaciones con negative_similarity_score > 0.35: {high_negative}")
print(f"Recomendaciones con item_item_negative_collab_score > 0.35: {high_negative_collab}")
print(f"Recomendaciones con semantic_profile_score > 0.5: {high_semantic}")
print(f"Recomendaciones con negative_semantic_score > 0.35: {high_negative_semantic}")
print(f"Recomendaciones con semantic_core_score > 0.5: {high_core_semantic}")
print(f"Recomendaciones con semantic_core_negative_score > 0.35: {high_core_negative_semantic}")
print(f"Recomendaciones con rating_mean < 3.2: {low_rating}")
print("Distribución de dominant_signal en top 20:")
print(recommendations["dominant_signal"].value_counts())
for diagnostic_column in [
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "latent_core_similarity_score",
    "semantic_relevance_score",
    "semantic_relevance_adjusted_score",
    "negative_semantic_relevance_score",
    "latent_core_preference_score",
    "negative_latent_preference_score",
    "negative_latent_similarity_score",
    "item_item_collab_score",
    "rating_score",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "year_affinity_score",
    "temporal_mismatch_penalty",
]:
    print_top_mean(diagnostic_column)
print("Top términos semánticos positivos del usuario:")
print(top_positive_semantic_terms[:10])
print("Top términos semánticos negativos del usuario:")
print(top_negative_semantic_terms[:10])

if already_watched > 0:
    warnings.warn("Hay recomendaciones que ya estaban vistas en Trakt.")
if already_rated > 0:
    warnings.warn("Hay recomendaciones que ya estaban valoradas en Trakt.")
if low_rating > 0:
    warnings.warn("Hay recomendaciones por debajo del umbral mínimo de rating_mean.")
if "semantic_relevance_adjusted_score" not in recommendations.columns:
    warnings.warn("semantic_relevance_adjusted_score no existe en recommendations.")
if "semantic_relevance_adjusted_score" in candidates_scored.columns and "contrib_semantic" in candidates_scored.columns:
    expected_contrib = weights["semantic"] * candidates_scored["semantic_relevance_adjusted_score"].fillna(0.0)
    max_contrib_delta = (candidates_scored["contrib_semantic"].fillna(0.0) - expected_contrib).abs().max()
    if max_contrib_delta > 1e-9:
        warnings.warn("hybrid_score no parece recalculado despues de semantic_relevance_adjusted_score.")
if "pre_latent_top20_titles" in globals() and set(recommendations["title"].astype(str)) == pre_latent_top20_titles:
    warnings.warn("El top 20 final es identico al top 20 anterior a la se?al latente.")
branch_warnings_final = recommendations.apply(_branch_warning_for_row, axis=1) if "semantic_branch" in recommendations.columns else pd.Series(dtype=str)
if (branch_warnings_final == "comedy_romance_as_psychological_thriller").any():
    warnings.warn("Comedy|Romance aparece como psychological_thriller.")
if int((recommendations["semantic_branch"] == "animation_family").sum()) > 3:
    warnings.warn("animation_family supera 3 peliculas en top 20.")
if "branch_strength" in recommendations.columns and int((recommendations["branch_strength"] == "unseen").sum()) > 1:
    warnings.warn("Una rama unseen supera 1 pelicula en top 20.")
if "rerank_reason" in recommendations.columns and recommendations["rerank_reason"].astype(str).str.startswith("fallback_fill").any():
    warnings.warn("Hay recomendaciones incluidas mediante fallback_fill.")
if n_recommendations < 20:
    warnings.warn("Hay menos de 20 recomendaciones finales.")


Número de recomendaciones: 20
Ya vistas: 0
Ya valoradas: 0
rating_mean medio top 20: 3.836
rating_count medio top 20: 12613.8
Géneros principales distintos: 8
Recomendaciones con negative_similarity_score > 0.35: 2
Recomendaciones con item_item_negative_collab_score > 0.35: 2
Recomendaciones con semantic_profile_score > 0.5: 14
Recomendaciones con negative_semantic_score > 0.35: 8
Recomendaciones con semantic_core_score > 0.5: 16
Recomendaciones con semantic_core_negative_score > 0.35: 1
Recomendaciones con rating_mean < 3.2: 0
Distribución de dominant_signal en top 20:
dominant_signal
semantic    20
Name: count, dtype: int64
Media semantic_profile_score top 20: 0.608
Media semantic_core_score top 20: 0.705
Media semantic_core_negative_score top 20: 0.101
Media semantic_net_score top 20: 0.739
Media latent_core_similarity_score top 20: 0.596
Media semantic_relevance_score top 20: 0.824
Media semantic_relevance_adjusted_score top 20: 0.819
Media negative_semantic_relevance_score top 20:

## 19. Exportación

In [22]:
EXPORT_COLUMNS = [
    "movieId",
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "rerank_reason",
    "selected_by_rerank",
    "semantic_branch",
    "branch_slot_limit",
    "branch_is_profile_supported",
    "branch_affinity_score",
    "branch_strength",
    "branch_share",
    "final_rank",
    "rerank_selection_score",
    "entered_by_rerank",
    "was_in_pre_rerank_top20",
    "pre_rerank_rank",
    "genre_profile_score",
    "semantic_raw_score",
    "negative_semantic_raw_score",
    "semantic_core_raw_score",
    "semantic_core_negative_raw_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_raw_score",
    "semantic_net_score",
    "nearest_disliked_movies_latent",
    "nearest_liked_movies_latent",
    "negative_semantic_relevance_score",
    "negative_semantic_relevance_raw",
    "semantic_relevance_score",
    "branch_support_multiplier",
    "semantic_relevance_raw",
    "semantic_relevance_adjusted_score",
    "negative_latent_similarity_score",
    "latent_core_similarity_score",
    "nearest_negative_movies_latent",
    "nearest_core_anchor_movies",
    "negative_latent_preference_score",
    "branch_prototype_similarity",
    "topk_anchor_similarity",
    "positive_centroid_similarity",
    "latent_core_preference_score",
    "latent_core_preference_raw",
    "semantic_negative_beta",
    "semantic_profile_score",
    "negative_genre_score",
    "negative_semantic_score",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "negative_semantic_terms",
    "content_profile_score",
    "negative_similarity_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "positive_collab_score",
    "negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "hybrid_score",
    "n_collab_evidence",
    "similar_liked_movies",
    "similar_disliked_movies",
    "explanation",
]

export_path = REPORTS_RESULTADOS / "recomendaciones_hibridas_item_item.csv"
recommendations[EXPORT_COLUMNS].to_csv(export_path, index=False)
print(f"Recomendaciones exportadas en: {export_path}")

Recomendaciones exportadas en: ..\reports\resultados\recomendaciones_hibridas_item_item.csv


In [23]:
diagnostic_cols = [
    "movieId",
    "title",
    "year",
    "genres",
    "rating_mean",
    "rating_count",
    "rerank_reason",
    "selected_by_rerank",
    "semantic_branch",
    "branch_slot_limit",
    "branch_is_profile_supported",
    "branch_affinity_score",
    "branch_strength",
    "branch_share",
    "final_rank",
    "rerank_selection_score",
    "entered_by_rerank",
    "was_in_pre_rerank_top20",
    "pre_rerank_rank",
    "genre_profile_score",
    "semantic_profile_score",
    "semantic_core_score",
    "semantic_core_negative_score",
    "semantic_net_score",
    "nearest_disliked_movies_latent",
    "nearest_liked_movies_latent",
    "negative_semantic_relevance_score",
    "negative_semantic_relevance_raw",
    "semantic_relevance_score",
    "branch_support_multiplier",
    "semantic_relevance_raw",
    "semantic_relevance_adjusted_score",
    "negative_latent_similarity_score",
    "latent_core_similarity_score",
    "nearest_negative_movies_latent",
    "nearest_core_anchor_movies",
    "negative_latent_preference_score",
    "branch_prototype_similarity",
    "topk_anchor_similarity",
    "positive_centroid_similarity",
    "latent_core_preference_score",
    "latent_core_preference_raw",
    "negative_genre_score",
    "negative_semantic_score",
    "item_item_collab_score",
    "item_item_negative_collab_score",
    "rating_score",
    "popularity_score",
    "allow_temporal_outlier",
    "is_temporal_outlier",
    "temporal_distance_from_profile",
    "temporal_mismatch_penalty",
    "year_affinity_score",
    "contrib_genre",
    "contrib_semantic",
    "contrib_collab",
    "contrib_rating",
    "contrib_popularity",
    "contrib_year_affinity",
    "penalty_negative_genre",
    "penalty_negative_semantic",
    "penalty_negative_collab",
    "penalty_temporal_mismatch",
    "dominant_signal",
    "semantic_explanation_terms",
    "core_semantic_explanation_terms",
    "core_negative_semantic_terms",
    "negative_semantic_terms",
    "hybrid_score",
    "explanation",
]

available_cols = [c for c in diagnostic_cols if c in recommendations.columns]

recommendations[available_cols].to_csv(
    REPORTS_RESULTADOS / "diagnostico_recomendaciones_top20.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Exportado:", REPORTS_RESULTADOS / "diagnostico_recomendaciones_top20.csv")

Exportado: ..\reports\resultados\diagnostico_recomendaciones_top20.csv


## Conclusión

Este notebook implementa el primer prototipo del recomendador híbrido final.

MovieLens se usa para aprender similitud colaborativa entre películas. Trakt se usa para personalizar el ranking con gustos reales y excluir películas vistas o valoradas. LightFM queda como experimento independiente, no como ranking final.

Próximas mejoras:

1. Reutilizar el perfil avanzado de 04d con géneros + tags.
2. Ajustar pesos del hybrid_score.
3. Evaluar con split train/test por usuario.
4. Mejorar diversidad.
5. Refactorizar funciones a src solo cuando el notebook sea estable.